# Mission 3 ver22 — 발화와 앞뒤 문맥에서 오심 근거를 다시 학습

Colab에서 GPU 런타임을 선택하고 **런타임 → 모두 실행**을 누르세요.
마지막 학습 셀만 실행하지 마세요. 런타임을 다시 연결하거나 초기화했다면 처음부터 모두 실행합니다.
준비 셀이 실패하면 그 셀의 첫 오류를 확인하고 해결한 뒤 다시 실행하세요.
Drive 연결 요청에는 기존 입력 데이터와 ver19 결과가 있는 **공용 계정**을 선택합니다.

**필요한 파일은 이 노트북, 기존 `mission3_ver5_inputs.zip`, 기존 ver19 결과 폴더 또는 ZIP입니다.**
ver19의 내부 평가용 모델과 전체 Training 모델의 가중치·assets가 모두 필요합니다.
이전에 업로드한 ver19 결과를 Drive에서 자동으로 찾습니다. 찾지 못하면 오류에 나온 안내에 따라
`source_run`에 ver19 결과 폴더 또는 ZIP의 전체 Drive 경로를 넣으세요.
입력 ZIP만으로 실행되는 버전이 아닙니다. 새 공개 모델을 추가로 내려받지 않습니다.

ver21에서는 마지막 판단층만 바꿔도 오심이 개선되지 않았습니다.
이번에는 **각 발화와 앞뒤 발화에서 오심 근거를 고르는 부분을 다시 학습**합니다.
비용이 큰 기존 언어 모델과 대화 전체의 판단 경로는 고정합니다.
다른 8개 증상의 출력은 기존 모델 그대로 유지합니다. 발화 특징을 한 번 저장해 두 학습법에 재사용합니다.
질문·부정·현재 증상을 별도 정답으로 추가하지는 않습니다. 원래 오심 정답만으로 문맥 표현을 학습합니다.
원래 정답과 모든 샘플·발화를 유지하며 입력은 `utterances[].text`뿐입니다.
threshold 탐색·사후 점수 보정·외부 학습 데이터 추가 없이 증상별 두 class의 `argmax`로 예측합니다.
F1 상승과 실제 Colab 속도는 아직 측정하지 않았습니다. 큰 폭의 상승을 보장하지 않습니다.

**작업 예산은 설치·데이터 준비·특징 계산·학습·최종 평가를 포함해 180분입니다.**
Drive 로그인 대기 시간은 제외합니다. 느린 GPU나 Drive 상태에 따라 새 학습을 생략할 수 있습니다.
시간 예산과 모든 단계의 완료는 다릅니다. 완료 표시가 없는 실행은 제출 결과로 사용하지 마세요.
결과 표시 후, 오류 발생 또는 시간 제한에 도달하면 GPU 런타임의 자동 반환을 요청합니다.
결과는 Drive에 남지만 `/content`의 임시 파일은 사라질 수 있습니다.
용량이 큰 발화 특징은 Colab 임시 공간에만 저장해 Drive 업로드 시간을 줄입니다.
런타임 반환 뒤 재개하면 특징을 다시 계산할 수 있습니다. 완료된 모델과 결과는 Drive에 보관됩니다.


## 비교와 최종 모델 선택

| 후보 | 학습 대상 | 학습 목표 |
|---|---|---|
| `baseline` | 기존 ver19 그대로 | 새 학습 없음 |
| `evidence_ce` | 오심 발화 문맥 변환·근거 선택·출력 | 기존 양성 가중치 3.0의 cross entropy |
| `evidence_ce_gce` | 동일한 근거 학습 부분과 동일한 시작 가중치 | 1 epoch CE 후 CE와 generalized cross entropy의 혼합 |

두 후보는 같은 ver19 근거 학습 가중치와 seed 42에서 시작합니다.
최대 8 epoch 학습하고 매 epoch 내부 평가하며, 3 epoch 동안 개선되지 않으면 조기 종료합니다.
저장한 발화 특징을 128건씩 학습해 언어 모델을 매번 다시 계산하는 비용을 줄입니다.
학습률 0.0001, weight decay 0.01을 사용하고 기존 dropout 0.1을 유지합니다.
두 번째 후보는 첫 epoch 이후 가중 CE와 GCE를 0.5씩 혼합하며 GCE의 q는 0.7입니다.
GCE는 잘 맞지 않는 정답이 학습에 미치는 영향을 완화하려는 실험입니다. 정답 오류가 확인됐다는 뜻은 아닙니다.

기존 Training 분할을 검증하고 **내부 fit 23,360건으로 학습한 ver19 모델**에서 특징을 추출해
동일한 holdout 5,840건에서 후보를 비교합니다. 전체 Training 모델을 이 비교에 사용하지 않습니다.
Training 내부 평가에서 기존 기준보다 **전체 Macro F1과 오심 F1이 모두 높아진 경우만** 새 후보를 채택합니다.
채택 전 실제 fit 모델로 holdout을 다시 추론해 캐시 평가와의 차이를 확인하고 실제 예측으로 선택을 확정합니다.
공식 Validation은 후보와 학습 횟수를 정한 뒤 한 번 평가하며 학습·선택에 사용하지 않습니다.
반복 사용한 Training holdout의 개선은 탐색 결과이며 새 독립 평가셋의 성능 보장이 아닙니다.

개선 후보가 있으면 기존 **전체 Training 29,200건으로 학습한 ver19 모델**에서 특징을 다시 계산합니다.
이 모델의 원래 오심 발화 근거 부분에서 시작해 선택한 방법·epoch 수로 다시 학습합니다.
fit 모델의 새 근거 가중치를 전체 Training 모델에 그대로 붙이지 않습니다.
개선에 실패하거나 새 학습을 완료할 시간이 없으면 **기존 전체 Training ver19 모델을 그대로 보존한 결과**를 저장합니다.
이 경우 새 모델의 성능 향상으로 표시하지 않습니다. 기존 Drive의 ver19 파일은 수정하지 않습니다.
필수 소스 확인·내부 비교·최종 저장 자체가 시간 내 끝나지 않으면 미완료로 종료합니다.


In [ ]:
# @title 실행 설정
SETTINGS = {'version': 'ver22', 'input_zip': '/content/drive/MyDrive/mission3_ver5_inputs.zip', 'output_dir': '/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver22', 'work_dir': '/content/mission3_ver22', 'train_dir': '', 'validation_dir': '', 'source_run': '', 'run_validation': True, 'seed': 42, 'holdout_size': 0.2, 'batch_size': 4, 'encoder_window_batch': 4, 'max_evidence_epochs': 8, 'evidence_batch_size': 128, 'evidence_learning_rate': 0.0001, 'evidence_weight_decay': 0.01, 'early_stopping_patience': 3, 'positive_weight': 3.0, 'gce_q': 0.7, 'gce_mix': 0.5, 'ce_warmup_epochs': 1, 'refit_full_training': True, 'allow_cpu': False, 'log_every': 250, 'time_budget_minutes': 180, 'finalization_reserve_seconds': 1500, 'budget_safety_factor': 1.25, 'resume_run': ''}
AUTO_RELEASE_GPU = True # @param {type:"boolean"}
RUNTIME_PREPARED = None
DEPENDENCIES_PREPARED = None
# Leave source_run empty to discover the existing ver19 result folder or ZIP.
# If discovery fails, set source_run to its complete Drive path.
# Each explicit rerun starts a NEW budget and consumes additional Colab resources.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
# @title 3시간 작업 예산 시작 (이후 패키지 설치 시간도 포함)
import time
import threading
RUNTIME_PREPARED = None
DEPENDENCIES_PREPARED = None
if not isinstance(globals().get("SETTINGS"), dict) or "AUTO_RELEASE_GPU" not in globals():
    raise RuntimeError("실행 설정이 없습니다. 런타임 → 모두 실행으로 처음 셀부터 실행하세요.")
BUDGET_STARTED_MONOTONIC = time.monotonic()
DEADLINE_MONOTONIC = BUDGET_STARTED_MONOTONIC + float(SETTINGS["time_budget_minutes"]) * 60
SETTINGS["deadline_monotonic"] = DEADLINE_MONOTONIC
previous_timer = globals().get("RELEASE_TIMER")
if previous_timer is not None:
    previous_timer.cancel()
RELEASE_TIMER = None
RELEASE_LOCK = threading.Lock()
RELEASE_REQUESTED = False

def release_gpu():
    global RELEASE_REQUESTED
    if not AUTO_RELEASE_GPU:
        return
    with RELEASE_LOCK:
        if RELEASE_REQUESTED:
            return
        RELEASE_REQUESTED = True
        if RELEASE_TIMER is not None:
            RELEASE_TIMER.cancel()
    try:
        from google.colab import runtime as colab_runtime
        print("Colab GPU 런타임 반환을 요청합니다. 저장된 결과는 Drive에서 확인하세요.", flush=True)
        colab_runtime.unassign()
    except Exception as error:
        print("자동 반환 요청 실패:", type(error).__name__,
              "· Colab 메뉴 '런타임 → 연결 해제 및 런타임 삭제'를 직접 실행하세요.", flush=True)

if AUTO_RELEASE_GPU:
    RELEASE_TIMER = threading.Timer(max(0, DEADLINE_MONOTONIC - time.monotonic()), release_gpu)
    RELEASE_TIMER.daemon = True
    RELEASE_TIMER.start()
print("작업 시간 예산:", SETTINGS["time_budget_minutes"], "분")


In [ ]:
# @title 실행 코드 준비 (코드 파일 추가 업로드 불필요)
import importlib.util
import sys
import tempfile
from pathlib import Path
RUNTIME_SOURCES = {'__init__.py': '"""Mission 3 ver22: frozen ver19 features and supervised nausea-head comparison."""\n', 'data.py': '"""Text-only Mission 3 data, grouped Training holdout, and discrete predictions."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport re\nimport tempfile\nimport unicodedata\n\nimport numpy as np\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nTARGET_SYMPTOMS = (\n    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",\n)\nLEXICAL_PATTERNS = {\n    "nausea": r"오심|메스꺼|메스껍|메슥|매스꺼|매스껍|매슥|울렁|구역|미식|느글|토할",\n    "question": r"\\?|나요|까요|습니까|신가|세요",\n    "negation": r"아니|없|않|못|안\\s|아닌",\n}\nLEXICAL_COUNTS_NOTE = (\n    "Regex matches are review candidates only: questions and negation may appear elsewhere "\n    "in the dialogue. These flags do not establish annotation errors or modify labels/predictions."\n)\n\n\ndef dialogue_text(annotation: dict) -> str:\n    """Read only utterance text, keeping turn boundaries and chronological order."""\n    key = "utterances" if "utterances" in annotation else "utterences"\n    utterances = annotation.get(key)\n    if not isinstance(utterances, list):\n        raise ValueError("utterances/utterences must be a list")\n    texts = []\n    for utterance in utterances:\n        if not isinstance(utterance, dict) or not isinstance(utterance.get("text"), str):\n            raise ValueError("Each utterance must be an object with string text")\n        text = " ".join(unicodedata.normalize("NFC", utterance["text"]).split())\n        if text:\n            texts.append(text)\n    return "\\n".join(texts)\n\n\ndef target_vector(annotation: dict) -> list[int]:\n    if "symptom" not in annotation:\n        raise ValueError("Training/evaluation JSON has no symptom target")\n    raw = annotation["symptom"]\n    if raw is None:\n        raw = []\n    if isinstance(raw, str):\n        raw = [raw]\n    if not isinstance(raw, list) or any(not isinstance(item, str) for item in raw):\n        raise ValueError("symptom must be a string, a list of strings, or null")\n    found = {item.strip() for item in raw}\n    return [int(name in found) for name in TARGET_SYMPTOMS]\n\n\ndef text_group(text: str) -> str:\n    normalized = " ".join(unicodedata.normalize("NFC", text).split())\n    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()\n\n\ndef load_dataset(label_dir, include_targets: bool = False):\n    label_dir = Path(label_dir)\n    if not label_dir.is_dir():\n        raise ValueError(f"Label directory does not exist: {label_dir}")\n    paths = sorted(path for path in label_dir.rglob("*.json") if path.is_file())\n    if not paths:\n        raise ValueError(f"No label JSON files: {label_dir}")\n    if len({path.name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate JSON filenames make submission identifiers ambiguous")\n\n    def read(path):\n        with path.open(encoding="utf-8-sig") as handle:\n            annotation = json.load(handle)\n        if not isinstance(annotation, dict):\n            raise ValueError(f"Expected a JSON object: {path}")\n        # Inference never inspects symptom, even when a supplied value is malformed.\n        return dialogue_text(annotation), target_vector(annotation) if include_targets else None\n\n    with ThreadPoolExecutor(max_workers=8) as executor:\n        rows = list(executor.map(read, paths))\n    texts = [row[0] for row in rows]\n    targets = np.asarray([row[1] for row in rows], dtype=np.int8) if include_targets else None\n    return paths, texts, targets\n\n\ndef _binary(values, rows=None, name="predictions"):\n    values = np.asarray(values)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or (rows is not None and values.shape[0] != rows)\n            or not np.isin(values, [0, 1]).all()):\n        raise ValueError(f"{name} must have shape (samples, 9) and contain only discrete 0/1 values")\n    return values.astype(np.int8)\n\n\ndef _greedy_groups(features, sizes, holdout_size, seed):\n    """Dependency-free iterative stratification of indivisible transcript groups.\n\n    This balances label presence and absence, then row counts. Group constraints\n    mean neither this fallback nor iterstrat can guarantee exact proportions.\n    """\n    rng = np.random.default_rng(seed)\n    active = np.ones(len(features), dtype=bool)\n    assignment = np.full(len(features), -1, dtype=np.int8)\n    remaining = features.sum(axis=0).astype(float)\n    proportions = np.asarray([1.0 - holdout_size, holdout_size])\n    desired_labels = proportions[:, None] * remaining[None, :]\n    desired_rows = proportions * sizes.sum()\n    coverage = np.zeros((2, features.shape[1]), dtype=bool)\n    while active.any():\n        available = np.flatnonzero(remaining > 0)\n        if not len(available):\n            candidates = np.flatnonzero(active)\n            rng.shuffle(candidates)\n            label = None\n        else:\n            rare = available[remaining[available] == remaining[available].min()]\n            label = int(rng.choice(rare))\n            candidates = np.flatnonzero(active & features[:, label].astype(bool))\n            rng.shuffle(candidates)\n        for group in candidates:\n            deficits = desired_rows if label is None else desired_labels[:, label]\n            sides = np.flatnonzero(deficits == deficits.max())\n            if len(sides) > 1:\n                sides = sides[desired_rows[sides] == desired_rows[sides].max()]\n            present = np.flatnonzero(features[group])\n            final_occurrences = present[remaining[present] == 1]\n            required = np.unique(np.where(~coverage[:, final_occurrences])[0])\n            if len(required) == 1:\n                # Reserve the last group for a side still missing this target.\n                sides = required\n            side = int(rng.choice(sides))\n            assignment[group] = side\n            desired_rows[side] -= sizes[group]\n            desired_labels[side] -= features[group]\n            remaining -= features[group]\n            coverage[side] |= features[group].astype(bool)\n            active[group] = False\n    return np.flatnonzero(assignment == 0), np.flatnonzero(assignment == 1)\n\n\ndef make_split(texts, targets, holdout_size: float = .2, seed: int = 42):\n    """Split Training only; normalized duplicate transcripts cannot cross sides."""\n    if not 0 < holdout_size < .5:\n        raise ValueError("holdout_size must be in (0, .5)")\n    targets = _binary(targets, len(texts), "targets")\n    if any(not isinstance(text, str) for text in texts):\n        raise TypeError("texts must contain strings only")\n    groups = np.asarray([text_group(text) for text in texts])\n    unique, inverse, sizes = np.unique(groups, return_inverse=True, return_counts=True)\n    presence = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int8)\n    absence = np.zeros_like(presence)\n    np.maximum.at(presence, inverse, targets)\n    np.maximum.at(absence, inverse, 1 - targets)\n    features = np.concatenate([presence, absence], axis=1)\n    if len(unique) < 2 or np.any(features.sum(axis=0) < 2):\n        raise ValueError("Each symptom needs positive and negative examples in at least two text groups")\n    try:\n        from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit\n    except ImportError:\n        MultilabelStratifiedShuffleSplit = None\n    best = None\n    # Retry only group allocation, never inspecting official Validation.\n    for attempt in range(8):\n        current_seed = seed + attempt\n        if MultilabelStratifiedShuffleSplit is None:\n            fit_groups, holdout_groups = _greedy_groups(features, sizes, holdout_size, current_seed)\n        else:\n            splitter = MultilabelStratifiedShuffleSplit(\n                n_splits=1, test_size=holdout_size, random_state=current_seed)\n            fit_groups, holdout_groups = next(splitter.split(np.arange(len(unique)), features))\n        fit = np.flatnonzero(np.isin(inverse, fit_groups))\n        holdout = np.flatnonzero(np.isin(inverse, holdout_groups))\n        valid = all(len(ids) and np.all(targets[ids].sum(axis=0) > 0)\n                    and np.all(targets[ids].sum(axis=0) < len(ids)) for ids in (fit, holdout))\n        if not valid and MultilabelStratifiedShuffleSplit is not None:\n            fit_groups, holdout_groups = _greedy_groups(features, sizes, holdout_size, current_seed)\n            fit = np.flatnonzero(np.isin(inverse, fit_groups))\n            holdout = np.flatnonzero(np.isin(inverse, holdout_groups))\n            valid = all(len(ids) and np.all(targets[ids].sum(axis=0) > 0)\n                        and np.all(targets[ids].sum(axis=0) < len(ids)) for ids in (fit, holdout))\n        if valid:\n            ratio = targets[holdout].sum(axis=0) / targets.sum(axis=0)\n            discrepancy = float(np.abs(ratio - holdout_size).mean()\n                                + abs(len(holdout) / len(targets) - holdout_size))\n            if best is None or discrepancy < best[0]:\n                best = (discrepancy, fit, holdout)\n    if best is None:\n        raise ValueError("Cannot split text groups with positive and negative examples for every symptom")\n    return best[1], best[2]\n\n\ndef _atomic_text(path, write, encoding="utf-8-sig"):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = None\n    try:\n        with tempfile.NamedTemporaryFile(mode="w", encoding=encoding, newline="",\n                                         dir=path.parent, prefix=".write_", suffix=".tmp", delete=False) as handle:\n            temporary = Path(handle.name)\n            write(handle)\n        temporary.replace(path)\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n\n\ndef write_json(path, value):\n    _atomic_text(path, lambda handle: json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False),\n                 encoding="utf-8")\n\n\ndef write_predictions(paths, predictions, output):\n    """Write already discrete model decisions; this API has no score or threshold."""\n    predictions = _binary(predictions, len(paths))\n    if len({Path(path).name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate prediction identifiers")\n\n    def write(handle):\n        writer = csv.writer(handle)\n        writer.writerow(["label file name", "symptom"])\n        for path, flags in zip(paths, predictions):\n            writer.writerow([Path(path).name, str([name for name, flag in zip(TARGET_SYMPTOMS, flags) if flag])])\n    _atomic_text(output, write)\n    return predictions\n\n\ndef score_predictions(targets, predictions):\n    targets = _binary(targets, name="targets")\n    predictions = _binary(predictions, len(targets))\n    if not len(targets):\n        raise ValueError("Cannot score an empty dataset")\n    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, zero_division=0)\n    return {\n        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),\n        "micro_f1": float(f1_score(targets, predictions, average="micro", zero_division=0)),\n        "samples": int(len(targets)),\n        "per_symptom": {name: {"f1": float(f1[i]), "precision": float(precision[i]),\n                                "recall": float(recall[i]), "support": int(support[i]),\n                                "false_positive": int(((targets[:, i] == 0) & (predictions[:, i] == 1)).sum()),\n                                "false_negative": int(((targets[:, i] == 1) & (predictions[:, i] == 0)).sum())}\n                        for i, name in enumerate(TARGET_SYMPTOMS)},\n    }\n\n\ndef training_diagnostics(texts, targets):\n    """Return aggregate counts only, including suspected nausea annotation ambiguity."""\n    targets = _binary(targets, len(texts), "targets")\n    groups = {}\n    for index, text in enumerate(texts):\n        groups.setdefault(text_group(text), []).append(index)\n    duplicates = [indices for indices in groups.values() if len(indices) > 1]\n    conflicts = [indices for indices in duplicates if len(np.unique(targets[indices], axis=0)) > 1]\n    nausea = targets[:, TARGET_SYMPTOMS.index("오심")].astype(bool)\n    vomiting = targets[:, TARGET_SYMPTOMS.index("구토")].astype(bool)\n    signals = {key: np.asarray([bool(re.search(pattern, text)) for text in texts])\n               for key, pattern in LEXICAL_PATTERNS.items()}\n    lexical = {key: int(flags.sum()) for key, flags in signals.items()}\n    for key in ("question", "negation"):\n        cooccurrence = signals["nausea"] & signals[key]\n        lexical[f"nausea_and_{key}"] = int(cooccurrence.sum())\n        lexical[f"nausea_and_{key}_label_positive"] = int((cooccurrence & nausea).sum())\n        lexical[f"nausea_and_{key}_label_negative"] = int((cooccurrence & ~nausea).sum())\n    lexical["nausea_mention_label_positive"] = int((signals["nausea"] & nausea).sum())\n    lexical["nausea_mention_label_negative"] = int((signals["nausea"] & ~nausea).sum())\n    return {\n        "samples": len(texts), "all_zero_samples": int((targets.sum(axis=1) == 0).sum()),\n        "empty_text_samples": sum(not text.strip() for text in texts),\n        "positive_counts": {name: int(targets[:, i].sum()) for i, name in enumerate(TARGET_SYMPTOMS)},\n        "text_groups": len(groups), "duplicate_groups": len(duplicates),\n        "duplicate_extra_samples": sum(len(indices) - 1 for indices in duplicates),\n        "conflicting_label_groups": len(conflicts),\n        "conflicting_label_samples": sum(len(indices) for indices in conflicts),\n        "nausea_vomiting_contingency": {\n            "neither": int((~nausea & ~vomiting).sum()), "nausea_only": int((nausea & ~vomiting).sum()),\n            "vomiting_only": int((~nausea & vomiting).sum()), "both": int((nausea & vomiting).sum())},\n        "lexical_cooccurrence_counts": lexical,\n        "lexical_patterns": LEXICAL_PATTERNS,\n        "lexical_counts_note": LEXICAL_COUNTS_NOTE,\n    }\n\n\ndef export_errors(paths, texts, targets, predictions, output_dir, max_per_label: int = 20):\n    """Save full text locally for manual review; never print sensitive dialogue."""\n    targets = _binary(targets, len(texts), "targets")\n    predictions = _binary(predictions, len(texts))\n    if len(paths) != len(texts) or type(max_per_label) is not int or max_per_label < 1:\n        raise ValueError("paths/texts must align and max_per_label must be a positive integer")\n    output_dir = Path(output_dir)\n    outputs = []\n    signals = {key: np.asarray([bool(re.search(pattern, text)) for text in texts])\n               for key, pattern in LEXICAL_PATTERNS.items()}\n    header = ["label file name", "symptom", "error_type", "true_symptoms", "predicted_symptoms", "text",\n              "review_category", "review_note", "has_nausea_expression", "has_question_expression",\n              "has_negation_expression"]\n\n    def review_row(index, symptom, category):\n        actual = [name for name, flag in zip(TARGET_SYMPTOMS, targets[index]) if flag]\n        predicted = [name for name, flag in zip(TARGET_SYMPTOMS, predictions[index]) if flag]\n        return [Path(paths[index]).name, symptom, category, str(actual), str(predicted), texts[index], "", "",\n                int(signals["nausea"][index]), int(signals["question"][index]), int(signals["negation"][index])]\n\n    for label, name in enumerate(TARGET_SYMPTOMS):\n        fn = np.flatnonzero((targets[:, label] == 1) & (predictions[:, label] == 0)).tolist()\n        fp = np.flatnonzero((targets[:, label] == 0) & (predictions[:, label] == 1)).tolist()\n        indices = []\n        while (fn or fp) and len(indices) < max_per_label:\n            for pending in (fn, fp):\n                if pending and len(indices) < max_per_label:\n                    indices.append(pending.pop(0))\n        output = output_dir / f"errors_{name}.csv"\n\n        def write(handle):\n            writer = csv.writer(handle)\n            writer.writerow(header)\n            for index in indices:\n                writer.writerow(review_row(index, name, "FN" if targets[index, label] else "FP"))\n        _atomic_text(output, write)\n        outputs.append(output)\n\n    nausea_index, vomiting_index = TARGET_SYMPTOMS.index("오심"), TARGET_SYMPTOMS.index("구토")\n    nausea, vomiting = targets[:, nausea_index].astype(bool), targets[:, vomiting_index].astype(bool)\n    combinations = {"neither": ~nausea & ~vomiting, "nausea_only": nausea & ~vomiting,\n                    "vomiting_only": ~nausea & vomiting, "both": nausea & vomiting}\n    group_counts = {}\n    for name, mask in combinations.items():\n        row = {"n": int(mask.sum())}\n        for symptom, column in (("nausea", nausea_index), ("vomiting", vomiting_index)):\n            actual, predicted = targets[:, column].astype(bool), predictions[:, column].astype(bool)\n            row[f"predicted_{symptom}"] = int((mask & predicted).sum())\n            row[f"{symptom}_tp"] = int((mask & actual & predicted).sum())\n            row[f"{symptom}_fp"] = int((mask & ~actual & predicted).sum())\n            row[f"{symptom}_fn"] = int((mask & actual & ~predicted).sum())\n            row[f"{symptom}_tn"] = int((mask & ~actual & ~predicted).sum())\n        group_counts[name] = row\n    summary_path = output_dir / "nausea_vomiting_error_groups.json"\n    write_json(summary_path, group_counts)\n    outputs.append(summary_path)\n\n    predicted_nausea = predictions[:, nausea_index].astype(bool)\n    controls = {"TP": nausea & predicted_nausea, "FP": ~nausea & predicted_nausea,\n                "FN": nausea & ~predicted_nausea, "TN": ~nausea & ~predicted_nausea}\n    rng = np.random.default_rng(42)\n    sampled = []\n    for category, mask in controls.items():\n        candidates = np.flatnonzero(mask)\n        chosen = rng.choice(candidates, size=min(10, len(candidates)), replace=False)\n        sampled.extend((int(index), category) for index in chosen)\n    # Mix categories for human review while preserving reproducibility.\n    rng.shuffle(sampled)\n    review_path = output_dir / "review_오심.csv"\n\n    def write_controls(handle):\n        writer = csv.writer(handle)\n        writer.writerow(header)\n        for index, category in sampled:\n            writer.writerow(review_row(index, "오심", category))\n    _atomic_text(review_path, write_controls)\n    outputs.append(review_path)\n    return outputs\n', 'bootstrap.py': '"""Safely restore label JSONs only; old model weights are never extracted."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\nTRAIN_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/"\nVALID_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/"\nMARKER_NAME = ".mission3_label_bundle_complete.json"\n\n\ndef _relative(name):\n    path = PurePosixPath(name)\n    if (not isinstance(name, str) or not name or "\\\\" in name or ":" in name\n            or path.is_absolute() or ".." in path.parts\n            or path.as_posix() != name.rstrip("/")):\n        raise ValueError(f"Unsafe bundle path: {name}")\n    return Path(*path.parts)\n\n\ndef _target(root, name):\n    target = root / _relative(name)\n    if not target.resolve().is_relative_to(root):\n        raise ValueError(f"Bundle path escapes extraction directory: {name}")\n    # Existing links must not redirect even to another location inside the root.\n    for component in (target, *target.parents):\n        if component == root.parent:\n            break\n        if component.is_symlink():\n            raise ValueError(f"Extraction destination contains a symlink: {name}")\n    return target\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as reader:\n        for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _selected_files(archive):\n    infos = archive.infolist()\n    names = set()\n    entry_types = {}\n    for info in infos:\n        # Windows ZipInfo normalizes backslashes in filename; orig_filename\n        # retains the stored member spelling that must also pass validation.\n        _relative(info.orig_filename)\n        _relative(info.filename)\n        normalized = info.filename.rstrip("/").casefold()\n        if normalized in names:\n            raise ValueError(f"Duplicate ZIP member or directory: {info.filename}")\n        names.add(normalized)\n        entry_types[normalized] = info.is_dir()\n        if stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 1:\n            raise ValueError("Links and encrypted ZIP entries are not supported")\n    for name in entry_types:\n        if any(parent.as_posix() in entry_types and not entry_types[parent.as_posix()]\n               for parent in PurePosixPath(name).parents if parent.as_posix() != "."):\n            raise ValueError("ZIP file/directory collision")\n    selected = [info for info in infos if not info.is_dir() and info.filename.endswith(".json")\n                and info.filename.startswith((TRAIN_PREFIX, VALID_PREFIX))]\n    for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n        subset = [info for info in selected if info.filename.startswith(prefix)]\n        if not subset:\n            raise ValueError(f"Input ZIP has no label JSONs in {prefix}")\n        if len({PurePosixPath(info.filename).name.casefold() for info in subset}) != len(subset):\n            raise ValueError("Duplicate label JSON filenames in input ZIP")\n    by_name = {info.filename: info for info in selected}\n    if "bundle_manifest.json" in archive.namelist():\n        raw = archive.read("bundle_manifest.json")\n        manifest = json.loads(raw)\n        if not isinstance(manifest, dict) or manifest.get("format") != "mission3_ver5_inputs_v1":\n            raise ValueError("Unsupported Mission 3 input bundle manifest")\n        files = manifest.get("files")\n        if not isinstance(files, list):\n            raise ValueError("Bundle manifest has no files")\n        metadata, listed = {}, set()\n        for row in files:\n            if not isinstance(row, dict) or not isinstance(row.get("path"), str):\n                raise ValueError("Invalid bundle manifest file entry")\n            name = row["path"]\n            _relative(name)\n            if name.casefold() in listed:\n                raise ValueError(f"Duplicate manifest entry: {name}")\n            listed.add(name.casefold())\n            if name not in by_name:\n                continue\n            size, digest = row.get("size"), row.get("sha256")\n            if (type(size) is not int or size < 0 or not isinstance(digest, str)\n                    or re.fullmatch(r"[0-9a-f]{64}", digest) is None):\n                raise ValueError(f"Invalid manifest metadata: {name}")\n            if by_name[name].file_size != size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest}\n        if set(metadata) != set(by_name):\n            raise ValueError("Bundle manifest does not list every label JSON")\n        for prefix, key in ((TRAIN_PREFIX, "training_count"), (VALID_PREFIX, "validation_count")):\n            count = sum(name.startswith(prefix) for name in metadata)\n            if type(manifest.get(key)) is not int or manifest[key] != count:\n                raise ValueError(f"Bundle JSON count mismatch: {key}")\n        signature = hashlib.sha256(raw).hexdigest()\n    else:\n        # Without a manifest, hash only label contents, never the large old models.\n        metadata = {}\n        for name, info in sorted(by_name.items()):\n            digest, size = hashlib.sha256(), 0\n            with archive.open(info) as reader:\n                for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                    digest.update(chunk)\n                    size += len(chunk)\n            if size != info.file_size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest.hexdigest()}\n        signature = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()\n    return [metadata[name] for name in sorted(metadata)], signature\n\n\ndef _validate_directory(directory, name):\n    directory = Path(directory).expanduser().resolve()\n    if not directory.is_dir() or not any(path.is_file() for path in directory.rglob("*.json")):\n        raise ValueError(f"{name} label directory does not contain JSON files: {directory}")\n    return directory\n\n\ndef prepare_data(input_zip, work_dir, train_dir="", validation_dir=""):\n    """Use two explicit label directories, or verify/extract the existing bundle.\n\n    Reuse requires a completion marker plus fresh content hashes. This detects\n    changed files even when their size and modification time are preserved.\n    """\n    if bool(train_dir) != bool(validation_dir):\n        raise ValueError("Set both train_dir and validation_dir, or leave both empty")\n    if train_dir:\n        train = _validate_directory(train_dir, "Training")\n        valid = _validate_directory(validation_dir, "Validation")\n        if train == valid or train.is_relative_to(valid) or valid.is_relative_to(train):\n            raise ValueError("Training and Validation label directories must be distinct and disjoint")\n        return train, valid\n    archive_path = Path(input_zip).expanduser().resolve()\n    if not archive_path.is_file():\n        raise FileNotFoundError(f"Input ZIP is missing: {archive_path}")\n    requested_root = Path(work_dir).expanduser()\n    if requested_root.is_symlink():\n        raise ValueError("Extraction work_dir must not be a symlink")\n    root = requested_root.resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / MARKER_NAME\n    with zipfile.ZipFile(archive_path) as archive:\n        files, signature = _selected_files(archive)\n        # Resolve every destination before any extraction, including existing links.\n        destinations = {row["path"]: _target(root, row["path"]) for row in files}\n        expected_paths = set(destinations.values())\n        for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n            directory = root / prefix\n            if directory.exists():\n                unexpected = set(directory.rglob("*.json")) - expected_paths\n                if unexpected:\n                    raise ValueError("Extraction directory contains label JSONs from another input bundle")\n        previous = None\n        if marker.exists():\n            if marker.is_symlink():\n                raise ValueError("Completion marker must not be a symlink")\n            try:\n                previous = json.loads(marker.read_text(encoding="utf-8"))\n            except (OSError, ValueError):\n                previous = None\n        expected = {row["path"]: {"size": row["size"], "sha256": row["sha256"]} for row in files}\n        reuse = (isinstance(previous, dict) and previous.get("signature") == signature\n                 and previous.get("files") == expected)\n        pending = []\n        for row in files:\n            target = destinations[row["path"]]\n            verified = (target.is_file() and target.stat().st_size == row["size"]\n                        and _hash(target) == row["sha256"])\n            if not verified:\n                pending.append(row)\n        if reuse and not pending:\n            return root / TRAIN_PREFIX, root / VALID_PREFIX\n        if marker.exists():\n            marker.unlink()\n        if shutil.disk_usage(root).free < sum(row["size"] for row in pending):\n            raise OSError("Insufficient local disk space to extract label JSONs")\n        for row in pending:\n            target = destinations[row["path"]]\n            target.parent.mkdir(parents=True, exist_ok=True)\n            temporary = None\n            try:\n                digest, size = hashlib.sha256(), 0\n                with archive.open(row["path"]) as reader, tempfile.NamedTemporaryFile(\n                        dir=target.parent, prefix=".extract_", suffix=".tmp", delete=False) as writer:\n                    temporary = Path(writer.name)\n                    for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                        writer.write(chunk)\n                        digest.update(chunk)\n                        size += len(chunk)\n                if size != row["size"] or digest.hexdigest() != row["sha256"]:\n                    raise ValueError(f"ZIP content verification failed: {row[\'path\']}")\n                temporary.replace(target)\n            finally:\n                if temporary is not None:\n                    temporary.unlink(missing_ok=True)\n        # Do not publish an incomplete extraction as a reusable data set.\n        temporary = None\n        try:\n            with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=root,\n                                             prefix=".marker_", suffix=".tmp", delete=False) as writer:\n                temporary = Path(writer.name)\n                json.dump({"signature": signature, "files": expected}, writer, ensure_ascii=False)\n            temporary.replace(marker)\n        finally:\n            if temporary is not None:\n                temporary.unlink(missing_ok=True)\n    return root / TRAIN_PREFIX, root / VALID_PREFIX\n', 'budget.py': '"""Share a wall-clock deadline across setup, training, and final export.\n\nCreate the monotonic deadline once per notebook invocation, before package\ninstallation. Do not restore a monotonic timestamp from an earlier Colab VM.\n``Budget`` reserves finalization time when ``training=True``; subprocess\nsupervision always enforces the actual invocation deadline.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom numbers import Real\nimport os\nimport signal\nimport subprocess\nimport time\n\n\nclass BudgetExceeded(TimeoutError):\n    """Work stopped without declaring an incomplete artifact complete."""\n\n    def __init__(self, stage, *, deadline_monotonic=None):\n        self.stage = str(stage)\n        self.deadline_monotonic = deadline_monotonic\n        super().__init__(f"Wall-clock budget exhausted during {self.stage}")\n\n\ndef _finite_nonnegative(value, name):\n    if isinstance(value, bool) or not isinstance(value, Real):\n        raise ValueError(f"{name} must be a finite nonnegative number")\n    value = float(value)\n    if not math.isfinite(value) or value < 0:\n        raise ValueError(f"{name} must be a finite nonnegative number")\n    return value\n\n\nclass Budget:\n    """A deadline with conservative admission checks for expensive stages.\n\n    ``remaining(training=True)`` subtracts the export/evaluation reserve.\n    ``check`` raises at an exhausted deadline and otherwise returns seconds left.\n    ``can_start`` applies the safety factor to an estimated duration; it does not\n    extend the deadline or admit work when no time remains.\n    """\n\n    def __init__(self, deadline_monotonic, finalization_reserve_s=1200,\n                 safety_factor=1.25, clock=time.monotonic):\n        self.deadline_monotonic = _finite_nonnegative(deadline_monotonic, "deadline_monotonic")\n        self.finalization_reserve_s = _finite_nonnegative(finalization_reserve_s, "finalization_reserve_s")\n        self.safety_factor = _finite_nonnegative(safety_factor, "safety_factor")\n        if self.safety_factor < 1:\n            raise ValueError("safety_factor must be at least one")\n        if not callable(clock):\n            raise ValueError("clock must be callable")\n        self.clock = clock\n\n    def remaining(self, training=False):\n        if not isinstance(training, bool):\n            raise ValueError("training must be a boolean")\n        now = _finite_nonnegative(self.clock(), "clock result")\n        reserve = self.finalization_reserve_s if training else 0\n        return max(0.0, self.deadline_monotonic - reserve - now)\n\n    def check(self, stage, training=False):\n        remaining = self.remaining(training=training)\n        if remaining <= 0:\n            raise BudgetExceeded(stage, deadline_monotonic=self.deadline_monotonic)\n        return remaining\n\n    def can_start(self, estimated_seconds, training=True):\n        estimate = _finite_nonnegative(estimated_seconds, "estimated_seconds")\n        remaining = self.remaining(training=training)\n        return remaining > 0 and estimate * self.safety_factor <= remaining\n\n\ndef _signal_process(process, force=False):\n    """Signal only this supervisor\'s new process group (or Windows child)."""\n    try:\n        if os.name == "posix":\n            # Popen(start_new_session=True) makes this child\'s PID its group ID.\n            # Signal the group even if its leader exited, to stop descendants.\n            os.killpg(process.pid, signal.SIGKILL if force else signal.SIGTERM)\n        elif process.poll() is None:\n            process.kill() if force else process.terminate()\n    except ProcessLookupError:\n        pass\n\n\ndef _stop_process(process, deadline_monotonic):\n    """Use at most the remaining shutdown margin for graceful stop and reaping."""\n    _signal_process(process)\n    remaining = max(0.0, deadline_monotonic - time.monotonic())\n    try:\n        process.wait(timeout=min(5.0, remaining / 2))\n    except subprocess.TimeoutExpired:\n        pass\n    # A descendant may ignore SIGTERM even when the group leader has exited.\n    _signal_process(process, force=True)\n    try:\n        process.wait(timeout=max(0.0, deadline_monotonic - time.monotonic()))\n    except subprocess.TimeoutExpired:\n        # SIGKILL/TerminateProcess has already been issued; never wait unbounded.\n        pass\n\n\ndef run_supervised(command, deadline_monotonic, shutdown_margin_s=10, cwd=None, env=None):\n    """Run a command with inherited output and a bounded wall-clock watchdog.\n\n    Returns zero on success. Nonzero exit raises ``CalledProcessError``; timeout\n    raises ``BudgetExceeded`` after terminating the child. POSIX descendants are\n    stopped as a group; Windows supervision stops only its own direct child.\n    The shutdown margin is inside the supplied deadline, not added after it.\n    Commands must be argument sequences and are never executed through a shell.\n    """\n    deadline = _finite_nonnegative(deadline_monotonic, "deadline_monotonic")\n    margin = _finite_nonnegative(shutdown_margin_s, "shutdown_margin_s")\n    if margin <= 0:\n        raise ValueError("shutdown_margin_s must be positive")\n    if isinstance(command, (str, bytes)) or not isinstance(command, (list, tuple)) or not command:\n        raise ValueError("command must be a nonempty argument sequence")\n    if deadline - margin - time.monotonic() <= 0:\n        raise BudgetExceeded("subprocess startup", deadline_monotonic=deadline)\n    process = subprocess.Popen(command, cwd=cwd, env=env, start_new_session=os.name == "posix")\n    try:\n        returncode = process.wait(timeout=max(0.0, deadline - margin - time.monotonic()))\n    except subprocess.TimeoutExpired:\n        _stop_process(process, deadline)\n        raise BudgetExceeded("subprocess execution", deadline_monotonic=deadline) from None\n    except BaseException:\n        _stop_process(process, min(deadline, time.monotonic() + margin))\n        raise\n    if returncode:\n        _stop_process(process, min(deadline, time.monotonic() + margin))\n        raise subprocess.CalledProcessError(returncode, command)\n    return returncode\n', 'neural.py': '"""Text-only dialogue classifiers with ordinary binary class decisions.\n\nEach symptom has two class logits. Predictions use the winning class directly;\nthere is no probability cutoff search, calibration, or post-hoc label weighting.\nPublic pretrained assets are prepared separately from local checkpoint weights.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom bisect import bisect_right\nfrom collections import Counter\nfrom numbers import Real\nfrom pathlib import Path\nimport random\nimport re\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import Dataset\n\n\nNUM_LABELS = 9\nCONTRASTIVE_DIM = 64\n_MODEL_INPUTS = ("input_ids", "attention_mask", "token_type_ids")\n\n\ndef seed_everything(seed):\n    if isinstance(seed, bool) or not isinstance(seed, int) or seed < 0:\n        raise ValueError("seed must be a nonnegative integer")\n    random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef prepare_encoder(model_id, revision, asset_dir):\n    """Download an encoder once; store only its config and tokenizer assets.\n\n    The caller stores the trained encoder and head together in its checkpoint.\n    Transformers imports are lazy so CPU tests do not require that dependency.\n    """\n    from transformers import AutoConfig, AutoModel, AutoTokenizer\n\n    if not isinstance(model_id, str) or not model_id.strip():\n        raise ValueError("model_id must be a nonempty string")\n    if not isinstance(revision, str) or not revision.strip():\n        raise ValueError("revision must be explicit")\n    config = AutoConfig.from_pretrained(model_id, revision=revision, trust_remote_code=False)\n    resolved = getattr(config, "_commit_hash", None)\n    if resolved is None and re.fullmatch(r"[0-9a-f]{40}", revision):\n        resolved = revision\n    if not isinstance(resolved, str) or not re.fullmatch(r"[0-9a-f]{40}", resolved):\n        raise RuntimeError("The immutable pretrained model revision was not resolved")\n    # Resolve a mutable branch exactly once, then pin every other asset to it.\n    options = {"revision": resolved, "trust_remote_code": False}\n    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, **options)\n    if not getattr(tokenizer, "is_fast", False):\n        raise ValueError("Complete overflow windows require a fast tokenizer")\n    encoder = AutoModel.from_pretrained(model_id, config=config, **options)\n    if hasattr(encoder.config, "use_cache"):\n        encoder.config.use_cache = False\n    if not getattr(encoder, "supports_gradient_checkpointing", False):\n        raise ValueError("The selected encoder must support gradient checkpointing")\n    encoder.gradient_checkpointing_enable(\n        gradient_checkpointing_kwargs={"use_reentrant": False},\n    )\n    asset_dir = Path(asset_dir)\n    encoder_dir, tokenizer_dir = asset_dir / "encoder", asset_dir / "tokenizer"\n    encoder_dir.mkdir(parents=True, exist_ok=True)\n    tokenizer_dir.mkdir(parents=True, exist_ok=True)\n    encoder.config.save_pretrained(encoder_dir)\n    tokenizer.save_pretrained(tokenizer_dir)\n    return encoder, tokenizer, resolved\n\n\ndef _binary_targets(targets, count):\n    values = torch.as_tensor(targets)\n    if values.shape != (count, NUM_LABELS):\n        raise ValueError(f"targets must have shape ({count}, {NUM_LABELS})")\n    if not bool(torch.isfinite(values).all()) or not bool(((values == 0) | (values == 1)).all()):\n        raise ValueError("targets must contain only finite binary values")\n    return values.to(dtype=torch.long)\n\n\ndef _turn_spans(text, separator):\n    """Locate nonempty raw turns in the unchanged separator-joined input."""\n    pieces = re.split(r"\\r\\n?|\\n", text)\n    delimiter = f" {separator} "\n    spans, cursor = [], 0\n    for piece in pieces:\n        if piece.strip():\n            spans.append((cursor, cursor + len(piece), len(spans)))\n        cursor += len(piece) + len(delimiter)\n    return delimiter.join(pieces), spans\n\n\nclass EncodeDataset(Dataset):\n    """Retain all fast-tokenizer overflow windows, including the final tokens."""\n\n    def __init__(self, texts, targets, tokenizer, max_length=512, overlap=64, *,\n                 utterance_evidence=False):\n        texts = list(texts)\n        if type(utterance_evidence) is not bool:\n            raise ValueError("utterance_evidence must be boolean")\n        if any(not isinstance(text, str) for text in texts):\n            raise ValueError("Every dialogue must be a text string")\n        for name, value in (("max_length", max_length), ("overlap", overlap)):\n            if isinstance(value, bool) or not isinstance(value, int):\n                raise ValueError(f"{name} must be an integer")\n        usable = max_length - tokenizer.num_special_tokens_to_add(pair=False)\n        if usable < 1 or not 0 <= overlap < usable:\n            raise ValueError("max_length must fit special tokens and 0 <= overlap < usable tokens")\n        if getattr(tokenizer, "pad_token_id", None) is None:\n            raise ValueError("The tokenizer must define a padding token")\n        separator = getattr(tokenizer, "sep_token", None)\n        if not isinstance(separator, str) or not separator:\n            raise ValueError("The tokenizer must define an utterance separator token")\n        self.tokenizer = tokenizer\n        self.utterance_evidence = utterance_evidence\n        self.targets = None if targets is None else _binary_targets(targets, len(texts)).clone()\n        self.documents = [[] for _ in texts]\n        # Unknown tokens still represent original content. Structural markers do\n        # not become local evidence, but remain in the unchanged global input.\n        structural_ids = set(getattr(tokenizer, "all_special_ids", []))\n        structural_ids.discard(getattr(tokenizer, "unk_token_id", None))\n        for start in range(0, len(texts), 128):\n            dialogues = [re.sub(r"\\r\\n?|\\n", f" {separator} ", text)\n                         for text in texts[start:start + 128]]\n            turn_spans = [_turn_spans(text, separator)[1] for text in texts[start:start + 128]] if utterance_evidence else None\n            options = {"return_offsets_mapping": True} if utterance_evidence else {}\n            encoded = tokenizer(\n                dialogues, truncation=True, max_length=max_length, stride=overlap,\n                return_overflowing_tokens=True, padding=False,\n                return_attention_mask=True,\n                **options,\n            )\n            if "overflow_to_sample_mapping" not in encoded:\n                raise ValueError("A fast tokenizer must return overflow_to_sample_mapping")\n            if "input_ids" not in encoded or "attention_mask" not in encoded:\n                raise ValueError("The tokenizer must return input_ids and attention_mask")\n            if utterance_evidence and "offset_mapping" not in encoded:\n                raise ValueError("Utterance evidence requires fast-tokenizer offset_mapping")\n            for index, source in enumerate(encoded["overflow_to_sample_mapping"]):\n                source = int(source)\n                if not 0 <= source < len(dialogues):\n                    raise ValueError("Tokenizer overflow mapping is out of range")\n                window = {key: list(encoded[key][index])\n                          for key in _MODEL_INPUTS if key in encoded}\n                length = len(window["input_ids"])\n                if not 0 < length <= max_length or any(len(value) != length for value in window.values()):\n                    raise ValueError("Tokenizer returned an invalid window length")\n                if not all(value in (0, 1) for value in window["attention_mask"]) or not any(window["attention_mask"]):\n                    raise ValueError("A real token window must have a binary, nonempty attention mask")\n                if utterance_evidence:\n                    offsets = encoded["offset_mapping"][index]\n                    if len(offsets) != length:\n                        raise ValueError("Tokenizer offsets must align with every token")\n                    spans = turn_spans[source]\n                    starts = [span[0] for span in spans]\n                    turn_ids, occurrence_keys = [], []\n                    for token, attended, offset in zip(window["input_ids"], window["attention_mask"], offsets):\n                        left, right = map(int, offset)\n                        if not 0 <= left <= right <= len(dialogues[source]):\n                            raise ValueError("Tokenizer offset is outside the prepared dialogue")\n                        position = bisect_right(starts, left) - 1\n                        valid = (bool(attended) and token not in structural_ids and right > left\n                                 and position >= 0 and right <= spans[position][1])\n                        turn_ids.append(spans[position][2] if valid else -1)\n                        occurrence_keys.append((left, right, int(token)) if valid else None)\n                    window["turn_ids"] = turn_ids\n                    window["_occurrence_keys"] = occurrence_keys\n                self.documents[start + source].append(window)\n        if any(not document for document in self.documents):\n            raise ValueError("Tokenizer returned a dialogue without any windows")\n        if utterance_evidence:\n            for document in self.documents:\n                counts = Counter(key for window in document for key in window["_occurrence_keys"] if key is not None)\n                for window in document:\n                    # Each original token has total mass one across overlapping\n                    # windows. Contextual occurrences are averaged, then tokens\n                    # within a turn are averaged; long/duplicated turns do not\n                    # gain extra attention entries. Adjacent turns are joined\n                    # only after this complete document-level reconstruction.\n                    window["token_weights"] = [1.0 / counts[key] if key is not None else 0.0\n                                               for key in window.pop("_occurrence_keys")]\n\n    def __len__(self):\n        return len(self.documents)\n\n    def __getitem__(self, index):\n        item = {"windows": self.documents[index]}\n        if self.targets is not None:\n            item["targets"] = self.targets[index]\n        return item\n\n    def collate(self, examples):\n        """Pad tokens and windows; keep one target vector for each dialogue."""\n        if not examples or any(not item["windows"] for item in examples):\n            raise ValueError("A batch must contain nonempty dialogue windows")\n        keys = set(examples[0]["windows"][0])\n        if any(set(window) != keys for item in examples for window in item["windows"]):\n            raise ValueError("Token windows must use the same model input fields")\n        if any(("targets" in item) != ("targets" in examples[0]) for item in examples):\n            raise ValueError("A batch cannot mix labeled and unlabeled dialogues")\n        batch_size = len(examples)\n        window_count = max(len(item["windows"]) for item in examples)\n        token_count = max(len(window["input_ids"]) for item in examples for window in item["windows"])\n        batch = {key: torch.full(\n            (batch_size, window_count, token_count),\n            self.tokenizer.pad_token_id if key == "input_ids" else 0,\n            dtype=torch.long,\n        ) for key in _MODEL_INPUTS if key in keys}\n        if self.utterance_evidence:\n            batch["turn_ids"] = torch.full((batch_size, window_count, token_count), -1, dtype=torch.long)\n            batch["token_weights"] = torch.zeros((batch_size, window_count, token_count), dtype=torch.float32)\n        batch["window_mask"] = torch.zeros((batch_size, window_count), dtype=torch.bool)\n        for row, item in enumerate(examples):\n            for column, window in enumerate(item["windows"]):\n                length = len(window["input_ids"])\n                for key in keys:\n                    batch[key][row, column, :length] = torch.tensor(window[key], dtype=batch[key].dtype)\n                batch["window_mask"][row, column] = True\n        if "targets" in examples[0]:\n            batch["targets"] = torch.stack([item["targets"] for item in examples])\n        return batch\n\n\ndef validate_evidence_config(hidden=128, dropout=0.1):\n    if type(hidden) is not int or not 1 <= hidden <= 512:\n        raise ValueError("evidence_hidden must be an integer between 1 and 512")\n    if (isinstance(dropout, bool) or not isinstance(dropout, Real)\n            or not math.isfinite(dropout) or not 0 <= dropout < 1):\n        raise ValueError("evidence_dropout must be finite in [0, 1)")\n    return {"evidence_hidden": hidden, "evidence_dropout": float(dropout)}\n\n\nclass UtteranceEvidence(nn.Module):\n    """Learn an ordered neighboring-turn residual for the nausea head only."""\n\n    def __init__(self, encoder_hidden, global_hidden, hidden=128, dropout=0.1):\n        super().__init__()\n        validate_evidence_config(hidden, dropout)\n        with torch.random.fork_rng(devices=[]):\n            self.projection = nn.Sequential(nn.Linear(encoder_hidden * 3 + global_hidden, hidden),\n                                            nn.GELU(), nn.Dropout(dropout))\n            self.attention = nn.Linear(hidden, 1)\n            self.output = nn.Linear(hidden, 2)\n            nn.init.zeros_(self.output.weight)\n            nn.init.zeros_(self.output.bias)\n\n    def forward(self, turns, turn_mask, global_features):\n        if turns.ndim != 3 or turn_mask.shape != turns.shape[:2] or turn_mask.dtype != torch.bool:\n            raise ValueError("Turn representations require a matching boolean mask")\n        # Compact valid turns in their original chronological order. This also\n        # handles a rare turn composed entirely of structural special tokens.\n        compact = turns.new_zeros(turns.shape)\n        compact_mask = torch.zeros_like(turn_mask)\n        for row in range(len(turns)):\n            count = int(turn_mask[row].sum())\n            compact[row, :count] = turns[row, turn_mask[row]]\n            compact_mask[row, :count] = True\n        previous = torch.cat((torch.zeros_like(compact[:, :1]), compact[:, :-1]), dim=1)\n        following = torch.cat((compact[:, 1:], torch.zeros_like(compact[:, :1])), dim=1)\n        global_context = global_features.unsqueeze(1).expand(-1, compact.shape[1], -1)\n        evidence = self.projection(torch.cat((previous, compact, following, global_context), dim=-1))\n        scores = self.attention(evidence).squeeze(-1).float()\n        has_turn = compact_mask.any(1)\n        safe_mask = compact_mask.clone()\n        safe_mask[~has_turn, 0] = True\n        scores = scores.masked_fill(~safe_mask, -torch.inf)\n        attention = scores.softmax(1).to(evidence.dtype)\n        summary = (attention.unsqueeze(-1) * evidence).sum(1)\n        residual = self.output(summary)\n        return residual * has_turn.unsqueeze(-1).to(residual.dtype)\n\n\nclass DialogueClassifier(nn.Module):\n    """Train nine independent binary heads from all windows of each dialogue."""\n\n    def __init__(self, encoder, label_attention=False, contrastive=False, feature_dim=256, *,\n                 utterance_evidence=False, evidence_hidden=128, evidence_dropout=0.1):\n        super().__init__()\n        if isinstance(feature_dim, bool) or not isinstance(feature_dim, int) or feature_dim < 1:\n            raise ValueError("feature_dim must be a positive integer")\n        if not isinstance(label_attention, bool) or not isinstance(contrastive, bool):\n            raise ValueError("label_attention and contrastive must be boolean")\n        if type(utterance_evidence) is not bool:\n            raise ValueError("utterance_evidence must be boolean")\n        validate_evidence_config(evidence_hidden, evidence_dropout)\n        if utterance_evidence and (label_attention or contrastive):\n            raise ValueError("Utterance evidence requires the single global CE pathway")\n        self.encoder = encoder\n        self.label_attention = label_attention\n        self.contrastive = contrastive\n        self.encoder_window_batch = 8\n        hidden_size = int(encoder.config.hidden_size)\n        self.projection = nn.Sequential(nn.Linear(hidden_size * 2, feature_dim), nn.GELU())\n        self.dropout = nn.Dropout(0.1)\n        if label_attention:\n            self.token_queries = nn.Parameter(torch.empty(NUM_LABELS, hidden_size))\n            self.window_queries = nn.Parameter(torch.empty(NUM_LABELS, feature_dim))\n            nn.init.normal_(self.token_queries, std=0.02)\n            nn.init.normal_(self.window_queries, std=0.02)\n        else:\n            self.register_parameter("token_queries", None)\n            self.register_parameter("window_queries", None)\n        self.class_weights = nn.Parameter(torch.empty(NUM_LABELS, feature_dim, 2))\n        self.class_bias = nn.Parameter(torch.zeros(NUM_LABELS, 2))\n        self.feature_weights = nn.Parameter(torch.empty(NUM_LABELS, feature_dim, CONTRASTIVE_DIM))\n        self.feature_bias = nn.Parameter(torch.zeros(NUM_LABELS, CONTRASTIVE_DIM))\n        nn.init.xavier_uniform_(self.class_weights)\n        nn.init.xavier_uniform_(self.feature_weights)\n        self.evidence = (UtteranceEvidence(hidden_size, feature_dim, evidence_hidden, evidence_dropout)\n                         if utterance_evidence else None)\n\n    def forward(self, batch):\n        window_mask = batch.get("window_mask")\n        ids, token_mask = batch.get("input_ids"), batch.get("attention_mask")\n        if not isinstance(ids, torch.Tensor) or ids.ndim != 3:\n            raise ValueError("input_ids must have shape (batch, windows, tokens)")\n        if not isinstance(token_mask, torch.Tensor) or token_mask.shape != ids.shape:\n            raise ValueError("attention_mask must match input_ids")\n        if not isinstance(window_mask, torch.Tensor) or window_mask.shape != ids.shape[:2] or window_mask.dtype != torch.bool:\n            raise ValueError("window_mask must be a boolean (batch, windows) tensor")\n        if ids.shape[0] < 1 or not bool(window_mask.any(dim=1).all()):\n            raise ValueError("Every dialogue must contain at least one real window")\n        if not bool(((token_mask == 0) | (token_mask == 1)).all()):\n            raise ValueError("attention_mask must be binary")\n        if not bool((token_mask.sum(-1)[window_mask] > 0).all()):\n            raise ValueError("Real windows must contain attended tokens")\n        chunk_size = self.encoder_window_batch\n        if isinstance(chunk_size, bool) or not isinstance(chunk_size, int) or chunk_size < 1:\n            raise ValueError("encoder_window_batch must be a positive integer")\n        flat_mask = window_mask.flatten()\n        turn_sum = turn_mass = None\n        if self.evidence is not None:\n            turn_ids, token_weights = batch.get("turn_ids"), batch.get("token_weights")\n            if (not isinstance(turn_ids, torch.Tensor) or turn_ids.shape != ids.shape or turn_ids.dtype != torch.long\n                    or not isinstance(token_weights, torch.Tensor) or token_weights.shape != ids.shape\n                    or not token_weights.is_floating_point()):\n                raise ValueError("Utterance evidence requires aligned turn_ids and token_weights")\n            valid_turn_tokens = turn_ids >= 0\n            if (bool((turn_ids < -1).any()) or not bool(torch.isfinite(token_weights).all())\n                    or bool((token_weights < 0).any()) or bool((token_weights > 1).any())\n                    or bool((token_weights[valid_turn_tokens] <= 0).any())\n                    or bool((token_weights[~valid_turn_tokens] != 0).any())\n                    or bool((valid_turn_tokens & ~token_mask.bool()).any())\n                    or bool((valid_turn_tokens & ~window_mask.unsqueeze(-1)).any())):\n                raise ValueError("Invalid turn IDs, padding masks, or overlap occurrence weights")\n            turn_count = max(1, int(turn_ids.max()) + 1)\n            if turn_count > int(token_mask.sum()):\n                raise ValueError("Turn IDs exceed the available token positions")\n            selected_turn_ids = turn_ids.flatten(0, 1)[flat_mask]\n            selected_weights = token_weights.flatten(0, 1)[flat_mask]\n            document_ids = torch.arange(ids.shape[0], device=ids.device).unsqueeze(1).expand_as(window_mask).flatten()[flat_mask]\n            turn_sum = torch.zeros((ids.shape[0] * turn_count, self.encoder.config.hidden_size),\n                                   dtype=torch.float32, device=ids.device)\n            turn_mass = torch.zeros(ids.shape[0] * turn_count, dtype=torch.float32, device=ids.device)\n        inputs = {}\n        for key in _MODEL_INPUTS:\n            if key in batch:\n                if batch[key].shape != ids.shape:\n                    raise ValueError(f"{key} must match input_ids")\n                inputs[key] = batch[key].flatten(0, 1)[flat_mask]\n        representations = []\n        for start in range(0, len(inputs["input_ids"]), chunk_size):\n            chunk = {key: value[start:start + chunk_size] for key, value in inputs.items()}\n            hidden = self.encoder(**chunk).last_hidden_state\n            if not bool(torch.isfinite(hidden).all()):\n                raise FloatingPointError("Encoder produced nonfinite hidden states")\n            if self.evidence is not None:\n                local_ids = selected_turn_ids[start:start + chunk_size]\n                local_weights = selected_weights[start:start + chunk_size]\n                present = local_ids >= 0\n                rows = document_ids[start:start + chunk_size].unsqueeze(1) * turn_count + local_ids.clamp_min(0)\n                turn_sum = turn_sum.index_add(0, rows[present], hidden.float()[present] * local_weights[present, None])\n                turn_mass = turn_mass.index_add(0, rows[present], local_weights[present])\n            if self.label_attention:\n                token_scores = torch.einsum("nth,lh->ntl", hidden.float(), self.token_queries.float())\n                token_scores = token_scores / math.sqrt(hidden.shape[-1])\n                token_scores = token_scores.masked_fill(~chunk["attention_mask"].bool().unsqueeze(-1), -torch.inf)\n                token_attention = torch.softmax(token_scores, dim=1).to(hidden.dtype)\n                context = torch.einsum("ntl,nth->nlh", token_attention, hidden)\n                cls = hidden[:, 0].unsqueeze(1).expand(-1, NUM_LABELS, -1)\n                representations.append(self.projection(torch.cat((cls, context), dim=-1)))\n            else:\n                mask = chunk["attention_mask"].unsqueeze(-1).to(hidden.dtype)\n                mean = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)\n                representations.append(self.projection(torch.cat((hidden[:, 0], mean), dim=-1)))\n        valid = self.dropout(torch.cat(representations, dim=0))\n        if not bool(torch.isfinite(valid).all()):\n            raise FloatingPointError("Window projection produced nonfinite features")\n        padded_shape = (*window_mask.shape, *valid.shape[1:])\n        windows = valid.new_zeros(padded_shape)\n        windows = windows.flatten(0, 1).index_copy(\n            0, flat_mask.nonzero(as_tuple=False).flatten(), valid,\n        ).reshape(padded_shape)\n        if self.label_attention:\n            scores = (windows.float() * self.window_queries.float()[None, None]).sum(-1)\n            scores = scores / math.sqrt(windows.shape[-1])\n            scores = scores.masked_fill(~window_mask.unsqueeze(-1), -torch.inf)\n            attention = torch.softmax(scores, dim=1).to(windows.dtype)\n            pooled = (attention.unsqueeze(-1) * windows).sum(dim=1)\n        else:\n            pooled = windows.sum(dim=1) / window_mask.sum(dim=1, keepdim=True).to(windows.dtype)\n            global_features = pooled\n            pooled = pooled.unsqueeze(1).expand(-1, NUM_LABELS, -1)\n        pooled = self.dropout(pooled)\n        logits = torch.einsum("bld,ldc->blc", pooled, self.class_weights) + self.class_bias\n        features = torch.einsum("bld,ldf->blf", pooled, self.feature_weights) + self.feature_bias\n        features = F.normalize(features.float(), dim=-1)\n        if self.evidence is not None:\n            turns = (turn_sum / turn_mass.clamp_min(1).unsqueeze(-1)).reshape(ids.shape[0], turn_count, -1)\n            turn_mask = turn_mass.reshape(ids.shape[0], turn_count) > 0\n            # No new stochastic operation runs until the original global logits\n            # and features are complete. All other symptom logits are untouched.\n            residual = self.evidence(turns.to(global_features.dtype), turn_mask, global_features)\n            nausea_mask = logits.new_zeros((1, NUM_LABELS, 1))\n            nausea_mask[:, 6] = 1\n            logits = logits + nausea_mask * residual.unsqueeze(1)\n        if not bool(torch.isfinite(logits).all()) or not bool(torch.isfinite(features).all()):\n            raise FloatingPointError("Classifier produced nonfinite logits or features")\n        return logits, features\n\n\ndef predictions(logits):\n    """The learned head\'s winning class; no fitted post-training cutoff."""\n    if logits.ndim != 3 or logits.shape[1:] != (NUM_LABELS, 2):\n        raise ValueError("logits must have shape (batch, 9, 2)")\n    if not bool(torch.isfinite(logits).all()):\n        raise FloatingPointError("Cannot predict from nonfinite logits")\n    return logits.argmax(dim=-1)\n\n\ndef classification_loss(logits, targets, nausea_positive_weight=1.0, *,\n                        positive_class_weights=None):\n    """Weight positive CE items by label, retaining the batch-by-label mean.\n\n    The scalar nausea weight remains supported for existing callers. Weights\n    affect supervised training only; prediction still uses the raw argmax.\n    """\n    if isinstance(nausea_positive_weight, bool) or not isinstance(nausea_positive_weight, Real):\n        raise ValueError("nausea_positive_weight must be a finite positive real number")\n    try:\n        positive_weight = float(nausea_positive_weight)\n    except (OverflowError, ValueError) as error:\n        raise ValueError("nausea_positive_weight must be a finite positive real number") from error\n    if not math.isfinite(positive_weight) or positive_weight <= 0:\n        raise ValueError("nausea_positive_weight must be a finite positive real number")\n    if positive_class_weights is None:\n        class_weights = [1.0] * NUM_LABELS\n        class_weights[6] = positive_weight\n    else:\n        if positive_weight != 1.0:\n            raise ValueError("Use positive_class_weights or a nondefault nausea_positive_weight, not both")\n        if not isinstance(positive_class_weights, (list, tuple)) or len(positive_class_weights) != NUM_LABELS:\n            raise ValueError("positive_class_weights must contain exactly nine finite positive real numbers")\n        class_weights = []\n        for value in positive_class_weights:\n            if isinstance(value, bool) or not isinstance(value, Real):\n                raise ValueError("positive_class_weights must contain exactly nine finite positive real numbers")\n            try:\n                weight = float(value)\n            except (OverflowError, ValueError) as error:\n                raise ValueError("positive_class_weights must contain exactly nine finite positive real numbers") from error\n            if not math.isfinite(weight) or weight <= 0:\n                raise ValueError("positive_class_weights must contain exactly nine finite positive real numbers")\n            class_weights.append(weight)\n    predictions(logits)\n    targets = _binary_targets(targets, len(logits)).to(logits.device)\n    if len(logits) < 1:\n        raise ValueError("classification_loss requires a nonempty batch")\n    if all(weight == 1.0 for weight in class_weights):\n        loss = F.cross_entropy(logits.float().reshape(-1, 2), targets.reshape(-1))\n    else:\n        per_item = F.cross_entropy(\n            logits.float().reshape(-1, 2), targets.reshape(-1), reduction="none",\n        ).reshape(-1, NUM_LABELS)\n        positive_weights = per_item.new_tensor(class_weights).unsqueeze(0)\n        weights = torch.where(targets == 1, positive_weights, 1.0)\n        loss = (per_item * weights).mean()\n    if not bool(torch.isfinite(loss)):\n        raise FloatingPointError("Classification loss is nonfinite")\n    return loss\n\n\ndef validate_loss_config(loss_mode="ce", gce_q=0.7, gce_mix=0.5, ce_warmup_epochs=1):\n    """Validate a predefined loss recipe; it never changes inference decisions."""\n    if loss_mode not in ("ce", "nausea_ce_gce"):\n        raise ValueError("loss_mode must be ce or nausea_ce_gce")\n    for name, value in (("gce_q", gce_q), ("gce_mix", gce_mix)):\n        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value):\n            raise ValueError(name + " must be a finite real number")\n    if not 0 < gce_q <= 1:\n        raise ValueError("gce_q must be in (0, 1]")\n    if not 0 <= gce_mix <= 0.5:\n        raise ValueError("gce_mix must be in [0, 0.5] to retain at least half of CE")\n    if type(ce_warmup_epochs) is not int or ce_warmup_epochs < 1:\n        raise ValueError("ce_warmup_epochs must be a positive integer")\n    return {"loss_mode": loss_mode, "gce_q": float(gce_q),\n            "gce_mix": float(gce_mix), "ce_warmup_epochs": ce_warmup_epochs}\n\n\ndef training_loss(logits, targets, *, positive_class_weights=None, loss_mode="ce",\n                  gce_q=0.7, gce_mix=0.5, ce_warmup_epochs=1, epoch=1):\n    """Mix GCE into nausea only, after CE warmup, preserving every labeled row.\n\n    For the recorded target probability p, GCE is (1 - p**q) / q.\n    The nausea gradient is the weighted-CE gradient times\n    (1 - mix) + mix * p**q, so even very hard targets retain at least\n    half their original CE gradient. This also reduces the nausea task\'s\n    relative gradient; an improvement would not prove annotation noise.\n    """\n    config = validate_loss_config(loss_mode, gce_q, gce_mix, ce_warmup_epochs)\n    if type(epoch) is not int or epoch < 1:\n        raise ValueError("epoch must be a positive integer")\n    base = classification_loss(logits, targets, positive_class_weights=positive_class_weights)\n    if config["loss_mode"] == "ce" or epoch <= config["ce_warmup_epochs"] or config["gce_mix"] == 0:\n        return base\n    target = _binary_targets(targets, len(logits)).to(logits.device)[:, 6]\n    log_probability = F.log_softmax(logits[:, 6].float(), dim=-1).gather(1, target[:, None]).squeeze(1)\n    ce = -log_probability\n    gce = -torch.expm1(config["gce_q"] * log_probability) / config["gce_q"]\n    positive_weight = 1.0 if positive_class_weights is None else float(positive_class_weights[6])\n    weight = torch.where(target == 1, positive_weight, 1.0)\n    # The unchanged CE path includes all nine labels. Only replace part of\n    # the nausea term; retain its original sample weights and denominator.\n    correction = (weight * (gce - ce)).mean() * (config["gce_mix"] / NUM_LABELS)\n    loss = base + correction\n    if not bool(torch.isfinite(loss)):\n        raise FloatingPointError("Mixed nausea loss is nonfinite")\n    return loss\n\n\ndef binary_supervised_contrastive_loss(features1, features2, targets, temperature=0.1):\n    """Binary supervised contrastive loss per symptom, with balanced anchors.\n\n    The two views come from independent dropout forwards of the same dialogues.\n    Same-class examples are positives, self matches are excluded, and each\n    present binary class receives equal anchor weight within each symptom.\n    Thus the many symptom-negative examples cannot dominate the anchor average.\n    """\n    if features1.ndim != 3 or features1.shape[1] != NUM_LABELS or features1.shape != features2.shape:\n        raise ValueError("Both feature views must have matching (batch, 9, feature_dim) shapes")\n    if features1.shape[0] < 1 or features1.shape[2] < 1:\n        raise ValueError("Contrastive loss requires nonempty feature views")\n    if not bool(torch.isfinite(features1).all()) or not bool(torch.isfinite(features2).all()):\n        raise FloatingPointError("Contrastive features must be finite")\n    if isinstance(temperature, bool) or not math.isfinite(float(temperature)) or temperature <= 0:\n        raise ValueError("temperature must be positive and finite")\n    labels = _binary_targets(targets, len(features1)).to(features1.device)\n    features = F.normalize(torch.cat((features1.float(), features2.float()), dim=0), dim=-1)\n    labels = torch.cat((labels, labels), dim=0)\n    self_mask = torch.eye(len(features), dtype=torch.bool, device=features.device)\n    losses = []\n    for label in range(NUM_LABELS):\n        scores = features[:, label] @ features[:, label].T / float(temperature)\n        denominator = torch.logsumexp(scores.masked_fill(self_mask, -torch.inf), dim=1, keepdim=True)\n        log_probability = scores - denominator\n        positive = (labels[:, label, None] == labels[None, :, label]) & ~self_mask\n        anchor_loss = -(log_probability * positive).sum(1) / positive.sum(1)\n        class_losses = [anchor_loss[labels[:, label] == value].mean()\n                        for value in (0, 1) if bool((labels[:, label] == value).any())]\n        losses.append(torch.stack(class_losses).mean())\n    loss = torch.stack(losses).mean()\n    if not bool(torch.isfinite(loss)):\n        raise FloatingPointError("Supervised contrastive loss is nonfinite")\n    return loss\n', 'head.py': '"""Frozen ver19 representations and supervised, independent nausea heads.\n\nThe encoder and every existing module stay in FP32 evaluation mode. Only a\nseparate 256-by-2 weight and two biases are optimized. All decisions remain raw\ntwo-class argmax; no thresholds, fitted output offsets, or calibration exist.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nfrom numbers import Real\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader\n\nfrom . import data\nfrom .budget import BudgetExceeded\nfrom .neural import DialogueClassifier, EncodeDataset, seed_everything\n\n\nNAUSEA = 6\nFEATURE_DIM = 256\nMODEL_ID = "klue/roberta-base"\nREVISION = "02f94ba5e3fcb7e2a58a390b8639b0fac974a8da"\nOTHER_LABELS = [index for index in range(9) if index != NAUSEA]\nRECIPES = ("head_ce", "head_ce_soft_f1")\n\n\ndef _safe(root, relative):\n    if not isinstance(relative, str) or not relative or Path(relative).is_absolute():\n        raise ValueError("Checkpoint asset paths must be nonempty relative paths")\n    root = Path(root).resolve()\n    result = (root / relative).resolve()\n    if not result.is_relative_to(root) or result == root:\n        raise ValueError("Checkpoint asset path escapes its directory")\n    return result\n\n\ndef load_model(checkpoint, device):\n    """Strictly restore local ver19/ver22 assets without fetching any model."""\n    from transformers import AutoConfig, AutoModel, AutoTokenizer\n\n    checkpoint = Path(checkpoint)\n    manifest = json.loads(checkpoint.read_text(encoding="utf-8-sig"))\n    if (manifest.get("format") != "mission3_categorical_v1"\n            or manifest.get("kind") != "neural"\n            or manifest.get("version") not in ("ver19", "ver22")\n            or tuple(manifest.get("labels", ())) != data.TARGET_SYMPTOMS\n            or manifest.get("label_attention") is not False\n            or manifest.get("contrastive") is not False\n            or manifest.get("utterance_evidence") is not True\n            or manifest.get("model_id") != MODEL_ID\n            or manifest.get("resolved_revision") != REVISION):\n        raise ValueError("Unsupported checkpoint, architecture, revision, or label ordering")\n    architecture = {key: manifest[key] for key in\n                    ("utterance_evidence", "evidence_hidden", "evidence_dropout")}\n    assets = _safe(checkpoint.parent, manifest["assets"])\n    weights_path = _safe(checkpoint.parent, manifest["weights"])\n    config = AutoConfig.from_pretrained(assets / "encoder", local_files_only=True,\n                                       trust_remote_code=False)\n    if (config.model_type != "roberta" or config.num_hidden_layers != 12\n            or config.hidden_size != 768):\n        raise ValueError("The checkpoint must contain the pinned base encoder architecture")\n    encoder = AutoModel.from_config(config, trust_remote_code=False)\n    tokenizer = AutoTokenizer.from_pretrained(assets / "tokenizer", local_files_only=True,\n                                              trust_remote_code=False)\n    if not getattr(tokenizer, "is_fast", False):\n        raise ValueError("Complete overflow windows require a fast tokenizer")\n    model = DialogueClassifier(encoder, label_attention=False, contrastive=False,\n                               **architecture)\n    chunk_size = manifest["encoder_window_batch"]\n    if type(chunk_size) is not int or chunk_size < 1:\n        raise ValueError("encoder_window_batch must be a positive integer")\n    model.encoder_window_batch = chunk_size\n    state = torch.load(weights_path, map_location="cpu", weights_only=True)\n    if not isinstance(state, dict) or any(not isinstance(value, torch.Tensor) for value in state.values()):\n        raise ValueError("Checkpoint weights must be a tensor-only state dictionary")\n    model.load_state_dict(state, strict=True)\n    model.requires_grad_(False)\n    model.float().to(torch.device(device)).eval()\n    return model, tokenizer, manifest\n\n\ndef _positive_int(value, name):\n    if type(value) is not int or value < 1:\n        raise ValueError(f"{name} must be a positive integer")\n    return value\n\n\ndef _finite(value, name, *, minimum=0, strict=False):\n    if (isinstance(value, bool) or not isinstance(value, Real)\n            or not math.isfinite(value) or value < minimum or (strict and value == minimum)):\n        raise ValueError(f"{name} is outside its permitted finite range")\n    return float(value)\n\n\ndef _settings(settings):\n    seed = settings.get("seed", 42)\n    if type(seed) is not int or seed < 0:\n        raise ValueError("seed must be a nonnegative integer")\n    return {\n        "seed": seed,\n        "max_head_epochs": _positive_int(settings.get("max_head_epochs", 80), "max_head_epochs"),\n        "head_eval_every": _positive_int(settings.get("head_eval_every", 5), "head_eval_every"),\n        "head_learning_rate": _finite(settings.get("head_learning_rate", .001), "head_learning_rate", strict=True),\n        "head_weight_decay": _finite(settings.get("head_weight_decay", 0), "head_weight_decay"),\n        "positive_weight": _finite(settings.get("positive_weight", 3), "positive_weight", strict=True),\n        "soft_f1_weight": _finite(settings.get("soft_f1_weight", .5), "soft_f1_weight"),\n    }\n\n\ndef _sync(device):\n    if torch.device(device).type == "cuda":\n        torch.cuda.synchronize(device)\n\n\ndef extract_cache(model, tokenizer, texts, settings, device, budget=None, *, training=True):\n    """Read unchanged forward-pass features with hooks, retaining all turns.\n\n    The measured first 64 rows are part of this pass, never extra inference.\n    Chunked tokenization permits budget checks and keeps very large datasets\n    from delaying the first pilot. Neither labels nor sample identifiers enter\n    this cache. The caller must attach checkpoint/dataset/order identities.\n    """\n    texts = list(texts)\n    if not texts:\n        raise ValueError("Cannot extract an empty feature cache")\n    if model.label_attention or model.contrastive or model.evidence is None:\n        raise ValueError("Cache extraction requires the ver19 global/evidence architecture")\n    if tuple(model.class_weights.shape) != (9, FEATURE_DIM, 2):\n        raise ValueError("Cache extraction requires 256-dimensional global features")\n    if any(not isinstance(text, str) for text in texts):\n        raise ValueError("Every dialogue must be a text string")\n    batch_size = _positive_int(settings.get("batch_size", 4), "batch_size")\n    cache_chunk = 64\n    training_stage = training\n    if type(training_stage) is not bool:\n        raise ValueError("training must be boolean")\n    model.requires_grad_(False)\n    model.float().to(torch.device(device)).eval()\n    feature_rows, evidence_rows, logits_rows = [], [], []\n    captured = {}\n\n    def capture_global(module, inputs, output):\n        if output.ndim == 3:\n            if output.shape[1:] != (9, FEATURE_DIM):\n                raise ValueError("Unexpected global feature hook shape")\n            if not torch.equal(output, output[:, :1].expand_as(output)):\n                raise ValueError("Global features differ across symptom labels")\n            if "global" in captured:\n                raise ValueError("Multiple global feature hook invocations in one forward")\n            captured["global"] = output[:, 0].detach()\n\n    def capture_evidence(module, inputs, output):\n        if output.ndim != 2 or output.shape[1] != 2 or "evidence" in captured:\n            raise ValueError("Unexpected utterance evidence hook invocation")\n        captured["evidence"] = output.detach()\n\n    hooks = [model.dropout.register_forward_hook(capture_global),\n             model.evidence.register_forward_hook(capture_evidence)]\n    started = time.monotonic()\n    pilot = None\n    maximum_error = 0.0\n    batches = 0\n    try:\n        with torch.inference_mode():\n            for start in range(0, len(texts), cache_chunk):\n                if budget:\n                    budget.check("tokenize frozen feature cache", training=training_stage)\n                dataset = EncodeDataset(\n                    texts[start:start + cache_chunk], None, tokenizer,\n                    max_length=settings.get("max_length", 512),\n                    overlap=settings.get("overlap", 64), utterance_evidence=True,\n                )\n                loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,\n                                    collate_fn=dataset.collate, num_workers=0)\n                for batch in loader:\n                    if budget:\n                        budget.check("extract frozen features", training=training_stage)\n                    captured.clear()\n                    batch = {key: value.to(device) for key, value in batch.items()}\n                    logits, _ = model(batch)\n                    if set(captured) != {"global", "evidence"}:\n                        raise RuntimeError("A required frozen representation hook did not run")\n                    global_features, residual = captured["global"], captured["evidence"]\n                    rebuilt = (torch.einsum("bd,dc->bc", global_features,\n                                            model.class_weights[NAUSEA])\n                               + model.class_bias[NAUSEA] + residual)\n                    error = float((rebuilt - logits[:, NAUSEA]).abs().max())\n                    maximum_error = max(maximum_error, error)\n                    if (not torch.allclose(rebuilt, logits[:, NAUSEA], rtol=1e-4, atol=1e-5)\n                            or not torch.equal(rebuilt.argmax(-1), logits[:, NAUSEA].argmax(-1))):\n                        raise RuntimeError("Frozen feature reconstruction changed original nausea decisions")\n                    feature_rows.append(global_features.float().cpu().numpy().copy())\n                    evidence_rows.append(residual.float().cpu().numpy().copy())\n                    logits_rows.append(logits.float().cpu().numpy().copy())\n                    batches += 1\n                    if batches % 250 == 0:\n                        print(f"Frozen cache: {sum(len(rows) for rows in feature_rows)}/{len(texts)} rows", flush=True)\n                if pilot is None:\n                    _sync(device)\n                    rows = min(cache_chunk, len(texts))\n                    elapsed = time.monotonic() - started\n                    estimate = elapsed / rows * (len(texts) - rows)\n                    pilot = {"rows": rows, "elapsed_seconds": elapsed,\n                             "estimated_remaining_seconds": estimate,\n                             "includes_tokenization": True, "device": str(device)}\n                    print(f"Frozen cache pilot: {rows} rows, {elapsed:.1f}s; remaining estimate {estimate:.1f}s", flush=True)\n                    if budget and not budget.can_start(estimate, training=training_stage):\n                        raise BudgetExceeded("frozen cache pilot admission",\n                                             deadline_monotonic=budget.deadline_monotonic)\n        result = {\n            "global_features": np.concatenate(feature_rows),\n            "evidence": np.concatenate(evidence_rows),\n            "logits": np.concatenate(logits_rows),\n            "initial_weight": model.class_weights[NAUSEA].detach().cpu().numpy().copy(),\n            "initial_bias": model.class_bias[NAUSEA].detach().cpu().numpy().copy(),\n        }\n        result["predictions"] = result["logits"].argmax(-1).astype(np.int8)\n        result["_timing"] = {"pilot": pilot, "rows": len(texts),\n                             "elapsed_seconds": time.monotonic() - started,\n                             "maximum_reconstruction_error": maximum_error,\n                             "evaluation_dtype": "float32", "trainable_model_parameters": 0}\n        _validate_cache(result)\n        return result\n    finally:\n        for handle in hooks:\n            handle.remove()\n\n\ndef _validate_cache(cache):\n    count = len(cache["global_features"])\n    if count < 1:\n        raise ValueError("Feature cache must not be empty")\n    shapes = {"global_features": (count, FEATURE_DIM), "evidence": (count, 2),\n              "logits": (count, 9, 2), "initial_weight": (FEATURE_DIM, 2),\n              "initial_bias": (2,)}\n    for name, shape in shapes.items():\n        value = np.asarray(cache[name])\n        if value.shape != shape or value.dtype != np.float32 or not np.isfinite(value).all():\n            raise ValueError(f"Invalid finite FP32 cache array: {name}")\n    predicted = data._binary(cache["predictions"], count, "cache predictions")\n    if not np.array_equal(predicted, np.asarray(cache["logits"]).argmax(-1)):\n        raise ValueError("Cache predictions do not match the stored raw logits")\n    return count\n\n\ndef _head_arrays(weight, bias):\n    weight, bias = np.asarray(weight), np.asarray(bias)\n    if (weight.shape != (FEATURE_DIM, 2) or bias.shape != (2,)\n            or weight.dtype != np.float32 or bias.dtype != np.float32\n            or not np.isfinite(weight).all() or not np.isfinite(bias).all()):\n        raise ValueError("Head parameters must be finite FP32 arrays with shapes (256, 2) and (2,)")\n    return weight, bias\n\n\ndef apply_head(cache, weight, bias):\n    """Replace only nausea logits; preserve eight original logits byte-for-byte."""\n    _validate_cache(cache)\n    weight, bias = _head_arrays(weight, bias)\n    # Whole-batch torch operations use the same kernel as head optimization.\n    with torch.inference_mode():\n        values = (torch.from_numpy(cache["global_features"]) @ torch.from_numpy(weight)\n                  + torch.from_numpy(bias) + torch.from_numpy(cache["evidence"]))\n    logits = cache["logits"].copy()\n    logits[:, NAUSEA] = values.numpy()\n    if not np.isfinite(logits).all():\n        raise FloatingPointError("Head produced nonfinite logits")\n    predicted = logits.argmax(-1).astype(np.int8)\n    if not np.array_equal(predicted[:, OTHER_LABELS], cache["predictions"][:, OTHER_LABELS]):\n        raise RuntimeError("A frozen symptom decision changed")\n    return logits, predicted\n\n\ndef head_loss(logits, targets, candidate_id, *, positive_weight=3., soft_f1_weight=.5):\n    """One true whole-fit CE/soft-F1 objective, without gradient accumulation."""\n    if candidate_id not in RECIPES:\n        raise ValueError("Unknown supervised head recipe")\n    positive_weight = _finite(positive_weight, "positive_weight", strict=True)\n    soft_f1_weight = _finite(soft_f1_weight, "soft_f1_weight")\n    if (logits.ndim != 2 or logits.shape[1] != 2 or logits.shape[0] < 1\n            or targets.shape != (len(logits),) or targets.dtype != torch.long\n            or not bool(((targets == 0) | (targets == 1)).all())\n            or not bool(torch.isfinite(logits).all())):\n        raise ValueError("Head loss requires finite Nx2 logits and N binary long targets")\n    costs = torch.where(targets.bool(), logits.new_tensor(positive_weight), logits.new_tensor(1.))\n    ce = (F.cross_entropy(logits, targets, reduction="none") * costs).mean()\n    probability = logits.softmax(-1)[:, 1]\n    truth = targets.to(probability.dtype)\n    soft_f1 = (2 * (probability * truth).sum() + 1e-7) / (probability.sum() + truth.sum() + 1e-7)\n    loss = ce + (soft_f1_weight * (1 - soft_f1) if candidate_id == "head_ce_soft_f1" else 0)\n    return loss, {"weighted_ce": float(ce.detach()), "soft_f1": float(soft_f1.detach())}\n\n\nclass IndependentHead(nn.Module):\n    def __init__(self, weight, bias):\n        super().__init__()\n        weight, bias = _head_arrays(weight, bias)\n        self.weight = nn.Parameter(torch.from_numpy(weight.copy()))\n        self.bias = nn.Parameter(torch.from_numpy(bias.copy()))\n\n    def forward(self, features, evidence):\n        return features @ self.weight + self.bias + evidence\n\n\ndef _indices(indices, count, name):\n    values = np.asarray(indices)\n    if (values.ndim != 1 or not len(values) or not np.issubdtype(values.dtype, np.integer)\n            or np.any(values < 0) or np.any(values >= count)\n            or len(np.unique(values)) != len(values)):\n        raise ValueError(f"Invalid unique {name} indices")\n    return values.astype(np.int64, copy=False)\n\n\ndef _key(metrics, baseline):\n    nausea = data.TARGET_SYMPTOMS[NAUSEA]\n    joint = (metrics["macro_f1"] > baseline["macro_f1"]\n             and metrics["per_symptom"][nausea]["f1"] > baseline["per_symptom"][nausea]["f1"])\n    return joint, metrics["macro_f1"]\n\n\ndef _snapshot(head):\n    return (head.weight.detach().cpu().numpy().copy(), head.bias.detach().cpu().numpy().copy())\n\n\ndef _subset(cache, indices):\n    return {key: (value[indices] if key in ("global_features", "evidence", "logits", "predictions") else value)\n            for key, value in cache.items()}\n\n\ndef _new_training(cache, targets, indices, options):\n    seed_everything(options["seed"])\n    head = IndependentHead(cache["initial_weight"], cache["initial_bias"])\n    optimizer = torch.optim.AdamW(head.parameters(), lr=options["head_learning_rate"],\n                                 weight_decay=options["head_weight_decay"])\n    features = torch.from_numpy(cache["global_features"][indices].copy())\n    evidence = torch.from_numpy(cache["evidence"][indices].copy())\n    truth = torch.from_numpy(targets[indices, NAUSEA].astype(np.int64))\n    return head, optimizer, features, evidence, truth\n\n\ndef _step(head, optimizer, features, evidence, truth, candidate_id, options):\n    head.train()\n    optimizer.zero_grad(set_to_none=True)\n    loss, details = head_loss(head(features, evidence), truth, candidate_id,\n                              positive_weight=options["positive_weight"],\n                              soft_f1_weight=options["soft_f1_weight"])\n    if not bool(torch.isfinite(loss)):\n        raise FloatingPointError("Nonfinite supervised head loss")\n    loss.backward()\n    if any(parameter.grad is None or not bool(torch.isfinite(parameter.grad).all())\n           for parameter in head.parameters()):\n        raise FloatingPointError("Missing or nonfinite supervised head gradient")\n    optimizer.step()\n    return {"loss": float(loss.detach()), **details}\n\n\ndef head_candidates(cache, targets, fit, holdout, settings, budget=None):\n    """Compare two predeclared supervised recipes against unchanged ver19.\n\n    Only fit targets enter an optimizer. Held-out labels are used exclusively\n    to select completed epochs. Exact ties retain the earliest evaluated epoch.\n    """\n    count = _validate_cache(cache)\n    targets = data._binary(targets, count, "targets")\n    fit, holdout = _indices(fit, count, "fit"), _indices(holdout, count, "holdout")\n    if len(np.intersect1d(fit, holdout)) or len(fit) + len(holdout) != count:\n        raise ValueError("Fit and holdout must form a disjoint complete partition")\n    options = _settings(settings)\n    held_cache = _subset(cache, holdout)\n    baseline_predictions = held_cache["predictions"].copy()\n    baseline = data.score_predictions(targets[holdout], baseline_predictions)\n    records = [{"candidate_id": "baseline", "epoch": 0, "metrics": baseline,\n                "weight": cache["initial_weight"].copy(), "bias": cache["initial_bias"].copy(),\n                "predictions": baseline_predictions, "history": [], "improved": False,\n                "stop_reason": "unchanged_original_head", "trainable_parameters": 0}]\n    for candidate_id in RECIPES:\n        head, optimizer, features, evidence, truth = _new_training(cache, targets, fit, options)\n        history, best, completed = [], None, 0\n        stop_reason = "max_head_epochs"\n        for epoch in range(1, options["max_head_epochs"] + 1):\n            try:\n                if budget:\n                    budget.check(f"{candidate_id} epoch {epoch}", training=True)\n            except BudgetExceeded:\n                stop_reason = "budget"\n                break\n            details = _step(head, optimizer, features, evidence, truth, candidate_id, options)\n            completed = epoch\n            if epoch % options["head_eval_every"] == 0 or epoch == options["max_head_epochs"]:\n                weight, bias = _snapshot(head)\n                _, predicted = apply_head(held_cache, weight, bias)\n                metrics = data.score_predictions(targets[holdout], predicted)\n                history.append({"epoch": epoch, **details, "metrics": metrics,\n                                "whole_fit_batch_size": len(fit)})\n                if best is None or _key(metrics, baseline) > _key(best["metrics"], baseline):\n                    best = {"epoch": epoch, "metrics": metrics, "weight": weight,\n                            "bias": bias, "predictions": predicted,\n                            "improved": _key(metrics, baseline)[0]}\n        # Never publish an unevaluated or partially completed optimizer step.\n        if best is not None:\n            records.append({"candidate_id": candidate_id, **best, "history": history,\n                            "stop_reason": stop_reason, "completed_epochs": completed,\n                            "trainable_parameters": sum(p.numel() for p in head.parameters())})\n        if stop_reason == "budget":\n            records[0]["search_stop_reason"] = "budget"\n            break\n    return records\n\n\ndef refit_head(cache, targets, candidate_id, epochs, settings, budget=None):\n    """Apply a fixed recipe to an independently cached full-Training encoder.\n\n    Initialization is the full encoder\'s own original head, never the weights\n    selected on the fit-only encoder. No held-out or Validation labels occur.\n    """\n    count = _validate_cache(cache)\n    targets = data._binary(targets, count, "targets")\n    if candidate_id not in RECIPES:\n        raise ValueError("Only a selected supervised recipe can be refitted")\n    options = _settings(settings)\n    epochs = _positive_int(epochs, "epochs")\n    if epochs > options["max_head_epochs"]:\n        raise ValueError("Refit epochs exceed the declared optimization budget")\n    head, optimizer, features, evidence, truth = _new_training(cache, targets, np.arange(count), options)\n    history = []\n    for epoch in range(1, epochs + 1):\n        if budget:\n            budget.check(f"full-Training {candidate_id} epoch {epoch}", training=True)\n        details = _step(head, optimizer, features, evidence, truth, candidate_id, options)\n        history.append({"epoch": epoch, **details, "whole_fit_batch_size": count})\n    weight, bias = _snapshot(head)\n    return weight, bias, history\n\n\ndef _state_hashes(model):\n    return {name: hashlib.sha256(value.detach().cpu().contiguous().numpy().tobytes()).hexdigest()\n            for name, value in model.state_dict().items()}\n\n\ndef set_nausea_head(model, weight, bias):\n    """Patch only symptom slice 6, proving every other model tensor unchanged."""\n    weight, bias = _head_arrays(weight, bias)\n    if tuple(model.class_weights.shape) != (9, FEATURE_DIM, 2):\n        raise ValueError("Cannot patch a different classifier architecture")\n    before = _state_hashes(model)\n    other_weight = model.class_weights[OTHER_LABELS].detach().cpu().clone()\n    other_bias = model.class_bias[OTHER_LABELS].detach().cpu().clone()\n    with torch.no_grad():\n        model.class_weights[NAUSEA].copy_(torch.from_numpy(weight).to(model.class_weights.device))\n        model.class_bias[NAUSEA].copy_(torch.from_numpy(bias).to(model.class_bias.device))\n    after = _state_hashes(model)\n    changed = [name for name in before if before[name] != after[name]]\n    if (set(changed) - {"class_weights", "class_bias"}\n            or not torch.equal(other_weight, model.class_weights[OTHER_LABELS].detach().cpu())\n            or not torch.equal(other_bias, model.class_bias[OTHER_LABELS].detach().cpu())):\n        raise RuntimeError("Updating nausea unexpectedly changed frozen model state")\n    model.requires_grad_(False).eval()\n    return {"updated_label_index": NAUSEA, "learned_parameter_count": FEATURE_DIM * 2 + 2,\n            "other_eight_head_slices_identical": True, "all_other_tensors_identical": True,\n            "changed_state_entries": changed, "checked_state_entries": len(before)}\n', 'evidence.py': '"""Learn contextual nausea evidence while preserving the frozen ver19 model.\n\nAll encoder windows are evaluated once per source checkpoint. Exact FP32\nutterance representations are stored as ragged arrays; no turn is truncated.\nOnly the existing nausea evidence module is optimized on original fit labels.\n"""\nfrom __future__ import annotations\n\nimport time\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader\n\nfrom . import data\nfrom .budget import BudgetExceeded\nfrom .head import (FEATURE_DIM, NAUSEA, OTHER_LABELS, _finite, _indices, _key,\n                   _positive_int, _state_hashes, _sync, load_model)\nfrom .neural import EncodeDataset, UtteranceEvidence, seed_everything\n\n\nRECIPES = ("evidence_ce", "evidence_ce_gce")\nSTATE_PREFIX = "initial_evidence__"\nSTATE_KEYS = ("projection.0.weight", "projection.0.bias", "attention.weight",\n              "attention.bias", "output.weight", "output.bias")\nROW_KEYS = ("global_features", "global_logits", "evidence", "logits", "predictions")\n\n\ndef _settings(settings):\n    settings = settings or {}\n    seed = settings.get("seed", 42)\n    if type(seed) is not int or seed < 0:\n        raise ValueError("seed must be a nonnegative integer")\n    result = {\n        "seed": seed,\n        "max_evidence_epochs": _positive_int(settings.get("max_evidence_epochs", 8), "max_evidence_epochs"),\n        "evidence_batch_size": _positive_int(settings.get("evidence_batch_size", 128), "evidence_batch_size"),\n        "evidence_learning_rate": _finite(settings.get("evidence_learning_rate", 1e-4), "evidence_learning_rate", strict=True),\n        "evidence_weight_decay": _finite(settings.get("evidence_weight_decay", .01), "evidence_weight_decay"),\n        "early_stopping_patience": _positive_int(settings.get("early_stopping_patience", 3), "early_stopping_patience"),\n        "positive_weight": _finite(settings.get("positive_weight", 3.), "positive_weight", strict=True),\n        "gce_q": _finite(settings.get("gce_q", .7), "gce_q", strict=True),\n        "gce_mix": _finite(settings.get("gce_mix", .5), "gce_mix"),\n        "ce_warmup_epochs": _positive_int(settings.get("ce_warmup_epochs", 1), "ce_warmup_epochs"),\n    }\n    if result["gce_q"] > 1 or result["gce_mix"] > .5:\n        raise ValueError("GCE requires q <= 1 and mix <= 0.5")\n    return result\n\n\ndef initial_state(cache):\n    return {name: np.asarray(cache[STATE_PREFIX + name]).copy() for name in STATE_KEYS}\n\n\ndef _validate_state(state, encoder_hidden, evidence_hidden=None):\n    if not isinstance(state, dict) or set(state) != set(STATE_KEYS):\n        raise ValueError("Evidence state must contain exactly the original module tensors")\n    weight = np.asarray(state["projection.0.weight"])\n    if weight.ndim != 2 or not 1 <= weight.shape[0] <= 512:\n        raise ValueError("Invalid evidence projection shape")\n    hidden = weight.shape[0]\n    if evidence_hidden is not None and hidden != evidence_hidden:\n        raise ValueError("Evidence state has a different hidden dimension")\n    shapes = {"projection.0.weight": (hidden, encoder_hidden * 3 + FEATURE_DIM),\n              "projection.0.bias": (hidden,), "attention.weight": (1, hidden),\n              "attention.bias": (1,), "output.weight": (2, hidden), "output.bias": (2,)}\n    for name, shape in shapes.items():\n        value = np.asarray(state[name])\n        if value.shape != shape or value.dtype != np.float32 or not np.isfinite(value).all():\n            raise ValueError("Invalid finite FP32 evidence parameter: " + name)\n    return hidden\n\n\ndef _validate_cache(cache):\n    features = np.asarray(cache["global_features"])\n    if features.ndim != 2 or features.shape[1] != FEATURE_DIM or len(features) < 1:\n        raise ValueError("Cache requires nonempty 256-dimensional global features")\n    count = len(features)\n    shapes = {"global_features": (count, FEATURE_DIM), "global_logits": (count, 2),\n              "evidence": (count, 2), "logits": (count, 9, 2)}\n    for name, shape in shapes.items():\n        value = np.asarray(cache[name])\n        if value.shape != shape or value.dtype != np.float32 or not np.isfinite(value).all():\n            raise ValueError("Invalid finite FP32 cache array: " + name)\n    turns, offsets = np.asarray(cache["turn_features"]), np.asarray(cache["turn_offsets"])\n    if (turns.ndim != 2 or turns.shape[1] < 1 or turns.dtype != np.float32\n            or offsets.shape != (count + 1,)\n            or offsets.dtype != np.int64 or offsets[0] != 0\n            or offsets[-1] != len(turns) or np.any(np.diff(offsets) < 0)):\n        raise ValueError("Invalid ragged utterance cache or row offsets")\n    # The full Training cache can exceed 2.5 GiB. Keep validation scratch space\n    # bounded, including when the caller supplies memory-mapped arrays.\n    for start in range(0, len(turns), 8192):\n        if not np.isfinite(turns[start:start + 8192]).all():\n            raise ValueError("Nonfinite cached utterance representation")\n    dropout = np.asarray(cache["evidence_dropout"])\n    if dropout.shape != () or dropout.dtype != np.float32 or not 0 <= float(dropout) < 1:\n        raise ValueError("Invalid evidence dropout configuration")\n    _validate_state(initial_state(cache), turns.shape[1])\n    predicted = data._binary(cache["predictions"], count, "cache predictions")\n    if not np.array_equal(predicted, cache["logits"].argmax(-1)):\n        raise ValueError("Stored predictions differ from raw logit argmax")\n    rebuilt = cache["global_logits"] + cache["evidence"]\n    if (not np.allclose(rebuilt, cache["logits"][:, NAUSEA], rtol=1e-4, atol=1e-5)\n            or not np.array_equal(rebuilt.argmax(-1), predicted[:, NAUSEA])):\n        raise ValueError("Frozen global and evidence logits do not reconstruct nausea")\n    return count\n\n\ndef extract_cache(model, tokenizer, texts, settings, device, budget=None, *, training=True):\n    """Capture original evidence inputs after complete overlap reconstruction."""\n    texts = list(texts)\n    if not texts or any(not isinstance(text, str) for text in texts):\n        raise ValueError("Cache extraction requires nonempty text-only dialogues")\n    if type(training) is not bool:\n        raise ValueError("training must be boolean")\n    if (model.label_attention or model.contrastive or model.evidence is None\n            or tuple(model.class_weights.shape) != (9, FEATURE_DIM, 2)):\n        raise ValueError("Cache extraction requires the original ver19 evidence architecture")\n    batch_size = _positive_int(settings.get("batch_size", 4), "batch_size")\n    model.requires_grad_(False)\n    model.float().to(torch.device(device)).eval()\n    rows = {name: [] for name in ROW_KEYS if name != "predictions"}\n    turn_rows, turn_counts = [], []\n    captured = {}\n\n    def capture_inputs(module, inputs):\n        if captured or len(inputs) != 3:\n            raise RuntimeError("Unexpected evidence input hook invocation")\n        turns, mask, global_features = inputs\n        if (turns.ndim != 3 or mask.shape != turns.shape[:2] or mask.dtype != torch.bool\n                or global_features.shape != (len(turns), FEATURE_DIM)):\n            raise ValueError("Unexpected evidence input shapes")\n        captured.update(turns=turns.detach(), mask=mask.detach(), global_features=global_features.detach())\n\n    def capture_output(module, inputs, output):\n        if output.shape != (len(captured["turns"]), 2) or "evidence" in captured:\n            raise ValueError("Unexpected evidence output shape")\n        captured["evidence"] = output.detach()\n\n    hooks = [model.evidence.register_forward_pre_hook(capture_inputs),\n             model.evidence.register_forward_hook(capture_output)]\n    started, pilot, maximum_error, batches = time.monotonic(), None, 0., 0\n    try:\n        with torch.inference_mode():\n            for start in range(0, len(texts), 64):\n                if budget:\n                    budget.check("tokenize contextual evidence cache", training=training)\n                dataset = EncodeDataset(texts[start:start + 64], None, tokenizer,\n                                        max_length=settings.get("max_length", 512),\n                                        overlap=settings.get("overlap", 64), utterance_evidence=True)\n                loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,\n                                    collate_fn=dataset.collate, num_workers=0)\n                for batch in loader:\n                    if budget:\n                        budget.check("extract contextual evidence cache", training=training)\n                    captured.clear()\n                    batch = {name: value.to(device) for name, value in batch.items()}\n                    logits, _ = model(batch)\n                    if set(captured) != {"turns", "mask", "global_features", "evidence"}:\n                        raise RuntimeError("Evidence capture did not complete")\n                    global_features, residual = captured["global_features"], captured["evidence"]\n                    # Repeat the original full-label kernel, avoiding subtraction\n                    # cancellation when separating the fixed and learned branches.\n                    global_logits = (torch.einsum("bld,ldc->blc",\n                                                  global_features[:, None].expand(-1, 9, -1),\n                                                  model.class_weights) + model.class_bias)[:, NAUSEA]\n                    reconstructed = global_logits + residual\n                    maximum_error = max(maximum_error, float((reconstructed - logits[:, NAUSEA]).abs().max()))\n                    if (not torch.allclose(reconstructed, logits[:, NAUSEA], rtol=1e-4, atol=1e-5)\n                            or not torch.equal(reconstructed.argmax(-1), logits[:, NAUSEA].argmax(-1))):\n                        raise RuntimeError("Cached evidence changed original nausea decisions")\n                    for turns, mask in zip(captured["turns"], captured["mask"]):\n                        valid = turns[mask].float().cpu().numpy().copy()\n                        turn_rows.append(valid)\n                        turn_counts.append(len(valid))\n                    for name, value in (("global_features", global_features), ("global_logits", global_logits),\n                                        ("evidence", residual), ("logits", logits)):\n                        rows[name].append(value.float().cpu().numpy().copy())\n                    batches += 1\n                    if batches % 250 == 0:\n                        print(f"Evidence cache: {len(turn_counts)}/{len(texts)} rows", flush=True)\n                if pilot is None:\n                    _sync(device)\n                    completed = min(64, len(texts))\n                    elapsed = time.monotonic() - started\n                    estimate = elapsed / completed * (len(texts) - completed)\n                    pilot = {"rows": completed, "elapsed_seconds": elapsed,\n                             "estimated_remaining_seconds": estimate, "includes_tokenization": True,\n                             "device": str(device)}\n                    print(f"Evidence cache pilot: {completed} rows, {elapsed:.1f}s; remaining estimate {estimate:.1f}s", flush=True)\n                    if budget and not budget.can_start(estimate, training=training):\n                        raise BudgetExceeded("contextual cache pilot admission",\n                                             deadline_monotonic=budget.deadline_monotonic)\n        result = {name: np.concatenate(values) for name, values in rows.items()}\n        result.update(turn_features=np.concatenate(turn_rows),\n                      turn_offsets=np.concatenate((np.zeros(1, dtype=np.int64),\n                                                   np.cumsum(turn_counts, dtype=np.int64))),\n                      evidence_dropout=np.asarray(model.evidence.projection[2].p, dtype=np.float32))\n        result.update({STATE_PREFIX + name: value.detach().cpu().numpy().copy()\n                       for name, value in model.evidence.state_dict().items()})\n        result["predictions"] = result["logits"].argmax(-1).astype(np.int8)\n        result["_timing"] = {"pilot": pilot, "rows": len(texts), "turns": len(result["turn_features"]),\n                             "elapsed_seconds": time.monotonic() - started,\n                             "maximum_reconstruction_error": maximum_error,\n                             "evaluation_dtype": "float32", "trainable_model_parameters": 0}\n        _validate_cache(result)\n        return result\n    finally:\n        for handle in hooks:\n            handle.remove()\n\n\ndef _subset(cache, indices):\n    indices = _indices(indices, len(cache["global_features"]), "subset")\n    result = {name: value.copy() if isinstance(value, np.ndarray) else value\n              for name, value in cache.items() if name not in ROW_KEYS\n              and name not in ("turn_features", "turn_offsets", "_timing")}\n    result.update({name: cache[name][indices] for name in ROW_KEYS})\n    offsets = cache["turn_offsets"]\n    pieces = [cache["turn_features"][offsets[index]:offsets[index + 1]] for index in indices]\n    result["turn_features"] = np.concatenate(pieces)\n    result["turn_offsets"] = np.concatenate((np.zeros(1, dtype=np.int64),\n                                             np.cumsum([len(piece) for piece in pieces], dtype=np.int64)))\n    return result\n\n\ndef merge_caches(blocks, groups, count):\n    """Scatter separately extracted fit/holdout passes into original row order."""\n    count = _positive_int(count, "count")\n    if not blocks or len(blocks) != len(groups):\n        raise ValueError("Every cache block needs its original row indices")\n    groups = [_indices(group, count, "merge") for group in groups]\n    joined = np.concatenate(groups)\n    if len(joined) != count or len(np.unique(joined)) != count:\n        raise ValueError("Cache groups must partition all original rows exactly once")\n    for block, group in zip(blocks, groups):\n        if _validate_cache(block) != len(group):\n            raise ValueError("Cache rows do not match their original indices")\n        if (block["turn_features"].shape[1] != blocks[0]["turn_features"].shape[1]\n                or not np.array_equal(block["evidence_dropout"], blocks[0]["evidence_dropout"])\n                or any(not np.array_equal(block[STATE_PREFIX + name], blocks[0][STATE_PREFIX + name])\n                       for name in STATE_KEYS)):\n            raise ValueError("Cache blocks came from different evidence modules")\n    result = {STATE_PREFIX + name: blocks[0][STATE_PREFIX + name].copy() for name in STATE_KEYS}\n    result["evidence_dropout"] = blocks[0]["evidence_dropout"].copy()\n    for name in ROW_KEYS:\n        result[name] = np.empty((count, *blocks[0][name].shape[1:]), dtype=blocks[0][name].dtype)\n        for block, group in zip(blocks, groups):\n            result[name][group] = block[name]\n    counts = np.empty(count, dtype=np.int64)\n    for block, group in zip(blocks, groups):\n        counts[group] = np.diff(block["turn_offsets"])\n    offsets = np.concatenate((np.zeros(1, dtype=np.int64), np.cumsum(counts)))\n    result["turn_offsets"] = offsets\n    result["turn_features"] = np.empty((offsets[-1], blocks[0]["turn_features"].shape[1]), dtype=np.float32)\n    for block, group in zip(blocks, groups):\n        for row, destination in enumerate(group):\n            result["turn_features"][offsets[destination]:offsets[destination + 1]] = block["turn_features"][\n                block["turn_offsets"][row]:block["turn_offsets"][row + 1]]\n    _validate_cache(result)\n    return result\n\n\ndef _new_module(cache, state, device):\n    hidden = _validate_state(state, cache["turn_features"].shape[1])\n    module = UtteranceEvidence(cache["turn_features"].shape[1], FEATURE_DIM, hidden,\n                               float(cache["evidence_dropout"]))\n    module.load_state_dict({name: torch.from_numpy(np.asarray(value).copy()) for name, value in state.items()}, strict=True)\n    return module.float().to(torch.device(device))\n\n\ndef _batch(cache, indices, device):\n    offsets = cache["turn_offsets"]\n    counts = offsets[indices + 1] - offsets[indices]\n    width = max(1, int(counts.max()))\n    turns = np.zeros((len(indices), width, cache["turn_features"].shape[1]), dtype=np.float32)\n    mask = np.arange(width)[None] < counts[:, None]\n    for row, index in enumerate(indices):\n        turns[row, :counts[row]] = cache["turn_features"][offsets[index]:offsets[index + 1]]\n    return (torch.from_numpy(turns).to(device), torch.from_numpy(mask).to(device),\n            torch.from_numpy(cache["global_features"][indices]).to(device),\n            torch.from_numpy(cache["global_logits"][indices]).to(device))\n\n\ndef _predict(module, cache, options, device, budget=None, *, training=False):\n    module.eval()\n    result = cache["logits"].copy()\n    with torch.inference_mode():\n        for start in range(0, len(result), options["evidence_batch_size"]):\n            if budget:\n                budget.check("evaluate contextual evidence", training=training)\n            indices = np.arange(start, min(len(result), start + options["evidence_batch_size"]))\n            turns, mask, features, global_logits = _batch(cache, indices, device)\n            logits = global_logits + module(turns, mask, features)\n            if not bool(torch.isfinite(logits).all()):\n                raise FloatingPointError("Evidence module produced nonfinite logits")\n            result[indices, NAUSEA] = logits.cpu().numpy()\n    predicted = result.argmax(-1).astype(np.int8)\n    if (not np.array_equal(result[:, OTHER_LABELS], cache["logits"][:, OTHER_LABELS])\n            or not np.array_equal(predicted[:, OTHER_LABELS], cache["predictions"][:, OTHER_LABELS])):\n        raise RuntimeError("A frozen symptom output changed")\n    return result, predicted\n\n\ndef apply_evidence(cache, state, settings=None, device="cpu", budget=None, *, training=False):\n    _validate_cache(cache)\n    module = _new_module(cache, state, device)\n    return _predict(module, cache, _settings(settings), device, budget, training=training)\n\n\ndef evidence_loss(logits, targets, candidate_id, *, epoch=1, positive_weight=3., gce_q=.7,\n                  gce_mix=.5, ce_warmup_epochs=1):\n    """Original labels supervise ordinary CE or the predefined CE/GCE mixture."""\n    if candidate_id not in RECIPES:\n        raise ValueError("Unknown evidence objective")\n    options = _settings({"positive_weight": positive_weight, "gce_q": gce_q, "gce_mix": gce_mix,\n                         "ce_warmup_epochs": ce_warmup_epochs})\n    epoch = _positive_int(epoch, "epoch")\n    if (logits.ndim != 2 or logits.shape[1] != 2 or len(logits) < 1\n            or targets.shape != (len(logits),) or targets.dtype != torch.long\n            or not bool(((targets == 0) | (targets == 1)).all())\n            or not bool(torch.isfinite(logits).all())):\n        raise ValueError("Evidence loss requires finite Nx2 logits and binary long targets")\n    weights = torch.where(targets.bool(), logits.new_tensor(options["positive_weight"]), logits.new_tensor(1.))\n    ce = F.cross_entropy(logits, targets, reduction="none")\n    log_probability = F.log_softmax(logits, dim=-1).gather(1, targets[:, None]).squeeze(1)\n    # The log-domain form retains finite gradients even when a wrong-class\n    # probability would underflow to zero before a fractional power.\n    gce = -torch.expm1(options["gce_q"] * log_probability) / options["gce_q"]\n    mix = options["gce_mix"] if candidate_id == "evidence_ce_gce" and epoch > options["ce_warmup_epochs"] else 0.\n    loss = (weights * ((1 - mix) * ce + mix * gce)).mean()\n    return loss, {"weighted_ce": float((weights * ce).mean().detach()),\n                  "weighted_gce": float((weights * gce).mean().detach()), "gce_mix_applied": mix}\n\n\ndef _snapshot(module):\n    return {name: value.detach().cpu().numpy().copy() for name, value in module.state_dict().items()}\n\n\ndef _new_training(cache, options, device):\n    seed_everything(options["seed"])\n    module = _new_module(cache, initial_state(cache), device)\n    optimizer = torch.optim.AdamW(module.parameters(), lr=options["evidence_learning_rate"],\n                                  weight_decay=options["evidence_weight_decay"])\n    return module, optimizer, np.random.default_rng(options["seed"])\n\n\ndef _train_epoch(module, optimizer, rng, cache, targets, indices, candidate_id, epoch, options, device, budget):\n    module.train()\n    order = rng.permutation(indices)\n    totals = {"loss": 0., "weighted_ce": 0., "weighted_gce": 0.}\n    steps, mix = 0, 0.\n    started = time.monotonic()\n    for start in range(0, len(order), options["evidence_batch_size"]):\n        if budget:\n            budget.check(f"{candidate_id} epoch {epoch}", training=True)\n        chosen = order[start:start + options["evidence_batch_size"]]\n        turns, mask, features, global_logits = _batch(cache, chosen, device)\n        truth = torch.from_numpy(targets[chosen, NAUSEA].astype(np.int64)).to(device)\n        optimizer.zero_grad(set_to_none=True)\n        loss, details = evidence_loss(global_logits + module(turns, mask, features), truth, candidate_id,\n                                      epoch=epoch, **{key: options[key] for key in\n                                                       ("positive_weight", "gce_q", "gce_mix", "ce_warmup_epochs")})\n        if not bool(torch.isfinite(loss)):\n            raise FloatingPointError("Nonfinite contextual evidence loss")\n        loss.backward()\n        if any(parameter.grad is None or not bool(torch.isfinite(parameter.grad).all()) for parameter in module.parameters()):\n            raise FloatingPointError("Missing or nonfinite contextual evidence gradient")\n        torch.nn.utils.clip_grad_norm_(module.parameters(), max_norm=1., error_if_nonfinite=True)\n        optimizer.step()\n        for name, value in {"loss": float(loss.detach()), "weighted_ce": details["weighted_ce"],\n                            "weighted_gce": details["weighted_gce"]}.items():\n            totals[name] += value * len(chosen)\n        steps += 1\n        mix = details["gce_mix_applied"]\n    _sync(device)\n    return {name: value / len(order) for name, value in totals.items()} | {\n        "gce_mix_applied": mix, "optimizer_steps": steps, "fitted_samples": len(order),\n        "elapsed_seconds": time.monotonic() - started}\n\n\ndef evidence_candidates(cache, targets, fit, holdout, settings, budget=None, device="cpu"):\n    count = _validate_cache(cache)\n    targets = data._binary(targets, count, "targets")\n    fit, holdout = _indices(fit, count, "fit"), _indices(holdout, count, "holdout")\n    if len(np.intersect1d(fit, holdout)) or len(fit) + len(holdout) != count:\n        raise ValueError("Fit and holdout must form a disjoint complete partition")\n    options = _settings(settings)\n    held_cache = _subset(cache, holdout)\n    baseline_predictions = held_cache["predictions"].copy()\n    baseline = data.score_predictions(targets[holdout], baseline_predictions)\n    records = [{"candidate_id": "baseline", "epoch": 0, "metrics": baseline,\n                "state": initial_state(cache), "predictions": baseline_predictions, "history": [],\n                "improved": False, "stop_reason": "unchanged_original_evidence", "trainable_parameters": 0}]\n    epoch_estimate = None\n    for candidate_id in RECIPES:\n        module, optimizer, rng = _new_training(cache, options, device)\n        history, best, completed, stale = [], None, 0, 0\n        stop_reason = "max_evidence_epochs"\n        for epoch in range(1, options["max_evidence_epochs"] + 1):\n            try:\n                if budget and epoch_estimate is not None and not budget.can_start(epoch_estimate, training=True):\n                    raise BudgetExceeded("contextual evidence epoch admission",\n                                         deadline_monotonic=budget.deadline_monotonic)\n                epoch_started = time.monotonic()\n                details = _train_epoch(module, optimizer, rng, cache, targets, fit, candidate_id,\n                                       epoch, options, device, budget)\n                _, predicted = _predict(module, held_cache, options, device, budget, training=True)\n            except BudgetExceeded:\n                stop_reason = "budget"\n                break\n            completed = epoch\n            _sync(device)\n            epoch_estimate = time.monotonic() - epoch_started\n            metrics = data.score_predictions(targets[holdout], predicted)\n            history.append({"epoch": epoch, **details, "metrics": metrics,\n                            "training_and_evaluation_seconds": epoch_estimate})\n            if best is None or _key(metrics, baseline) > _key(best["metrics"], baseline):\n                best = {"epoch": epoch, "metrics": metrics, "state": _snapshot(module),\n                        "predictions": predicted, "improved": _key(metrics, baseline)[0]}\n                stale = 0\n            else:\n                stale += 1\n            print(f"{candidate_id} epoch {epoch}: holdout Macro F1={metrics[\'macro_f1\']:.6f}; "\n                  f"nausea F1={metrics[\'per_symptom\'][data.TARGET_SYMPTOMS[NAUSEA]][\'f1\']:.6f}", flush=True)\n            if stale >= options["early_stopping_patience"]:\n                stop_reason = "early_stopping"\n                break\n        if best is not None:\n            records.append({"candidate_id": candidate_id, **best, "history": history,\n                            "completed_epochs": completed, "stop_reason": stop_reason,\n                            "trainable_parameters": sum(parameter.numel() for parameter in module.parameters())})\n        if stop_reason == "budget":\n            records[0]["search_stop_reason"] = "budget"\n            break\n    return records\n\n\ndef refit_evidence(cache, targets, candidate_id, epochs, settings, budget=None, device="cpu"):\n    """Refit the fixed recipe from the separate full-Training source branch."""\n    count = _validate_cache(cache)\n    targets = data._binary(targets, count, "targets")\n    if candidate_id not in RECIPES:\n        raise ValueError("Only a selected evidence recipe can be refitted")\n    options = _settings(settings)\n    epochs = _positive_int(epochs, "epochs")\n    if epochs > options["max_evidence_epochs"]:\n        raise ValueError("Refit epochs exceed the declared training budget")\n    module, optimizer, rng = _new_training(cache, options, device)\n    history = []\n    for epoch in range(1, epochs + 1):\n        details = _train_epoch(module, optimizer, rng, cache, targets, np.arange(count), candidate_id,\n                               epoch, options, device, budget)\n        history.append({"epoch": epoch, **details})\n        if budget and epoch < epochs and not budget.can_start(details["elapsed_seconds"] * (epochs - epoch), training=True):\n            raise BudgetExceeded("remaining full-Training evidence refit admission",\n                                 deadline_monotonic=budget.deadline_monotonic)\n    return _snapshot(module), history\n\n\ndef set_evidence_state(model, state):\n    """Change only the pre-existing nausea evidence tensors, with hash proof."""\n    if model.evidence is None or tuple(model.class_weights.shape) != (9, FEATURE_DIM, 2):\n        raise ValueError("Cannot patch an incompatible model")\n    _validate_state(state, int(model.encoder.config.hidden_size), model.evidence.projection[0].out_features)\n    before = _state_hashes(model)\n    model.evidence.load_state_dict({name: torch.from_numpy(np.asarray(value).copy()) for name, value in state.items()}, strict=True)\n    after = _state_hashes(model)\n    changed = [name for name in before if before[name] != after[name]]\n    if any(not name.startswith("evidence.") for name in changed):\n        raise RuntimeError("Updating evidence changed a frozen model tensor")\n    model.requires_grad_(False).eval()\n    return {"updated_label_index": NAUSEA,\n            "learned_parameter_count": sum(parameter.numel() for parameter in model.evidence.parameters()),\n            "other_eight_head_slices_identical": True, "all_global_heads_identical": True,\n            "all_non_evidence_tensors_identical": True, "changed_state_entries": changed,\n            "checked_state_entries": len(before)}\n', 'source_bundle.py': '"""Recover only the pinned ver19 Training models; never execute source code."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport stat\nimport uuid\nimport zipfile\n\nimport numpy as np\n\nfrom . import data\n\n\nREFERENCE_PATH = Path(__file__).with_name("source_reference.json")\nREFERENCE_FILE_SHA256 = "efcae59b68909dc1ba94e6ad954cfd3ec6ace8c4ffb2d4f25ae1f5972ea855ef"\nCHUNK_SIZE = 4 * 1024 * 1024\nMAX_ZIP_MEMBERS = 100000\n\n\ndef _check(budget, stage):\n    if budget is not None:\n        budget.check(stage)\n\n\ndef _equal(actual, expected, name):\n    if actual != expected:\n        raise ValueError(f"ver19 source mismatch: {name}")\n\n\ndef _relative(name):\n    """Accept one canonical, portable relative path, without normalization."""\n    if (not isinstance(name, str) or not name or len(name) > 1024\n            or "\\\\" in name or ":" in name or "\\0" in name):\n        raise ValueError("Unsafe source path")\n    path = PurePosixPath(name)\n    if (path.is_absolute() or path.as_posix() != name\n            or any(part in {"", ".", ".."} for part in path.parts)\n            or any(part.rstrip(" .") != part for part in path.parts)):\n        raise ValueError("Unsafe source path")\n    return path\n\n\ndef _safe_file(root, relative):\n    root = Path(root).resolve()\n    path = root.joinpath(*_relative(relative).parts)\n    if not path.resolve().is_relative_to(root):\n        raise ValueError("Source file escapes its root")\n    for part in (path, *path.parents):\n        if part == root:\n            break\n        if part.is_symlink():\n            raise ValueError("Source symlinks are not permitted")\n    return path\n\n\ndef _hash_file(path, budget=None):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        while True:\n            _check(budget, "source checksum")\n            block = handle.read(CHUNK_SIZE)\n            if not block:\n                break\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _load_reference():\n    raw = REFERENCE_PATH.read_bytes()\n    _equal(hashlib.sha256(raw).hexdigest(), REFERENCE_FILE_SHA256, "pinned reference checksum")\n    reference = json.loads(raw.decode("utf-8"))\n    # Reuse the exact immutable ver19 reference introduced in ver21.\n    _equal(reference["format"], "mission3_ver21_pinned_source_v1", "reference format")\n    _equal(reference["version"], "ver19", "source version")\n    _equal(reference["labels"], list(data.TARGET_SYMPTOMS), "label order")\n    _equal(reference["decision_rule"], "categorical predict/argmax", "decision rule")\n    _equal(reference["provenance"]["validation_accessed"], False, "Training-only provenance")\n    for name in reference["files"]:\n        _relative(name)\n    return reference\n\n\ndef _verify_files(root, reference, budget=None):\n    verified = {}\n    for name, expected in reference["files"].items():\n        path = _safe_file(root, name)\n        if not path.is_file():\n            raise ValueError(f"Required ver19 source file is missing: {name}")\n        _equal(path.stat().st_size, expected["size"], f"file size ({name})")\n        digest = _hash_file(path, budget)\n        _equal(digest, expected["sha256"], f"file checksum ({name})")\n        verified[name] = digest\n    _verify_manifests(root, reference)\n    return verified\n\n\ndef _verify_manifests(root, reference):\n    for key, scope, count in (("fit_checkpoint", "fit_only", reference["fit_samples"]),\n                              ("full_checkpoint", "all_training", reference["training_samples"])):\n        relative = reference[key]\n        manifest = json.loads(_safe_file(root, relative).read_text(encoding="utf-8"))\n        for field, expected in (("version", "ver19"), ("training_scope", scope),\n                                ("fitted_samples", count), ("labels", reference["labels"]),\n                                ("decision_rule", reference["decision_rule"]),\n                                ("model_id", "klue/roberta-base"),\n                                ("resolved_revision", "02f94ba5e3fcb7e2a58a390b8639b0fac974a8da"),\n                                ("utterance_evidence", True), ("loss_mode", "nausea_ce_gce")):\n            _equal(manifest.get(field), expected, f"{key} {field}")\n        parent = PurePosixPath(relative).parent\n        weight = (parent / _relative(manifest["weights"])).as_posix()\n        assets = (parent / _relative(manifest["assets"])).as_posix()\n        if weight not in reference["files"] or not any(p.startswith(assets + "/") for p in reference["files"]):\n            raise ValueError("Unpinned model weights or assets")\n\n\ndef _zip_inventory(archive):\n    """Validate every member before selecting a small, exact-size whitelist."""\n    infos = archive.infolist()\n    if len(infos) > MAX_ZIP_MEMBERS:\n        raise ValueError("ver19 ZIP contains too many members")\n    names = {}\n    for info in infos:\n        if info.orig_filename != info.filename:\n            raise ValueError("ver19 ZIP contains a normalized or truncated unsafe path")\n        name = info.filename[:-1] if info.is_dir() else info.filename\n        _relative(name)\n        mode = info.external_attr >> 16\n        if (stat.S_ISLNK(mode) or stat.S_IFMT(mode) not in {0, stat.S_IFREG, stat.S_IFDIR}\n                or bool(info.flag_bits & 1)):\n            raise ValueError("ver19 ZIP contains a symlink, special file, or encrypted member")\n        if stat.S_ISDIR(mode) and not info.is_dir():\n            raise ValueError("ver19 ZIP directory type mismatch")\n        folded = name.casefold()\n        if folded in names:\n            raise ValueError("ver19 ZIP contains duplicate paths")\n        names[folded] = (name, info)\n    for name, info in names.values():\n        for parent in PurePosixPath(name).parents:\n            if str(parent) == ".":\n                break\n            entry = names.get(parent.as_posix().casefold())\n            if entry and not entry[1].is_dir():\n                raise ValueError("ver19 ZIP contains a file/directory collision")\n    return {name: info for name, info in names.values()}\n\n\ndef _zip_selection(archive, reference):\n    inventory = _zip_inventory(archive)\n    marker = reference["fit_checkpoint"]\n    prefixes = []\n    for name in inventory:\n        if name == marker:\n            prefixes.append("")\n        elif name.endswith("/" + marker):\n            prefixes.append(name[:-len(marker)])\n    if len(prefixes) != 1:\n        raise ValueError("ver19 ZIP must contain exactly one matching source run")\n    prefix = prefixes[0]\n    selected = {}\n    for relative, expected in reference["files"].items():\n        info = inventory.get(prefix + relative)\n        if info is None or info.is_dir():\n            raise ValueError(f"ver19 ZIP is missing a required source file: {relative}")\n        _equal(info.file_size, expected["size"], f"ZIP member size ({relative})")\n        selected[relative] = info\n    return selected\n\n\ndef _copy_stream(handle, destination, expected, budget=None):\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    digest, count = hashlib.sha256(), 0\n    with destination.open("xb") as output:\n        while True:\n            _check(budget, "source copy")\n            block = handle.read(min(CHUNK_SIZE, expected["size"] - count + 1))\n            if not block:\n                break\n            count += len(block)\n            if count > expected["size"]:\n                raise ValueError("Source stream exceeds its pinned size")\n            output.write(block)\n            digest.update(block)\n    _equal(count, expected["size"], "copied file size")\n    _equal(digest.hexdigest(), expected["sha256"], "copied file checksum")\n    return digest.hexdigest()\n\n\ndef _copy_source(location, destination, reference, budget=None):\n    verified = {}\n    if location.is_dir():\n        for relative, expected in reference["files"].items():\n            path = _safe_file(location, relative)\n            if not path.is_file():\n                raise ValueError(f"Required ver19 source file is missing: {relative}")\n            _equal(path.stat().st_size, expected["size"], f"source size ({relative})")\n            with path.open("rb") as handle:\n                verified[relative] = _copy_stream(handle, _safe_file(destination, relative), expected, budget)\n    elif location.is_file() and location.suffix.lower() == ".zip":\n        with zipfile.ZipFile(location) as archive:\n            selected = _zip_selection(archive, reference)\n            for relative, info in selected.items():\n                with archive.open(info) as handle:\n                    verified[relative] = _copy_stream(handle, _safe_file(destination, relative), reference["files"][relative], budget)\n    else:\n        raise ValueError("ver19 source must be a run directory or its ZIP archive")\n    _verify_manifests(destination, reference)\n    return verified\n\n\ndef _directory_runs(root, reference, budget=None):\n    """Bounded discovery of run wrappers without traversing data/model files."""\n    root = Path(root)\n    if not root.is_dir() or root.is_symlink():\n        return []\n    marker = reference["fit_checkpoint"]\n    found, pending = [], [(root, 0)]\n    while pending:\n        directory, depth = pending.pop(0)\n        _check(budget, "source discovery")\n        if (directory / marker).is_file():\n            found.append(directory)\n            continue\n        if depth >= 4:\n            continue\n        for child in sorted(directory.iterdir(), key=lambda p: p.name):\n            if child.is_dir() and not child.is_symlink() and (\n                    depth == 0 or child.name.startswith(("ver19", reference["run_id"]))):\n                pending.append((child, depth + 1))\n    return found\n\n\ndef _candidate_locations(source_location, reference, budget=None, search_roots=None):\n    if source_location:\n        path = Path(source_location).expanduser()\n        if path.is_dir():\n            runs = _directory_runs(path, reference, budget)\n            if not runs:\n                raise FileNotFoundError("The selected directory does not contain the required complete ver19 run")\n            return runs\n        if path.is_file() and path.suffix.lower() == ".zip":\n            return [path]\n        raise FileNotFoundError("ver19 source was not found; choose the complete ver19 result directory or ZIP")\n    roots = ([Path(p) for p in search_roots] if search_roots is not None else [\n        Path("/content/drive/MyDrive/DCC_AIdeas/results/mission3"),\n        Path("/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver19"),\n        Path("/content/drive/MyDrive"),\n        Path.cwd() / "results/mission3",\n    ])\n    found = []\n    for root in roots:\n        _check(budget, "source discovery")\n        if not root.is_dir():\n            continue\n        # Search only known project/result wrappers and ver19 named children.\n        candidates = [root] if (search_roots is not None or root.name.startswith("ver19") or root.name == "mission3") else [\n            root / "DCC_AIdeas/results/mission3", *sorted(root.glob("ver19*")),\n        ]\n        for path in candidates:\n            if path.is_file() and path.suffix.lower() == ".zip":\n                found.append(path)\n            elif path.is_dir():\n                found.extend(_directory_runs(path, reference, budget))\n                found.extend(sorted(path.glob("ver19*.zip")))\n    return list(dict.fromkeys(path.resolve() for path in found))\n\n\ndef _result(root, reference, verified):\n    return {"root": root, "fit_checkpoint": root / reference["fit_checkpoint"],\n            "full_checkpoint": root / reference["full_checkpoint"], "reference": reference,\n            "source_hashes_verified": verified}\n\n\ndef _prepare_source(source_location, work_dir, reference, reference_hash, budget=None, search_roots=None):\n    """Internal injectable helper for synthetic tests; public entry pins reference."""\n    work = Path(work_dir).resolve()\n    work.mkdir(parents=True, exist_ok=True)\n    prefix = "source_ver19_" + reference_hash[:16] + "_"\n    if not source_location:\n        for cached in sorted(work.glob(prefix + "*")):\n            if not cached.is_dir() or cached.is_symlink() or cached.name.endswith(".partial"):\n                continue\n            marker = cached / "source_complete.json"\n            if marker.is_file() and not marker.is_symlink():\n                try:\n                    complete = json.loads(marker.read_text(encoding="utf-8"))\n                    _equal(complete["reference_sha256"], reference_hash, "cache reference")\n                    verified = _verify_files(cached, reference, budget)\n                    _equal(complete["artifact_hashes"], verified, "cache artifact hashes")\n                    return _result(cached, reference, verified)\n                except (ValueError, KeyError, OSError) as error:\n                    if isinstance(error, TimeoutError):\n                        raise\n                    continue\n    candidates = _candidate_locations(source_location, reference, budget, search_roots)\n    if not candidates:\n        raise FileNotFoundError("Required ver19 result not found. Keep the complete ver19 result folder or ZIP in DCC_AIdeas/results/mission3, or set SETTINGS[\'source_run\'] explicitly.")\n    failures = []\n    for candidate in candidates:\n        temporary = work / (prefix + uuid.uuid4().hex[:12] + ".partial")\n        temporary.mkdir()\n        try:\n            verified = _copy_source(candidate, temporary, reference, budget)\n            marker = {"format": "mission3_ver22_source_cache_v1", "reference_sha256": reference_hash,\n                      "artifact_hashes": verified, "validation_files_copied": False}\n            (temporary / "source_complete.json").write_text(json.dumps(marker, indent=2) + "\\n", encoding="utf-8")\n            finished = temporary.with_suffix("")\n            os.replace(temporary, finished)\n            return _result(finished, reference, verified)\n        except (ValueError, OSError, zipfile.BadZipFile, RuntimeError) as error:\n            if isinstance(error, TimeoutError):\n                raise\n            failures.append(type(error).__name__ + ": " + str(error))\n    raise ValueError("No candidate matched the pinned ver19 source. " + "; ".join(failures[:3]))\n\n\ndef prepare_source(source_location, work_dir, budget=None, search_roots=None):\n    """Return a local, hash-verified minimal ver19 bundle from a folder or ZIP.\n\n    Selection MUST use ``fit_checkpoint``. ``full_checkpoint`` is reserved for\n    refitting a chosen head after Training-holdout selection is complete.\n    Incomplete caches are never reused and Drive originals are never modified.\n    """\n    reference = _load_reference()\n    return _prepare_source(source_location, work_dir, reference, REFERENCE_FILE_SHA256,\n                           budget=budget, search_roots=search_roots)\n\n\ndef array_digest(values, dtype):\n    values = np.ascontiguousarray(values, dtype=np.dtype(dtype))\n    digest = hashlib.sha256(json.dumps(list(values.shape)).encode("ascii") + b"\\0")\n    digest.update(values.tobytes())\n    return digest.hexdigest()\n\n\ndef dataset_digest(texts, targets):\n    """Exactly the ver19 runtime text/order/target digest, without identifiers."""\n    if len(texts) != len(targets):\n        raise ValueError("Training texts and targets must have equal lengths")\n    digest = hashlib.sha256()\n    for text, target in zip(texts, targets):\n        digest.update(text.encode("utf-8"))\n        digest.update(b"\\0")\n        digest.update(np.asarray(target, dtype=np.int8).tobytes())\n    return digest.hexdigest()\n\n\ndef _indices(values, total, name):\n    values = np.asarray(values)\n    if (values.ndim != 1 or values.dtype.kind not in "iu" or not len(values)\n            or (values < 0).any() or (values >= total).any()\n            or len(np.unique(values)) != len(values)):\n        raise ValueError(f"Invalid Training {name} indices")\n    return values.astype("<i8")\n\n\ndef verify_training(texts, targets, fit, holdout, source):\n    """Bind both source scopes and historical predictions to exact Training rows."""\n    reference = _load_reference()\n    _equal(source["reference"], reference, "prepared source reference")\n    root = Path(source["root"])\n    targets = data._binary(targets, reference["training_samples"], "targets")\n    fit = _indices(fit, len(targets), "fit")\n    holdout = _indices(holdout, len(targets), "holdout")\n    if len(fit) + len(holdout) != len(targets) or len(np.union1d(fit, holdout)) != len(targets):\n        raise ValueError("Training fit and holdout must partition all rows")\n    _equal(len(fit), reference["fit_samples"], "fit sample count")\n    _equal(len(holdout), reference["holdout_samples"], "holdout sample count")\n    _equal(dataset_digest(texts, targets), reference["training_digest"], "Training text/target/order digest")\n    hashes = reference["array_hashes"]\n    for name, values, dtype in (("fit_indices", fit, "<i8"), ("holdout_indices", holdout, "<i8"),\n                                ("training_targets", targets, "i1"), ("holdout_targets", targets[holdout], "i1")):\n        _equal(array_digest(values, dtype), hashes[name], name)\n    # Recheck small source arrays at use, even after a complete cache was loaded.\n    for name in ("training_split.npz", "holdout_predictions.npz", "run_identity.json"):\n        _equal(_hash_file(_safe_file(root, name)), reference["files"][name]["sha256"], name + " checksum")\n    with np.load(root / "training_split.npz", allow_pickle=False) as saved:\n        if not np.array_equal(fit, saved["fit"]) or not np.array_equal(holdout, saved["holdout"]):\n            raise ValueError("Stored ver19 Training split does not match")\n    with np.load(root / "holdout_predictions.npz", allow_pickle=False) as saved:\n        if not np.array_equal(holdout, saved["indices"]) or not np.array_equal(targets[holdout], saved["targets"]):\n            raise ValueError("Stored ver19 holdout indices or targets do not match")\n        predictions = data._binary(saved["predictions"], len(holdout))\n        _equal(array_digest(predictions, "i1"), hashes["predictions"], "holdout prediction hash")\n        metrics = data.score_predictions(targets[holdout], predictions)\n    _equal(metrics, reference["metrics"], "recomputed Training metrics")\n    identity = json.loads((root / "run_identity.json").read_text(encoding="utf-8"))\n    _equal(identity["training_digest"], reference["training_digest"], "source Training identity")\n    _verify_manifests(root, reference)\n    return {"verified": True, "version": "ver19", "source_run_id": reference["run_id"],\n            "training_digest": reference["training_digest"], "fit_samples": len(fit),\n            "holdout_samples": len(holdout), "reference_file_sha256": REFERENCE_FILE_SHA256,\n            "baseline_metrics": metrics, "validation_accessed": False,\n            "selection_source_scope": "fit_only", "refit_source_scope": "all_training"}\n', 'source_reference.json': '{\n  "version": "ver19",\n  "candidate_id": "nausea_gce",\n  "run_id": "20261008T072419Z_07dc5385",\n  "labels": [\n    "고열",\n    "구토",\n    "두통",\n    "복통",\n    "어지러움",\n    "열상",\n    "오심",\n    "전신쇠약",\n    "호흡곤란"\n  ],\n  "split_seed": 42,\n  "holdout_size": 0.2,\n  "training_samples": 29200,\n  "fit_samples": 23360,\n  "holdout_samples": 5840,\n  "training_digest": "1d0c54e5f27b507663f5690fce7c2fd350b20b33f4bafe2aabbfd4370bf30525",\n  "array_hashes": {\n    "fit_indices": "08b71a25fd798963eec81d98775112a6355d88be110c37e33d5adf37685cb437",\n    "holdout_indices": "20f8fcb447a5ca4de0434da94680da48a58b46f6c2c2e03139b60f8738e5b0ed",\n    "training_targets": "416fe9b7e8cf31471124b89a41fcc14255d12bc8441970b4d6203957fb286074",\n    "holdout_targets": "e43046eb56f91c77a261820c1ccf5151f5adf1911fbe45ff41f953e6cc307d7e",\n    "predictions": "b470a8060e202fe6749c35e7779d2354530821039762e576a41e9fefd550101b"\n  },\n  "metrics": {\n    "macro_f1": 0.6412755833526849,\n    "micro_f1": 0.6520991272217621,\n    "samples": 5840,\n    "per_symptom": {\n      "고열": {\n        "f1": 0.6606129398410897,\n        "precision": 0.8027586206896552,\n        "recall": 0.5612343297974928,\n        "support": 1037,\n        "false_positive": 143,\n        "false_negative": 455\n      },\n      "구토": {\n        "f1": 0.5701058201058201,\n        "precision": 0.6962843295638126,\n        "recall": 0.4826427771556551,\n        "support": 893,\n        "false_positive": 188,\n        "false_negative": 462\n      },\n      "두통": {\n        "f1": 0.531021897810219,\n        "precision": 0.5650485436893203,\n        "recall": 0.5008605851979346,\n        "support": 581,\n        "false_positive": 224,\n        "false_negative": 290\n      },\n      "복통": {\n        "f1": 0.8100759696121551,\n        "precision": 0.8831734960767219,\n        "recall": 0.7481536189069424,\n        "support": 1354,\n        "false_positive": 134,\n        "false_negative": 341\n      },\n      "어지러움": {\n        "f1": 0.6765531062124248,\n        "precision": 0.6872964169381107,\n        "recall": 0.6661404893449092,\n        "support": 1267,\n        "false_positive": 384,\n        "false_negative": 423\n      },\n      "열상": {\n        "f1": 0.8862094951017332,\n        "precision": 0.9274447949526814,\n        "recall": 0.8484848484848485,\n        "support": 693,\n        "false_positive": 46,\n        "false_negative": 105\n      },\n      "오심": {\n        "f1": 0.3708523096942095,\n        "precision": 0.32796317606444186,\n        "recall": 0.42664670658682635,\n        "support": 668,\n        "false_positive": 584,\n        "false_negative": 383\n      },\n      "전신쇠약": {\n        "f1": 0.5856079404466501,\n        "precision": 0.6190975865687304,\n        "recall": 0.5555555555555556,\n        "support": 1062,\n        "false_positive": 363,\n        "false_negative": 472\n      },\n      "호흡곤란": {\n        "f1": 0.6804407713498623,\n        "precision": 0.7406296851574213,\n        "recall": 0.6292993630573248,\n        "support": 785,\n        "false_positive": 173,\n        "false_negative": 291\n      }\n    }\n  },\n  "format": "mission3_ver21_pinned_source_v1",\n  "decision_rule": "categorical predict/argmax",\n  "fit_checkpoint": "experiments/nausea_gce/20261008T080049Z_74b9b8c0/candidate_model.ckpt",\n  "full_checkpoint": "final_model/20261008T083738Z_a6a9650c/final_model.ckpt",\n  "files": {\n    "run_identity.json": {\n      "sha256": "507e1030d9749fa1ea39e5b68f7966d76bfee44d88901b377517e43e5f612f51",\n      "size": 601\n    },\n    "training_split.npz": {\n      "sha256": "524a04326803a3eb41696b322a9a76e20f8afcfdbcccd37ebe089ebc6f77e264",\n      "size": 44889\n    },\n    "holdout_predictions.npz": {\n      "sha256": "84e2d1f158cf83d6000055d5ead377a5c479bb034fe054037fdea208b91888b1",\n      "size": 21368\n    },\n    "final_selection.json": {\n      "sha256": "cd3de6fc7eb388ff5a359a11bcd43d50690fbbb8cf43934fed3fdeb17ad0524b",\n      "size": 2388\n    },\n    "training_metrics.json": {\n      "sha256": "6e70346a77b296613a6dfe3927b3c65443bb38c3ad26d344539bb178e273695b",\n      "size": 2015\n    },\n    "experiments/nausea_gce/20261008T080049Z_74b9b8c0/candidate_model.ckpt": {\n      "sha256": "b295e859412a6ff26b34ab80ace13692e63077e1522d72bc8bd248cbcdeb728d",\n      "size": 1258\n    },\n    "experiments/nausea_gce/20261008T080049Z_74b9b8c0/selection_weights.pt": {\n      "sha256": "7cf8eb27e79dae667ee146f05ccc25a50f5048a0b201110f52a86ac86131bd11",\n      "size": 446041310\n    },\n    "experiments/nausea_gce/20261008T080049Z_74b9b8c0/assets/encoder/config.json": {\n      "sha256": "1d8a2d54cd8b25538c3d991c5a05f516718682a4f5b84dfc1045c8ba84362259",\n      "size": 705\n    },\n    "experiments/nausea_gce/20261008T080049Z_74b9b8c0/assets/tokenizer/special_tokens_map.json": {\n      "sha256": "63cd5b53789b3b5ae2c3d7aa377bcba1557ff0b41dc69ce07b94350ea32291c2",\n      "size": 971\n    },\n    "experiments/nausea_gce/20261008T080049Z_74b9b8c0/assets/tokenizer/tokenizer.json": {\n      "sha256": "4a45131245e310d1650c760a22038de8180f7311ace3575ad96b24f85ef38e1b",\n      "size": 751999\n    },\n    "experiments/nausea_gce/20261008T080049Z_74b9b8c0/assets/tokenizer/tokenizer_config.json": {\n      "sha256": "aea34e2ddddd2b9a7acecee7acf50143595848f218e62a076c0fe838b23e110e",\n      "size": 1314\n    },\n    "experiments/nausea_gce/20261008T080049Z_74b9b8c0/assets/tokenizer/vocab.txt": {\n      "sha256": "1ad4978b5dbe269dcc402a3c6eb71ceab21d712bca3b7d3b9994845e54cdcb39",\n      "size": 248477\n    },\n    "final_model/20261008T083738Z_a6a9650c/final_model.ckpt": {\n      "sha256": "a3b39454c0bf576273cffc2550fb23447c4e1cc91a937085e04071c235df2a15",\n      "size": 1258\n    },\n    "final_model/20261008T083738Z_a6a9650c/final_weights.pt": {\n      "sha256": "7e7776cd08294f71b97f348526b47da6f943f370f5a928e33624b3ccd4f28b2c",\n      "size": 446040442\n    },\n    "final_model/20261008T083738Z_a6a9650c/assets/encoder/config.json": {\n      "sha256": "1d8a2d54cd8b25538c3d991c5a05f516718682a4f5b84dfc1045c8ba84362259",\n      "size": 705\n    },\n    "final_model/20261008T083738Z_a6a9650c/assets/tokenizer/special_tokens_map.json": {\n      "sha256": "63cd5b53789b3b5ae2c3d7aa377bcba1557ff0b41dc69ce07b94350ea32291c2",\n      "size": 971\n    },\n    "final_model/20261008T083738Z_a6a9650c/assets/tokenizer/tokenizer.json": {\n      "sha256": "4a45131245e310d1650c760a22038de8180f7311ace3575ad96b24f85ef38e1b",\n      "size": 751999\n    },\n    "final_model/20261008T083738Z_a6a9650c/assets/tokenizer/tokenizer_config.json": {\n      "sha256": "aea34e2ddddd2b9a7acecee7acf50143595848f218e62a076c0fe838b23e110e",\n      "size": 1314\n    },\n    "final_model/20261008T083738Z_a6a9650c/assets/tokenizer/vocab.txt": {\n      "sha256": "1ad4978b5dbe269dcc402a3c6eb71ceab21d712bca3b7d3b9994845e54cdcb39",\n      "size": 248477\n    }\n  },\n  "provenance": {\n    "selection_source_scope": "fit_only",\n    "final_refit_source_scope": "all_training",\n    "validation_accessed": false,\n    "validation_files_included": false,\n    "training_reference_sha256": "6979dfc0d92e7441ebd50eb1fc531b9cd11f91eab3bb9aae3912784b6e9a5581"\n  }\n}\n', 'runtime.py': '"""Cached utterance-evidence representation experiments with Training-only selection."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nfrom datetime import datetime, timezone\nimport gc\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport time\nimport uuid\n\nimport numpy as np\n\nfrom . import data, head, evidence, source_bundle\nfrom .bootstrap import prepare_data\nfrom .budget import Budget, BudgetExceeded\n\nOTHER_LABELS = [0, 1, 2, 3, 4, 5, 7, 8]\n\n\ndef _stamp():\n    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]\n\n\ndef _read(path):\n    return json.loads(Path(path).read_text(encoding="utf-8-sig"))\n\n\ndef _digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()\n\n\ndef dataset_digest(texts, targets):\n    value = hashlib.sha256()\n    for text, target in zip(texts, targets):\n        value.update(text.encode("utf-8"))\n        value.update(b"\\0")\n        value.update(np.asarray(target, dtype=np.int8).tobytes())\n    return value.hexdigest()\n\n\ndef _hash(path, budget=None):\n    value = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        while block := stream.read(4 * 1024 * 1024):\n            if budget:\n                budget.check("hash artifact")\n            value.update(block)\n    return value.hexdigest()\n\n\ndef _safe(root, relative):\n    root = Path(root).resolve()\n    candidate = root / relative\n    result = candidate.resolve()\n    if result == root or not result.is_relative_to(root):\n        raise ValueError("Artifact path escapes its run directory")\n    for part in (candidate, *candidate.parents):\n        if part == root.parent:\n            break\n        if part.is_symlink():\n            raise ValueError("Symlink artifacts are not supported")\n    return result\n\n\ndef _verify_hashes(root, hashes, budget=None):\n    if not isinstance(hashes, dict) or not hashes:\n        raise ValueError("Missing completed-artifact hashes")\n    for relative, expected in hashes.items():\n        path = _safe(root, relative)\n        if not path.is_file() or _hash(path, budget) != expected:\n            raise ValueError("Saved artifact changed: " + relative)\n\n\ndef _copy_file(source, target, budget):\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_name(target.name + ".copying")\n    with Path(source).open("rb") as reader, temporary.open("wb") as writer:\n        while block := reader.read(4 * 1024 * 1024):\n            budget.check("publish verified artifact")\n            writer.write(block)\n    if _hash(source, budget) != _hash(temporary, budget):\n        raise OSError("Published copy content mismatch")\n    temporary.replace(target)\n\n\ndef _save_npz(path, **arrays):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    with temporary.open("wb") as stream:\n        np.savez_compressed(stream, **arrays)\n    temporary.replace(path)\n\n\ndef _save_feature_cache(directory, cache, identity, budget):\n    """Keep large FP32 arrays on local scratch disk, never in Drive results."""\n    directory.mkdir(parents=True, exist_ok=True)\n    metadata, hashes = {}, {}\n    for key, value in cache.items():\n        if not isinstance(value, np.ndarray):\n            continue\n        if Path(key).name != key or "/" in key or "\\\\" in key:\n            raise ValueError("Invalid feature-cache array name")\n        destination = directory / (key + ".npy")\n        temporary = destination.with_suffix(".npy.tmp")\n        budget.check("save local feature cache", training=True)\n        with temporary.open("wb") as stream:\n            np.save(stream, value, allow_pickle=False)\n        temporary.replace(destination)\n        hashes[destination.name] = _hash(destination, budget)\n        metadata[key] = {"shape": list(value.shape), "dtype": str(value.dtype), "bytes": int(value.nbytes)}\n    saved = {"identity": identity, "timing": _serial(cache.get("_timing", {})),\n             "arrays": metadata, "artifact_hashes": hashes, "storage": "local scratch only"}\n    data.write_json(directory / "complete.json", saved)\n    return saved\n\n\ndef validate_settings(settings):\n    result = dict(settings)\n    if result.get("version") != "ver22":\n        raise ValueError("This runner supports ver22 only")\n    if result.get("seed") != 42 or result.get("holdout_size") != .2:\n        raise ValueError("The verified ver19 split requires seed 42 and holdout_size 0.2")\n    if result.get("refit_full_training") is not True:\n        raise ValueError("Accepted recipes must be refit on the full-Training source model")\n    if result.get("batch_size") != 4 or result.get("encoder_window_batch") != 4:\n        raise ValueError("Verified source inference requires the original batch and window batch sizes of 4")\n    for key in ("batch_size", "encoder_window_batch", "max_evidence_epochs", "evidence_batch_size",\n                "early_stopping_patience", "ce_warmup_epochs", "log_every"):\n        if type(result.get(key)) is not int or result[key] <= 0:\n            raise ValueError(key + " must be a positive integer")\n    if result["max_evidence_epochs"] > 20 or result["evidence_batch_size"] > 256:\n        raise ValueError("Evidence training permits at most 20 epochs and batch size 256")\n    if result["ce_warmup_epochs"] >= result["max_evidence_epochs"]:\n        raise ValueError("CE warmup must leave at least one mixed-loss epoch")\n    for key in ("evidence_learning_rate", "positive_weight", "gce_q", "gce_mix", "time_budget_minutes",\n                "finalization_reserve_seconds", "budget_safety_factor"):\n        value = result.get(key)\n        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value <= 0:\n            raise ValueError(key + " must be finite and positive")\n    weight_decay = result.get("evidence_weight_decay")\n    if (isinstance(weight_decay, bool) or not isinstance(weight_decay, (int, float))\n            or not math.isfinite(weight_decay) or weight_decay < 0):\n        raise ValueError("evidence_weight_decay must be finite and nonnegative")\n    if result["gce_q"] > 1 or result["gce_mix"] > .5:\n        raise ValueError("GCE requires q <= 1 and mixture <= 0.5")\n    if result["time_budget_minutes"] > 180 or result["budget_safety_factor"] < 1:\n        raise ValueError("Budget must be at most 180 minutes with safety factor >= 1")\n    if result["finalization_reserve_seconds"] >= result["time_budget_minutes"] * 60:\n        raise ValueError("Finalization reserve must fit within the shared budget")\n    for key in ("run_validation", "allow_cpu"):\n        if type(result.get(key)) is not bool:\n            raise ValueError(key + " must be boolean")\n    for key in ("source_run", "resume_run", "work_dir", "output_dir", "input_zip", "train_dir", "validation_dir"):\n        if not isinstance(result.get(key, ""), str):\n            raise ValueError(key + " must be a path string")\n    return result\n\n\ndef _stable_settings(settings):\n    volatile = {"deadline_monotonic", "input_zip", "train_dir", "validation_dir", "source_run", "work_dir", "output_dir",\n                "resume_run", "log_every", "time_budget_minutes", "finalization_reserve_seconds", "budget_safety_factor"}\n    return {key: value for key, value in settings.items() if key not in volatile}\n\n\ndef _device(settings):\n    import torch\n    if not torch.cuda.is_available() and not settings["allow_cpu"]:\n        raise RuntimeError("Colab에서 GPU 런타임을 선택하세요. 전체 데이터 CPU 실행은 자동으로 시작하지 않습니다.")\n    return "cuda" if torch.cuda.is_available() else "cpu"\n\n\ndef _release():\n    import torch\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef _serial(value):\n    if isinstance(value, np.ndarray):\n        return value.tolist()\n    if isinstance(value, np.generic):\n        return value.item()\n    if isinstance(value, dict):\n        return {key: _serial(item) for key, item in value.items()}\n    if isinstance(value, (tuple, list)):\n        return [_serial(item) for item in value]\n    return value\n\n\ndef _cached_features(checkpoint, texts, targets, settings, root, role, budget):\n    """Scope-specific cache; immutable identity and fresh hashes gate reuse."""\n    checkpoint = Path(checkpoint)\n    manifest = _read(checkpoint)\n    identity = {"role": role, "training_digest": dataset_digest(texts, targets),\n                "checkpoint_digest": _hash(checkpoint, budget),\n                "weights_digest": _hash(_safe(checkpoint.parent, manifest["weights"]), budget),\n                "training_scope": manifest["training_scope"], "fitted_samples": manifest["fitted_samples"],\n                "cache_format": "ragged_turns_global256_fp32_v1", "batch_size": settings["batch_size"]}\n    scope = "fit_only" if role == "selection" else "all_training"\n    if manifest["training_scope"] != scope:\n        raise ValueError("Wrong encoder training scope for feature-cache role")\n    groups = [np.arange(len(texts))]\n    if role == "selection":\n        split_path = root / "training_split.npz"\n        identity["split_digest"] = _hash(split_path, budget)\n        with np.load(split_path, allow_pickle=False) as split:\n            groups = [split["fit"].copy(), split["holdout"].copy()]\n        if not np.array_equal(np.sort(np.concatenate(groups)), np.arange(len(texts))):\n            raise ValueError("Feature-cache split must partition all original Training rows")\n    cache_dir = Path(settings["work_dir"]) / "feature_caches" / root.name / role\n    marker = cache_dir / "complete.json"\n    summary = root / "caches" / role / "cache_summary.json"\n    if marker.is_file():\n        saved = _read(marker)\n        if saved.get("identity") != identity:\n            raise ValueError("Feature-cache identity changed; start a fresh run")\n        _verify_hashes(cache_dir, saved["artifact_hashes"], budget)\n        cache = {key: np.load(_safe(cache_dir, key + ".npy"), allow_pickle=False, mmap_mode="r")\n                 for key in saved["arrays"]}\n        evidence._validate_cache(cache)\n        if len(cache["predictions"]) != len(texts):\n            raise ValueError("Feature-cache row count mismatch")\n        cache["_timing"] = saved.get("timing", {})\n        data.write_json(summary, saved)\n        return cache\n    model, tokenizer, loaded = head.load_model(checkpoint, _device(settings))\n    try:\n        config = {**loaded, **settings, "encoder_window_batch": loaded["encoder_window_batch"]}\n        # Historical holdout predictions used a holdout-only ordered loader.\n        # Preserve its batchmates/padding to avoid avoidable FP32 decision drift.\n        blocks, timings = [], []\n        for indices in groups:\n            block = evidence.extract_cache(model, tokenizer, [texts[int(i)] for i in indices], config, _device(settings), budget)\n            blocks.append(block)\n            timings.append(block.get("_timing", {}))\n        cache = evidence.merge_caches(blocks, groups, len(texts))\n        del block, blocks\n        cache["_timing"] = {"batching": "fit_then_historical_holdout" if role == "selection" else "all_training",\n                            "groups": timings}\n    finally:\n        del model, tokenizer\n        _release()\n    budget.check("save completed feature cache", training=True)\n    saved = _save_feature_cache(cache_dir, cache, identity, budget)\n    data.write_json(summary, saved)\n    return cache\n\n\ndef _comparison(targets, before, after):\n    baseline, current = data.score_predictions(targets, before), data.score_predictions(targets, after)\n    if not np.array_equal(before[:, OTHER_LABELS], after[:, OTHER_LABELS]):\n        raise ValueError("A supposedly frozen symptom decision changed")\n    per_symptom = {}\n    for j, label in enumerate(data.TARGET_SYMPTOMS):\n        truth, old, new = targets[:, j], before[:, j], after[:, j]\n        per_symptom[label] = {"f1_delta": current["per_symptom"][label]["f1"] - baseline["per_symptom"][label]["f1"],\n            "paired": {"fn_to_tp": int(((truth == 1) & (old == 0) & (new == 1)).sum()),\n                       "tp_to_fn": int(((truth == 1) & (old == 1) & (new == 0)).sum()),\n                       "fp_to_tn": int(((truth == 0) & (old == 1) & (new == 0)).sum()),\n                       "tn_to_fp": int(((truth == 0) & (old == 0) & (new == 1)).sum())}}\n    macro = current["macro_f1"] - baseline["macro_f1"]\n    nausea = current["per_symptom"]["오심"]["f1"] - baseline["per_symptom"]["오심"]["f1"]\n    return {"available": True, "reference_version": "ver19", "reference_metrics": baseline,\n            "new_metrics": current, "macro_f1_delta": macro, "nausea_f1_delta": nausea,\n            "both_macro_and_nausea_improved": macro > 0 and nausea > 0,\n            "status": "improved" if macro > 0 and nausea > 0 else "not_improved",\n            "other_eight_predictions_unchanged": True, "per_symptom": per_symptom}\n\n\ndef _choose(records):\n    baseline = records[0]\n    improved = [record for record in records[1:]\n                if record["metrics"]["macro_f1"] > baseline["metrics"]["macro_f1"]\n                and record["metrics"]["per_symptom"]["오심"]["f1"] > baseline["metrics"]["per_symptom"]["오심"]["f1"]]\n    return max(improved, key=lambda record: record["metrics"]["macro_f1"]) if improved else baseline\n\n\ndef _parameter_count(state):\n    return sum(int(np.asarray(value).size) for value in state.values())\n\n\ndef _save_records(root, records, holdout, targets):\n    rows = []\n    for record in records:\n        destination = root / "experiments" / record["candidate_id"]\n        destination.mkdir(parents=True, exist_ok=True)\n        data.write_json(destination / "history.json", _serial(record["history"]))\n        data.write_json(destination / "training_metrics.json", record["metrics"])\n        _save_npz(destination / "evidence.npz", **record["state"])\n        _save_npz(destination / "holdout_predictions.npz", indices=holdout, targets=targets[holdout], predictions=record["predictions"])\n        rows.append({"candidate": record["candidate_id"], "macro_f1": record["metrics"]["macro_f1"],\n                     "nausea_f1": record["metrics"]["per_symptom"]["오심"]["f1"],\n                     "selected_epochs": record["epoch"], "stop_reason": record["stop_reason"]})\n    with (root / "candidate_summary.csv").open("w", encoding="utf-8-sig", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef _predict_loaded(model, tokenizer, texts, manifest, device, budget=None, *, training=False):\n    import torch\n    from torch.utils.data import DataLoader\n    from .neural import EncodeDataset\n    dataset = EncodeDataset(texts, None, tokenizer, manifest["max_length"], manifest["overlap"], utterance_evidence=True)\n    loader = DataLoader(dataset, batch_size=manifest["batch_size"], shuffle=False, num_workers=0, collate_fn=dataset.collate)\n    predictions = []\n    model.eval()\n    with torch.inference_mode():\n        for batch in loader:\n            if budget:\n                budget.check("actual model inference", training=training)\n            logits, _ = model({key: value.to(device) for key, value in batch.items()})\n            predictions.append(logits.argmax(-1).cpu().numpy().astype(np.int8))\n    if not predictions:\n        raise ValueError("Inference requires nonempty text input")\n    return np.concatenate(predictions)\n\n\ndef _confirm_selected(source, records, texts, targets, holdout, settings, budget):\n    """Confirm the chosen cached candidate through the actual fit-only model."""\n    selected = _choose(records)\n    audit = {"scope": "Training holdout only", "validation_accessed": False, "candidates": []}\n    if selected["candidate_id"] == "baseline":\n        audit["status"] = "no_improved_cached_candidate"\n        return selected, audit\n    budget.check("confirm actual holdout candidate", training=True)\n    device = _device(settings)\n    model, tokenizer, manifest = head.load_model(source["fit_checkpoint"], device)\n    confirmed = set()\n    try:\n        while selected["candidate_id"] != "baseline" and selected["candidate_id"] not in confirmed:\n            budget.check("confirm actual holdout candidate", training=True)\n            parameter_audit = evidence.set_evidence_state(model, selected["state"])\n            predicted = _predict_loaded(model, tokenizer, [texts[int(i)] for i in holdout],\n                                        manifest, device, budget, training=True)\n            comparison = _comparison(targets[holdout], records[0]["predictions"], predicted)\n            audit["candidates"].append({"candidate_id": selected["candidate_id"],\n                "cached_vs_actual_nausea_disagreements": int((selected["predictions"][:, 6] != predicted[:, 6]).sum()),\n                "actual_metrics": comparison["new_metrics"], "parameter_audit": parameter_audit})\n            selected["cached_metrics"] = selected["metrics"]\n            selected["metrics"] = comparison["new_metrics"]\n            selected["predictions"] = predicted\n            selected["prediction_scope"] = "actual fit-only source model forward"\n            confirmed.add(selected["candidate_id"])\n            selected = _choose(records)\n        audit["status"] = "confirmed" if selected["candidate_id"] != "baseline" else "no_actual_improvement"\n        return selected, audit\n    finally:\n        del model, tokenizer\n        _release()\n\n\ndef predict_checkpoint(checkpoint, texts, budget=None):\n    import torch\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    model, tokenizer, manifest = head.load_model(checkpoint, device)\n    try:\n        return _predict_loaded(model, tokenizer, texts, manifest, device, budget)\n    finally:\n        del model, tokenizer\n        _release()\n\n\ndef inference(checkpoint, label_dir, output, budget=None):\n    paths, texts, _ = data.load_dataset(label_dir, include_targets=False)\n    predicted = predict_checkpoint(checkpoint, texts, budget)\n    data.write_predictions(paths, predicted, output)\n    return predicted\n\n\ndef _export_and_evaluate(source, selected, trained_evidence, settings, root, texts, validation_dir, budget, fallback_reason):\n    """Keep original full encoder and global heads; export a complete evidence update."""\n    import torch\n    device = _device(settings)\n    checkpoint = source["full_checkpoint"]\n    model, tokenizer, original = head.load_model(checkpoint, device)\n    actual_candidate = selected["candidate_id"] if trained_evidence is not None else "baseline"\n    try:\n        if settings["run_validation"]:\n            val_paths, val_texts, truth = data.load_dataset(validation_dir, include_targets=True)\n        else:\n            val_paths, val_texts, truth = None, texts[:12], None\n        config = {**original, **settings, "batch_size": original["batch_size"], "encoder_window_batch": original["encoder_window_batch"]}\n        cache = evidence.extract_cache(model, tokenizer, val_texts, config, device, budget, training=False)\n        baseline = cache["predictions"].copy()\n        if trained_evidence is not None:\n            _, cache_predicted = evidence.apply_evidence(cache, trained_evidence, settings, device, budget)\n            audit = evidence.set_evidence_state(model, trained_evidence)\n            # Export fidelity uses the actual inference device and kernels.\n            # Cached evidence optimization and full forward may differ near an argmax tie.\n            expected = _predict_loaded(model, tokenizer, val_texts, original, device, budget)\n            audit["cache_vs_model_nausea_disagreements"] = int((cache_predicted[:, 6] != expected[:, 6]).sum())\n        else:\n            expected = baseline.copy()\n            audit = {"changed_parameters": [], "all_source_parameters_unchanged": True}\n        if not np.array_equal(expected[:, OTHER_LABELS], baseline[:, OTHER_LABELS]):\n            raise ValueError("Other-eight symptom preservation failed before export")\n        destination = root / "final_model"\n        destination.mkdir(parents=True, exist_ok=True)\n        source_assets = _safe(checkpoint.parent, original["assets"])\n        for path in sorted(source_assets.rglob("*")):\n            if path.is_file():\n                _copy_file(path, destination / "assets" / path.relative_to(source_assets), budget)\n        weight_path = destination / "weights.pt"\n        if trained_evidence is None:\n            _copy_file(_safe(checkpoint.parent, original["weights"]), weight_path, budget)\n        else:\n            temporary = weight_path.with_name("weights.pt.tmp")\n            torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, temporary)\n            budget.check("publish completed final evidence")\n            temporary.replace(weight_path)\n        manifest = {**original, "version": "ver22", "source_version": "ver19",\n            "weights": "final_model/weights.pt", "assets": "final_model/assets",\n            "selected_candidate": actual_candidate, "evidence_epochs": selected["epoch"] if trained_evidence is not None else 0,\n            "evidence_fitted_samples": len(texts) if trained_evidence is not None else 0,\n            "evidence_training": {"candidate_id": actual_candidate,\n                "learning_rate": settings["evidence_learning_rate"], "batch_size": settings["evidence_batch_size"],\n                "weight_decay": settings["evidence_weight_decay"], "positive_weight": settings["positive_weight"],\n                "gce_q": settings["gce_q"], "gce_mix": settings["gce_mix"] if actual_candidate == "evidence_ce_gce" else 0,\n                "ce_warmup_epochs": settings["ce_warmup_epochs"],\n                "encoder_frozen": True, "global_projection_frozen": True, "all_nine_global_heads_frozen": True,\n                "evidence_frozen": trained_evidence is None, "other_eight_heads_frozen": True,\n                "trainable_parameters": _parameter_count(trained_evidence) if trained_evidence is not None else 0},\n            "refit_skipped_reason": fallback_reason, "baseline_reused": trained_evidence is None,\n            "source_full_weights_sha256": _hash(_safe(checkpoint.parent, original["weights"]), budget)}\n        final_path = root / "mission3_ver22_model.ckpt"\n        data.write_json(final_path, manifest)\n    finally:\n        del model, tokenizer\n        _release()\n    actual = predict_checkpoint(final_path, val_texts, budget)\n    if not np.array_equal(actual, expected):\n        raise ValueError("Portable reloaded predictions differ from the in-memory evidence result")\n    if not np.array_equal(actual[:, OTHER_LABELS], baseline[:, OTHER_LABELS]):\n        raise ValueError("Frozen eight-symptom decisions changed after reload")\n    verification = {"samples": len(actual), "decisions": int(actual.size), "portable_reloaded_decisions_match": True,\n                    "other_eight_decisions_match_original_full_model": True, "parameter_audit": _serial(audit),\n                    "scope": "all Validation rows" if settings["run_validation"] else "Training probe only"}\n    data.write_json(root / "preservation_verification.json", verification)\n    if settings["run_validation"]:\n        data.write_predictions(val_paths, actual, root / "mission3.csv")\n        data.write_json(root / "validation_metrics.json", data.score_predictions(truth, actual))\n        report = _comparison(truth, baseline, actual)\n        report["comparison_scope"] = "Descriptive final Validation only; never used for recipe selection or reverting a trained evidence"\n        data.write_json(root / "validation_comparison.json", report)\n        _save_npz(root / "validation_predictions.npz", targets=truth, predictions=actual, baseline_predictions=baseline)\n    return manifest\n\n\ndef _environment():\n    import torch\n    from importlib.metadata import version\n    return {"packages": {name: version(name) for name in ("torch", "transformers", "numpy", "scikit-learn")},\n            "cuda_available": torch.cuda.is_available(), "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}\n\n\ndef run(settings, runtime_sources=None):\n    started = time.monotonic()\n    settings = validate_settings(settings)\n    deadline = settings.get("deadline_monotonic", started + settings["time_budget_minutes"] * 60)\n    if deadline > started + settings["time_budget_minutes"] * 60 + 1:\n        raise ValueError("Shared deadline exceeds the allowed budget")\n    budget = Budget(deadline, settings["finalization_reserve_seconds"], settings["budget_safety_factor"])\n    sources = runtime_sources or {p.name: p.read_text(encoding="utf-8-sig") for p in Path(__file__).parent.iterdir()\n                                if p.is_file() and p.suffix in (".py", ".json", ".md", ".txt")}\n    sources = {k: v for k, v in sources.items() if not k.startswith("test_") and k != "build_notebook.py"}\n    output = Path(settings["output_dir"]).expanduser().resolve()\n    resume = settings.get("resume_run", "")\n    root = Path(resume).expanduser().resolve() if resume else output / _stamp()\n    if root == output or not root.is_relative_to(output):\n        raise ValueError("resume_run must be inside output_dir")\n    if resume and not (root / "run_identity.json").is_file():\n        raise ValueError("resume_run is not initialized")\n    if not resume:\n        root.mkdir(parents=True, exist_ok=False)\n    writable = not bool(resume)\n    try:\n        budget.check("prepare inputs")\n        train_dir, valid_dir = prepare_data(settings["input_zip"], settings["work_dir"], settings.get("train_dir", ""), settings.get("validation_dir", ""))\n        _, texts, targets = data.load_dataset(train_dir, include_targets=True)\n        fit, holdout = data.make_split(texts, targets, .2, 42)\n        identity = {"version": "ver22", "training_digest": dataset_digest(texts, targets),\n                    "settings_digest": _digest(_stable_settings(settings)),\n                    "source_digest": _digest({k: v for k, v in sources.items() if not k.endswith(".md")})}\n        if resume:\n            if _read(root / "run_identity.json") != identity:\n                raise ValueError("Resume data, settings or source changed")\n            with np.load(root / "training_split.npz", allow_pickle=False) as saved:\n                if not np.array_equal(fit, saved["fit"]) or not np.array_equal(holdout, saved["holdout"]):\n                    raise ValueError("Resume split changed")\n        else:\n            data.write_json(root / "run_identity.json", identity)\n            data.write_json(root / "settings.json", {k: v for k, v in settings.items() if k != "deadline_monotonic"})\n            _save_npz(root / "training_split.npz", fit=fit, holdout=holdout)\n            for name, content in sources.items():\n                if Path(name).name != name:\n                    raise ValueError("Embedded source names must be flat")\n                destination = root / "submission" / name\n                destination.parent.mkdir(parents=True, exist_ok=True)\n                destination.write_bytes(content.encode("utf-8"))\n        if (root / "run_complete.json").is_file():\n            _verify_hashes(root, _read(root / "run_complete.json")["artifact_hashes"], budget)\n            print("Completed ver22 artifacts verified; no additional training.", flush=True)\n            return root\n        writable = True\n        _device(settings)\n        data.write_json(root / "runtime.json", {"status": "running", "message": "Only run_complete.json confirms completion"})\n        data.write_json(root / "environment.json", _environment())\n        source = source_bundle.prepare_source(settings.get("source_run", ""), Path(settings["work_dir"]) / "verified_ver19", budget)\n        verified = source_bundle.verify_training(texts, targets, fit, holdout, source)\n        data.write_json(root / "source_verified.json", _serial(verified))\n        print("Verified original ver19 fit-only and full-Training models. Extracting fixed features.", flush=True)\n        cache = _cached_features(source["fit_checkpoint"], texts, targets, settings, root, "selection", budget)\n        with np.load(source["root"] / "holdout_predictions.npz", allow_pickle=False) as saved:\n            if (not np.array_equal(saved["indices"], holdout) or not np.array_equal(saved["targets"], targets[holdout])\n                    or not np.array_equal(saved["predictions"], cache["predictions"][holdout])):\n                raise ValueError("Current fit-only encoder does not reproduce the pinned ver19 holdout predictions; no candidate was trained")\n        data.write_json(root / "baseline_verification.json", {"samples": len(holdout), "all_nine_decisions_match_pinned_ver19": True})\n        data.write_json(root / "pilot_report.json", _serial(cache.get("_timing", {})))\n        records = evidence.evidence_candidates(cache, targets, fit, holdout, settings, budget, device=_device(settings))\n        search_budget_limited = (records[0].get("search_stop_reason") == "budget"\n                                 or any(record["stop_reason"] == "budget" for record in records))\n        del cache\n        _release()\n        try:\n            selected, confirmation = _confirm_selected(source, records, texts, targets, holdout, settings, budget)\n        except BudgetExceeded:\n            search_budget_limited = True\n            selected = records[0]\n            confirmation = {"status": "budget_before_confirmation", "scope": "Training holdout only",\n                            "validation_accessed": False}\n        data.write_json(root / "candidate_confirmation.json", _serial(confirmation))\n        _save_records(root, records, holdout, targets)\n        report = _comparison(targets[holdout], records[0]["predictions"], selected["predictions"])\n        data.write_json(root / "training_reference_comparison.json", report)\n        data.write_json(root / "training_metrics.json", selected["metrics"])\n        _save_npz(root / "holdout_predictions.npz", indices=holdout, targets=targets[holdout], predictions=selected["predictions"])\n        improved = report["both_macro_and_nausea_improved"]\n        data.write_json(root / "final_selection.json", {"candidate_id": selected["candidate_id"], "selected_epochs": selected["epoch"],\n            "metrics": selected["metrics"], "selection_policy": "Training only: jointly improve Macro and nausea F1, then highest Macro F1; exact ties retain earlier candidate",\n            "evidence_candidates": [r["candidate_id"] for r in records], "evidence_search_budget_limited": search_budget_limited,\n            "both_macro_and_nausea_improved": improved})\n        trained_evidence = None\n        reason = "budget_before_complete_evidence_search" if search_budget_limited else "training_reference_not_jointly_improved"\n        if improved:\n            full_cache = None\n            try:\n                budget.check("begin full-Training evidence refit", training=True)\n                full_cache = _cached_features(source["full_checkpoint"], texts, targets, settings, root, "full_refit", budget)\n                state, history = evidence.refit_evidence(full_cache, targets, selected["candidate_id"], selected["epoch"], settings, budget,\n                                                        device=_device(settings))\n                # Publish only a refit that completed all selected epochs on full Training.\n                trained_evidence = state\n                data.write_json(root / "final_history.json", _serial(history))\n                _save_npz(root / "final_evidence.npz", **state)\n                reason = None\n            except BudgetExceeded:\n                reason = "budget_before_complete_evidence_refit"\n                trained_evidence = None\n                print("Full evidence refit incomplete; preserving the original full ver19 model.", flush=True)\n            finally:\n                del full_cache\n                _release()\n        manifest = _export_and_evaluate(source, selected, trained_evidence, settings, root, texts, valid_dir, budget, reason)\n        budget.check("complete verified run")\n        status = "complete" if trained_evidence is not None else ("complete_budget_limited" if improved or search_budget_limited else "complete_not_improved")\n        evidence_sidecars = {"final_evidence.npz", "final_history.json"}\n        data.write_json(root / "evidence_artifacts.json", {"active": trained_evidence is not None,\n            "active_paths": sorted(evidence_sidecars) if trained_evidence is not None else [],\n            "note": "Only active_paths belong to this export; other evidence sidecars may remain from an interrupted earlier attempt."})\n        data.write_json(root / "runtime.json", {"status": status, "seconds": time.monotonic() - started,\n            "seconds_scope": "worker invocation; shared notebook deadline includes installation", "selected_candidate": selected["candidate_id"],\n            "exported_candidate": manifest["selected_candidate"], "baseline_reused": manifest["baseline_reused"],\n            "evidence_search_budget_limited": search_budget_limited,\n            "both_macro_and_nausea_improved_on_training": improved, "training_scope": "all_training", "fitted_samples": len(texts),\n            "refit_skipped_reason": reason, "evidence_trainable_parameters": _parameter_count(records[0]["state"]), "other_eight_preserved": True,\n            "budget_minutes_remaining": budget.remaining() / 60})\n        files = sorted(p for p in root.rglob("*") if p.is_file() and p.name != "run_complete.json"\n                       and not p.name.endswith((".tmp", ".copying"))\n                       and (trained_evidence is not None or p.relative_to(root).as_posix() not in evidence_sidecars))\n        hashes = {p.relative_to(root).as_posix(): _hash(p, budget) for p in files}\n        data.write_json(root / "run_complete.json", {"status": status, "artifact_hashes": hashes})\n        print("ver22 artifacts complete:", status, flush=True)\n        return root\n    except BaseException as error:\n        if writable and not (root / "run_complete.json").exists():\n            data.write_json(root / "runtime.json", {"status": "budget_exhausted" if isinstance(error, BudgetExceeded) else "failed",\n                            "seconds": time.monotonic() - started, "error_type": type(error).__name__,\n                            "message": "No completion marker: artifacts are incomplete; keep the previous submission."})\n        raise\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--audio_dir", default="")\n    parser.add_argument("--label_dir", required=True)\n    parser.add_argument("--ckpt_path", "--checkpoint", dest="checkpoint", required=True)\n    parser.add_argument("--output", required=True)\n    args = parser.parse_args(argv)\n    inference(args.checkpoint, args.label_dir, args.output)\n\n\nif __name__ == "__main__":\n    main()\n', 'inference.py': '"""Official one-command inference entry point for a portable ver22 result folder."""\nimport importlib.util\nfrom pathlib import Path\nimport sys\n\nif __package__:\n    from .runtime import main\nelse:\n    root = Path(__file__).resolve().parent\n    name = "mission3_ver22_submission"\n    spec = importlib.util.spec_from_file_location(name, root / "__init__.py",\n                                                  submodule_search_locations=[str(root)])\n    package = importlib.util.module_from_spec(spec)\n    sys.modules[name] = package\n    spec.loader.exec_module(package)\n    from importlib import import_module\n    main = import_module(name + ".runtime").main\n\nif __name__ == "__main__":\n    main()\n', 'run_notebook.py': '"""Launch the bounded ver22 experiment in a fresh interpreter after installation."""\nfrom __future__ import annotations\n\nimport argparse\nimport importlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport re\nimport sys\nimport traceback\n\nSOURCE_FILES = (\n    "__init__.py", "data.py", "bootstrap.py", "budget.py", "neural.py", "head.py", "evidence.py",\n    "source_bundle.py", "source_reference.json", "runtime.py", "inference.py",\n    "run_notebook.py", "requirements.txt", "README.md",\n)\n\n\ndef _failure_report(error, stage):\n    """Describe failures without copying sample text, paths, or exception values."""\n    category = "unclassified"\n    reason = "실행 중 오류가 발생했습니다. 아래 단계와 코드 위치를 포함한 오류 기록을 확인하세요."\n    try:\n        message = str(error)\n    except Exception:\n        message = ""\n    if stage == "settings_load":\n        category = "settings_unreadable"\n        reason = "실행 설정을 읽지 못했습니다. 수정된 노트북을 처음 셀부터 모두 실행하세요."\n    elif isinstance(error, ImportError):\n        category = "dependency_import_failed"\n        reason = "실행에 필요한 패키지를 불러오지 못했습니다. 패키지 준비 셀의 설치 결과를 확인하세요."\n    elif "Colab에서 GPU 런타임을 선택하세요" in message:\n        category = "gpu_unavailable"\n        reason = "GPU가 연결되지 않았습니다. Colab 런타임 유형을 GPU로 설정한 뒤 처음부터 실행하세요."\n    elif "CUDA out of memory" in message or type(error).__name__ == "OutOfMemoryError":\n        category = "out_of_memory"\n        reason = "모델 실행에 필요한 메모리가 부족합니다. GPU 종류와 메모리 사용량을 확인하세요."\n    elif "Current fit-only encoder does not reproduce the pinned ver19 holdout predictions" in message:\n        category = "baseline_prediction_mismatch"\n        reason = "현재 환경에서 기존 ver19 내부 평가 예측이 정확히 재현되지 않아 새 후보 학습을 중단했습니다. 오류 기록으로 재현 조건을 확인해야 합니다."\n    elif "Input ZIP is missing:" in message:\n        category = "input_zip_missing"\n        reason = "입력 ZIP을 찾지 못했습니다. Drive 연결과 SETTINGS[\'input_zip\']의 실제 파일 경로를 확인하세요."\n    elif any(token in message for token in (\n        "Required ver19 result not found", "ver19 source was not found",\n        "selected directory does not contain the required complete ver19 run",\n    )):\n        category = "ver19_source_missing"\n        reason = "필요한 ver19 결과를 찾지 못했습니다. 가중치가 포함된 전체 결과 폴더 또는 ZIP의 Drive 경로를 SETTINGS[\'source_run\']에 지정하세요."\n    elif any(token in message for token in (\n        "ver19 source mismatch:", "No candidate matched the pinned ver19 source",\n        "Required ver19 source file is missing:", "ver19 ZIP is missing a required source file:",\n    )):\n        category = "ver19_source_verification_failed"\n        reason = "ver19 결과가 요구된 원본과 일치하지 않거나 필수 파일이 누락됐습니다. 전체 결과 폴더 또는 ZIP을 확인하세요."\n    elif type(error).__name__ == "BudgetExceeded":\n        category = "time_budget_exhausted"\n        reason = "설정된 작업 시간 예산이 끝나 실행을 중단했습니다. 완료 표시가 없는 결과는 제출에 사용하지 마세요."\n    elif stage == "source_read":\n        category = "runtime_sources_unreadable"\n        reason = "노트북에 포함된 실행 코드 일부를 읽지 못했습니다. 수정된 노트북을 처음 셀부터 모두 실행하세요."\n    elif stage == "result_write":\n        category = "result_record_failed"\n        reason = "실행 결과 위치를 기록하지 못했습니다. 오류 기록과 Drive 결과 폴더의 완료 표시를 확인하세요."\n    frames = [{"file": Path(frame.f_code.co_filename).name, "line": line,\n               "function": frame.f_code.co_name}\n              for frame, line in traceback.walk_tb(error.__traceback__)]\n    report = {"status": "failed", "stage": stage, "error_type": type(error).__name__,\n              "category": category, "reason": reason, "frames": frames[-20:]}\n    if (isinstance(error, ImportError) and isinstance(error.name, str)\n            and re.fullmatch(r"[A-Za-z_]\\w*(?:\\.[A-Za-z_]\\w*)*", error.name, flags=re.ASCII)):\n        report["module_name"] = error.name\n    if isinstance(error, OSError) and type(error.errno) is int:\n        report["errno"] = error.errno\n    return report\n\n\ndef _write_report(path, report):\n    path.write_text(json.dumps(report, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--settings", required=True)\n    parser.add_argument("--result_file", required=True)\n    args = parser.parse_args(argv)\n    root = Path(__file__).resolve().parent\n    error_file = root / "notebook_error.json"\n    stage = "initialize"\n    try:\n        _write_report(error_file, {"status": "starting", "stage": stage})\n        stage = "settings_load"\n        settings = json.loads(Path(args.settings).read_text(encoding="utf-8-sig"))\n        stage = "runtime_import"\n        name = "mission3_ver22_execution"\n        spec = importlib.util.spec_from_file_location(name, root / "__init__.py",\n                                                      submodule_search_locations=[str(root)])\n        package = importlib.util.module_from_spec(spec)\n        sys.modules[name] = package\n        spec.loader.exec_module(package)\n        runtime = importlib.import_module(name + ".runtime")\n        stage = "source_read"\n        sources = {name: (root / name).read_text(encoding="utf-8") for name in SOURCE_FILES}\n        stage = "runtime_run"\n        result = runtime.run(settings, runtime_sources=sources)\n        stage = "result_write"\n        Path(args.result_file).write_text(json.dumps(str(result)), encoding="utf-8")\n        _write_report(error_file, {"status": "complete", "stage": "complete"})\n    except Exception as error:\n        report = _failure_report(error, stage)\n        try:\n            _write_report(error_file, report)\n        except OSError:\n            report["diagnostic_file_saved"] = False\n        print(json.dumps(report, ensure_ascii=False, allow_nan=False), flush=True)\n        raise SystemExit(1) from None\n\n\nif __name__ == "__main__":\n    main()\n', 'requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy==2.1.3\nscikit-learn==1.6.1\nscipy>=1.11,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\nsentencepiece>=0.2,<0.3\nprotobuf>=4.25,<7\n', 'README.md': '# Mission 3 ver22\n\n## Cached utterance features and retrained nausea evidence\n\nver21 changed the final linear nausea head without improving the Training holdout. ver22 instead compares the unchanged ver19 baseline with `evidence_ce` and `evidence_ce_gce`, retraining the existing nausea utterance-evidence branch. Its local context projection, attention, and evidence output can change. Both candidates start from the same original ver19 evidence weights. The expensive encoder, global dialogue path, and other eight symptom heads remain frozen. Cached utterance representations are reused across candidates, so repeated evidence training does not repeatedly run the encoder. The original text, targets, samples, and overflow windows remain; model input is only `utterances[].text`. Predictions remain raw two-class argmax. There is no threshold search, post-hoc score calibration, external training data, or label correction.\n\n`evidence_ce` uses weighted cross entropy with the original nausea positive weight 3.0. `evidence_ce_gce` uses the same CE for the first epoch and then an equal mix of CE and generalized cross entropy with q=0.7. Defaults are seed 42, batch size 128 cached dialogues, learning rate 0.0001, weight decay 0.01, at most 8 epochs, evaluation every epoch, and early stopping after 3 epochs without improvement. The original evidence dropout 0.1 is retained. GCE is a hypothesis about robustness to difficult targets, not a declaration that those labels are incorrect. No manual evidence labels or explicit question/negation classes are added. New F1 and real Colab timings have not been measured locally. A large improvement is not guaranteed.\n\n## Leakage controls and the unchanged baseline\n\nThe existing grouped Training split is verified before use. Candidate fitting and selection use features from the ver19 selection checkpoint trained on 23,360 fit samples, never the final model trained on all 29,200 Training samples. The 5,840 holdout samples are used only for candidate and epoch selection. `source_reference.json` and source-bundle checks bind the source weights, data, labels, split, and saved reference to the intended ver19 run. A missing, altered, or incompatible source is an error rather than an unchecked replacement.\n\nFit and holdout features are extracted in separate passes using the original batching, then restored to the original sample order. The original holdout decisions must reproduce before candidate fitting starts. Final export verifies actual model forward decisions against the portable reloaded model and checks that all other eight symptoms remain unchanged. Any nausea decision differences between cached-evidence calculations and final model forward are recorded as diagnostics; the actual reloaded model produces the final CSV.\n\nOnly a candidate that increases both holdout Macro F1 and nausea F1 can replace the baseline. Other eight symptom predictions must remain unchanged. This repeatedly used holdout is exploratory, not an untouched test set. Official Validation is evaluated once after selection and is never used for optimization or selection.\n\nBefore full-Training refitting, the chosen cached candidate is checked through the actual fit-only model on the original holdout order. `candidate_confirmation.json` records any cached-versus-actual decision differences. Actual predictions replace cached metrics for final selection; a candidate that no longer improves cannot be accepted. A timeout before this confirmation retains the baseline with a budget-limited status.\n\nThe GCE objective follows [Zhang and Sabuncu, NeurIPS 2018](https://papers.nips.cc/paper_files/paper/2018/hash/f2925f97bc13ad2852a7a551802feea0-Abstract.html). Its published noisy-label experiments do not establish an expected F1 gain on this Korean dataset. Here both objectives use the original labels without manual edits.\n\nIf a candidate improves and time permits, features are extracted using the separate final ver19 model trained on all 29,200 Training samples. The selected objective and epoch count retrain a fresh evidence branch initialized from that final model\'s original evidence weights. The fit-model evidence branch is not transplanted onto final-model features. Existing encoder and other eight symptom outputs remain unchanged. If no candidate improves, or a complete new evidence branch cannot be produced within the budget, publication preserves the original full-Training ver19 model and clearly reports that no improvement was delivered. Original ver19 files are never modified.\n\nBaseline publication still requires source verification, completed internal comparison, and final inference/publication to finish within the deadline. A timeout in those mandatory stages produces an incomplete run rather than a completed fallback result.\n\n## Required files and execution\n\nOpen `mission3_ver22.ipynb` in Colab, select a GPU, and run all cells. Connect the shared Drive account containing both the existing `mission3_ver5_inputs.zip` and the complete existing ver19 result folder or its ZIP. The ver19 selection and full-Training weights, assets, and provenance files are required; the input ZIP alone is insufficient. Leave `source_run` blank to discover the existing ver19 result on Drive. If discovery fails, enter the complete Drive path to that folder or ZIP in `source_run`. No new pretrained model download is required. Runtime sources and reference metadata are embedded in the notebook. Dependencies match ver19 without additions or upgrades.\n\nRun cells from the beginning after reconnecting or restarting the runtime. Running the last cell alone previously produced `NameError: runtime_dir` followed by a cleanup `NameError: release_gpu`. Preparation is now checked before execution. Missing or failed setup shows recovery instructions without launching training or masking the first error. Code restoration and successful installation must belong to the current runtime folder and deadline; a failed retry invalidates earlier success. If an earlier cell fails, fix its first error before rerunning all cells. The execution cell never creates a new time budget on its own.\n\nWorker failures are recorded even when Colab does not display inherited subprocess output. Before requesting GPU release, the notebook shows the failed stage, exception type, safe diagnostic reason and code locations, and saves `ver22/_diagnostics/ver22_error_*.json` on Drive. The launcher also records import and settings-load failures. Reports exclude raw exception strings, dialogue content, per-record data paths, and local variables. An unclassified exception is reported as unclassified; the outer `CalledProcessError` alone is not treated as a diagnosis. If Drive cannot accept the report, its safe details remain in the cell output. Send that report for diagnosis instead of repeating training without knowing the cause.\n\nBy default results are saved in a new unique folder below `DCC_AIdeas/results/mission3/ver22`. The 180-minute shared deadline starts after Drive login and before dependency installation. Package installation has a 20-minute cap. A 25-minute reserve protects final inference and publication; timing estimates use a 1.25 safety factor. The watchdog bounds the worker process and requests Colab GPU release. These controls limit work; they do not guarantee completion of every stage on every GPU or Drive connection.\n\n`AUTO_RELEASE_GPU=True` requests runtime release after results are displayed, on errors, or at the shared deadline. Results saved to Drive remain; `/content` scratch files may disappear. A failed release request prints manual Runtime > Disconnect and delete runtime instructions. Actual Colab release behavior requires a Colab run to verify.\n\nLarge per-utterance feature caches stay in the Colab runtime below `/content/mission3_ver22/feature_caches/<run_id>/`, avoiding an extra multi-gigabyte Drive upload. Drive retains cache summaries, final model assets, metrics, and predictions. A compatible cache can be reused within the same runtime after identity checks; after runtime release, resuming may need to extract features again. Losing the temporary cache does not invalidate a completed saved model.\n\n## Results, fallback, and resuming\n\n- `candidate_summary.csv`: baseline and completed evidence candidates on the Training holdout.\n- `candidate_confirmation.json`: actual fit-only model confirmation of the selected cached candidate.\n- `training_reference_comparison.json`: paired nausea error transitions and F1 changes.\n- `final_selection.json`: selected objective, epoch count, and fallback reason when applicable.\n- `training_metrics.json`: Training holdout metrics for the selected candidate.\n- `validation_metrics.json` and `mission3.csv`: final model evaluation and predictions.\n- `mission3_ver22_model.ckpt`: relative-path model manifest; preserve the complete run folder.\n- `runtime.json`: operational status. `complete` indicates an improved candidate completed full-Training evidence fitting. `complete_not_improved` preserves the existing full ver19 model because neither new candidate passed selection. `complete_budget_limited` preserves the existing full ver19 model because candidate search or full-Training evidence fitting could not complete in budget, including a search stopped before its first evaluated candidate. Neither fallback is a performance improvement.\n- `final_evidence.npz`, `final_history.json`: new evidence weights and their full-Training learning history, when an improved candidate is exported.\n- `evidence_artifacts.json`: lists the active evidence sidecars belonging to the current export. Sidecars retained from an interrupted earlier attempt are not active when the current export falls back to the baseline. The model manifest and active-path record identify what was actually published.\n- `run_complete.json`: final artifact integrity marker. A missing marker or final CSV means the run is not a completed submission result, even if some intermediate files exist.\n\nTo resume, set `resume_run` to the previous ver22 Drive run path. Compatible completed results are verified and returned without more training. An interrupted run may restart feature extraction and evidence fitting, not continue at its last update. Each explicit rerun starts a new budget and uses additional Colab resources. Keep `resume_run` empty for a fresh experiment. No raw patient text is exported in comparison summaries or displayed in notebook output.\n\nPortable inference is available through `submission/inference.py`:\n\n```text\npython inference.py --checkpoint /path/to/run/mission3_ver22_model.ckpt --label_dir /path/to/labels --output /path/to/mission3.csv\n```\n\nThe CSV header is `label file name,symptom`. Preserve all relative model assets when moving a run.\n\n## Local verification\n\nFrom the repository root:\n\n```text\n.venv\\Scripts\\python.exe -B src/mission3/mission3_ver22/build_notebook.py\n.venv\\Scripts\\python.exe -B -m unittest discover -s src/mission3/mission3_ver22 -t src/mission3 -p "test_*.py"\n```\n\nFocused tests check the evidence objective, frozen outputs, source identity, notebook restoration, deadline sharing, GPU release requests, and inference/publication contracts. Synthetic checks establish execution behavior only; they do not establish actual F1 gains or a measured Colab duration.\n'}
RUNTIME_PREPARED = None
DEPENDENCIES_PREPARED = None
if ("DEADLINE_MONOTONIC" not in globals() or "time" not in globals()
        or not callable(globals().get("release_gpu"))):
    raise RuntimeError("작업 예산 준비가 완료되지 않았습니다. 런타임 → 모두 실행으로 처음 셀부터 실행하세요.")
try:
    if time.monotonic() >= DEADLINE_MONOTONIC:
        raise RuntimeError("이번 작업의 시간 예산이 끝났습니다. 다시 실행하려면 처음 셀부터 모두 실행하세요.")
    runtime_dir = Path(tempfile.mkdtemp(prefix="mission3_ver22_"))
    for name, source in RUNTIME_SOURCES.items():
        (runtime_dir / name).write_bytes(source.encode("utf-8"))
    budget_spec = importlib.util.spec_from_file_location("mission3_ver22_budget_supervisor", runtime_dir / "budget.py")
    budget_module = importlib.util.module_from_spec(budget_spec)
    sys.modules[budget_spec.name] = budget_module
    budget_spec.loader.exec_module(budget_module)
    RUNTIME_PREPARED = (str(runtime_dir), DEADLINE_MONOTONIC)
    print("실행 코드 준비 완료")
except BaseException:
    cleanup = globals().get("release_gpu")
    if callable(cleanup):
        try:
            cleanup()
        except Exception as cleanup_error:
            print("GPU 반환 중 오류:", type(cleanup_error).__name__, "· 위의 최초 오류를 확인하세요.")
    raise


In [ ]:
# @title 기존 프로젝트와 같은 버전의 패키지 준비 (최대 20분, 전체 예산에 포함)
DEPENDENCIES_PREPARED = None
required = ("runtime_dir", "budget_module", "DEADLINE_MONOTONIC", "time", "sys")
if (any(name not in globals() for name in required)
        or not callable(globals().get("release_gpu"))):
    raise RuntimeError("실행 코드 준비가 완료되지 않았습니다. 런타임 → 모두 실행으로 처음 셀부터 실행하세요.")
if (globals().get("RUNTIME_PREPARED") != (str(runtime_dir), DEADLINE_MONOTONIC)
        or not (runtime_dir / "requirements.txt").is_file()):
    raise RuntimeError("현재 실행의 코드 준비가 미완료입니다. 앞선 셀의 첫 오류를 확인하고 처음부터 모두 실행하세요.")
try:
    if time.monotonic() >= DEADLINE_MONOTONIC:
        raise RuntimeError("이번 작업의 시간 예산이 끝났습니다. 다시 실행하려면 처음 셀부터 모두 실행하세요.")
    package_deadline = min(DEADLINE_MONOTONIC, time.monotonic() + 20 * 60)
    budget_module.run_supervised(
        [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(runtime_dir / "requirements.txt")],
        deadline_monotonic=package_deadline, shutdown_margin_s=10,
    )
    DEPENDENCIES_PREPARED = (str(runtime_dir), DEADLINE_MONOTONIC)
    print("패키지 준비 완료 · 남은 작업 시간:", round(max(0, DEADLINE_MONOTONIC - time.monotonic()) / 60, 1), "분")
except BaseException:
    cleanup = globals().get("release_gpu")
    if callable(cleanup):
        try:
            cleanup()
        except Exception as cleanup_error:
            print("GPU 반환 중 오류:", type(cleanup_error).__name__, "· 위의 최초 오류를 확인하세요.")
    raise


In [ ]:
# @title 기존 모델 확인 → 특징 저장 · 후보 비교 → 최종 평가 · GPU 반환
import csv
import json
import subprocess
required = ("runtime_dir", "SETTINGS", "budget_module", "DEADLINE_MONOTONIC",
            "BUDGET_STARTED_MONOTONIC", "time", "Path", "sys")
if (any(name not in globals() for name in required)
        or not callable(globals().get("release_gpu"))):
    raise RuntimeError("준비 셀이 완료되지 않았거나 런타임이 초기화됐습니다. 마지막 셀만 실행하지 말고 런타임 → 모두 실행으로 처음부터 실행하세요. 앞선 셀이 실패했다면 그 셀의 첫 오류를 확인하세요.")
prepared_context = (str(runtime_dir), DEADLINE_MONOTONIC)
if (globals().get("RUNTIME_PREPARED") != prepared_context
        or globals().get("DEPENDENCIES_PREPARED") != prepared_context
        or not (runtime_dir / "run_notebook.py").is_file()):
    raise RuntimeError("현재 실행의 코드 또는 패키지 준비가 미완료입니다. 앞선 셀의 첫 오류를 확인하고 런타임 → 모두 실행으로 처음부터 실행하세요.")
try:
    if time.monotonic() >= DEADLINE_MONOTONIC:
        raise RuntimeError("이번 작업의 시간 예산이 끝났습니다. 다시 실행하려면 처음 셀부터 모두 실행하세요.")
    settings_file = runtime_dir / "notebook_settings.json"
    result_file = runtime_dir / "notebook_result.json"
    error_file = runtime_dir / "notebook_error.json"
    error_file.write_text(json.dumps({"status": "starting"}), encoding="utf-8")
    settings_file.write_text(json.dumps(SETTINGS, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    budget_module.run_supervised(
        [sys.executable, "-u", str(runtime_dir / "run_notebook.py"),
         "--settings", str(settings_file), "--result_file", str(result_file)],
        deadline_monotonic=DEADLINE_MONOTONIC, shutdown_margin_s=10,
    )
    result_dir = Path(json.loads(result_file.read_text(encoding="utf-8")))
    print("결과 폴더:", result_dir)
    summary = result_dir / "candidate_summary.csv"
    if summary.is_file():
        print("Training 내부 비교:")
        with summary.open(encoding="utf-8-sig", newline="") as handle:
            for row in csv.DictReader(handle):
                print(row["candidate"], "Macro F1 =", round(float(row["macro_f1"]), 6),
                      "오심 F1 =", round(float(row["nausea_f1"]), 6),
                      "선택 epoch =", row["selected_epochs"], "종료 이유 =", row["stop_reason"])
    comparison_path = result_dir / "training_reference_comparison.json"
    if comparison_path.is_file():
        comparison = json.loads(comparison_path.read_text(encoding="utf-8-sig"))
        if comparison["available"]:
            nausea = comparison["per_symptom"]["오심"]["paired"]
            print("동일 Training 내부 평가 기준:", comparison["reference_version"],
                  "· 전체 F1 변화:", round(comparison["macro_f1_delta"], 6),
                  "· 오심 F1 변화:", round(comparison["nausea_f1_delta"], 6))
            print("오심 누락 회복:", nausea["fn_to_tp"], "· 새 누락:", nausea["tp_to_fn"],
                  "· 오탐 제거:", nausea["fp_to_tn"], "· 새 오탐:", nausea["tn_to_fp"])
            if comparison["both_macro_and_nausea_improved"]:
                print("Training 내부 평가에서 전체 F1과 오심 F1이 모두 개선됐습니다.")
            else:
                print("전체 F1과 오심 F1의 동시 개선 조건을 충족하지 못했습니다.")
        else:
            print("기존 Training 결과 비교 불가:", comparison.get("reason", "기록 확인 필요"))
    for filename in ["training_metrics.json", "validation_metrics.json"]:
        path = result_dir / filename
        if path.is_file():
            report = json.loads(path.read_text(encoding="utf-8-sig"))
            print(filename, "Macro F1 =", round(report["macro_f1"], 6))
            for symptom, values in report["per_symptom"].items():
                print(symptom, "F1 =", round(values["f1"], 4),
                      "P =", round(values["precision"], 4), "R =", round(values["recall"], 4))
    manifest_path = result_dir / "mission3_ver22_model.ckpt"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8-sig"))
        print("저장된 후보:", manifest.get("selected_candidate", "모델 정보 확인"))
        print("저장 모델 학습 범위:", manifest.get("training_scope", "모델 정보 확인"),
              "· 학습 표본:", manifest.get("fitted_samples", "모델 정보 확인"),
              "· 오심 근거 학습 epoch:", manifest.get("evidence_epochs", 0))
        if manifest.get("refit_skipped_reason"):
            print("추가 학습 생략 이유:", manifest["refit_skipped_reason"])
    status_path = result_dir / "runtime.json"
    status = json.loads(status_path.read_text(encoding="utf-8-sig")) if status_path.is_file() else {}
    print("실행 상태:", status.get("status", "기록 확인 필요"))
    print("이번 노트북 소요 시간:", round((time.monotonic() - BUDGET_STARTED_MONOTONIC) / 60, 1), "분")
    print("선택 근거:", result_dir / "final_selection.json")
    complete = ((result_dir / "run_complete.json").is_file() and (result_dir / "mission3.csv").is_file()
                and status.get("status") in {"complete", "complete_budget_limited", "complete_not_improved"})
    if complete:
        if status.get("status") == "complete_not_improved":
            print("비교 완료 · 개선 미달로 기존 전체 Training ver19 모델을 그대로 보존했습니다.")
            print("성능 향상 결과가 아닙니다. 최종 예측:", result_dir / "mission3.csv")
        elif status.get("status") == "complete_budget_limited":
            print("예산 제한 · 새 학습을 완료하지 못해 기존 전체 Training ver19 모델을 보존했습니다.")
            print("성능 향상 결과가 아닙니다. 최종 예측:", result_dir / "mission3.csv")
        else:
            print("최종 작업 완료 · 새 오심 발화 근거 학습을 적용한 예측:", result_dir / "mission3.csv")
        print("기존 Drive의 ver19 원본 파일은 유지됩니다.")
    else:
        print("미완료 실행:", status.get("status", "상태 기록 없음"),
              "· 최종 제출 결과로 사용하지 마세요. 결과 폴더의 로그를 확인하세요.")
except subprocess.CalledProcessError as worker_error:
    failure = {"status": "failed", "stage": "worker_process", "error_type": "WorkerProcessError",
               "exit_code": worker_error.returncode,
               "reason": "실행 프로그램이 상세 오류를 저장하기 전에 종료됐습니다. 종료 코드와 준비 상태 확인이 필요합니다.",
               "frames": []}
    try:
        recorded = json.loads(error_file.read_text(encoding="utf-8"))
        if isinstance(recorded, dict) and recorded.get("status") == "failed":
            failure.update(recorded)
    except (OSError, ValueError):
        pass
    print("ver22 내부 오류 · 실패 단계:", failure["stage"], "· 오류 종류:", failure["error_type"], flush=True)
    print("확인 내용:", failure["reason"], flush=True)
    if failure.get("module_name"):
        print("불러오지 못한 모듈:", failure["module_name"], flush=True)
    for frame in failure.get("frames", [])[-6:]:
        print("코드 위치:", frame["file"], "·", frame["function"], "·", frame["line"], flush=True)
    saved_error = None
    try:
        from datetime import datetime, timezone
        import uuid
        error_dir = Path(SETTINGS["output_dir"]) / "_diagnostics"
        error_dir.mkdir(parents=True, exist_ok=True)
        saved_error = error_dir / ("ver22_error_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
                                  + "_" + uuid.uuid4().hex[:8] + ".json")
        saved_error.write_text(json.dumps(failure, ensure_ascii=False, indent=2), encoding="utf-8")
        print("오류 보고서:", saved_error, flush=True)
    except OSError as save_error:
        saved_error = None
        print("Drive 오류 보고서 저장 실패:", type(save_error).__name__, "· 위의 내부 오류 내용을 복사하세요.", flush=True)
    raise RuntimeError("ver22 실행 실패 [" + failure["stage"] + " / " + failure["error_type"] + "]: "
                       + failure["reason"] + (" 오류 보고서: " + str(saved_error) if saved_error else "")) from None
finally:
    cleanup = globals().get("release_gpu")
    if callable(cleanup):
        try:
            cleanup()
        except Exception as cleanup_error:
            print("GPU 반환 중 오류:", type(cleanup_error).__name__, "· Colab에서 직접 연결을 해제하세요.")


## 결과 확인 · 다시 실행할 때

결과는 `DCC_AIdeas/results/mission3/ver22/실행시각_고유번호/`에 저장합니다.
`candidate_summary.csv`, `training_reference_comparison.json`, `final_selection.json`,
`training_metrics.json`, `validation_metrics.json`, `runtime.json`을 확인하세요.
`complete`는 개선 후보를 전체 Training 특징으로 학습한 결과입니다.
`complete_not_improved`는 개선 미달, `complete_budget_limited`는 시간 제약 때문에
**기존 전체 Training ver19 모델을 그대로 보존한 결과**입니다. 새 성능 향상으로 해석하지 마세요.
`run_complete.json`과 최종 CSV가 없는 실행은 완료된 제출 결과가 아닙니다.
실행이 실패하면 GPU 반환 전에 실패 단계·오류 종류·코드 위치를 표시하고
`ver22/_diagnostics/ver22_error_*.json`에 같은 내용을 저장합니다.
원인 분석에는 이 오류 보고서를 보내주세요. 원문 대화와 개별 데이터 파일 이름은 포함하지 않습니다.

`resume_run`에 이전 실행 폴더의 전체 Drive 경로를 넣으면 코드·설정·데이터 일치를 확인합니다.
완료된 결과는 검증 후 반환하며, 중단된 학습은 마지막 갱신부터 이어지는 것을 보장하지 않습니다.
**재실행에는 새 시간 예산과 추가 Colab 리소스가 듭니다.** 새 실험은 `resume_run`을 비우고 실행합니다.
모델 `.ckpt`는 가중치·assets의 상대 경로를 담은 manifest이므로 **실행 폴더 전체를 보관**하세요.
제출용 추론 코드는 `submission/inference.py`이며 예측 CSV의 헤더는 `label file name,symptom`입니다.
원문 대화나 개인정보를 화면과 결과 요약에 출력하지 않습니다.
자동 반환이 실패하면 **런타임 → 연결 해제 및 런타임 삭제**를 직접 실행하세요.
자동 반환 기능의 실제 서비스 동작은 Colab에서 확인해야 합니다.
